# Public Portfolio Copy

This notebook is a GitHub-safe source copy of the ChemRisk-AI Phase 1 MVP. The original project used three private source workbooks: a sanitized public chemical-pair workbook, an initial gold-set audit workbook, and a prospective 15-row audit workbook.

The raw audit workbooks are intentionally omitted from the public repository because they contain detailed engineering audit notes and calibration decisions. Public readers should use the README, figures, model card, and final summary tables to review the completed result. To run every cell end-to-end, provide private or synthetic input files matching the schema documented in `data/README.md`.


# ChemRisk-AI: Data QA, Weak-Supervision Reproduction, and Legacy RTD Baseline

## Notebook 01 — Phase 1 MVP

ChemRisk-AI is a mechanism-aware decision-support system for chemical incompatibility screening.

The Phase 1 objective is to determine whether a resistance temperature detector (RTD) is a useful and creditable safeguard for a chemical-pair mixing scenario, and ultimately whether an RTD should be recommended for a storage-tank service.

This notebook establishes the reproducible data foundation for the project.

## Objectives

1. Load and validate the public chemical-pair dataset and the audited gold-set subset.
2. Separate the 21 audited development rows from the weak-label model-training pool.
3. Recreate the spreadsheet helper features and labeling functions in Python.
4. Confirm that the Python LF outputs match the spreadsheet reference values.
5. Quantify the legacy RTD over-prescription baseline before training a machine-learning model.
6. Prepare clean model-development datasets for the next notebook.

## Phase 1 Scope

The primary modeling target is pair-level RTD usefulness:

- `1` = RTD required / creditable as a thermal safeguard
- `0` = RTD not required as the primary thermal safeguard
- `Review` = conflicting or insufficient weak-supervision evidence

Severity-demotion rules are retained only as sparse engineering-review flags. A separate severity-demotion model will not be trained in Phase 1.

SDS/NLP ingestion and generalization to unseen chemicals remain a future Phase 2 extension.

In [ ]:
# ============================================================
# 1. Environment setup and imports
# ============================================================

from __future__ import annotations

import json
import random
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn import __version__ as sklearn_version

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Display settings
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

print("Environment ready.")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"scikit-learn: {sklearn_version}")
print(f"Random state: {RANDOM_STATE}")

# 2. Load Source Workbooks

This notebook uses two source workbooks:

1. The public chemical-pair dataset containing approximately 313 rows, helper features, weak-supervision labeling functions, and a static `Public_Colab_Values` sheet.
2. The gold-set audit workbook containing the 21 completed engineering audits used for LF calibration and preliminary development evaluation.

The audited rows will be excluded from the weak-label model-training pool to reduce leakage.

In [ ]:
# ============================================================
# 2.1 Upload the public and gold-set Excel workbooks
# ============================================================

# In Google Colab, upload the source workbooks interactively.
# Outside Colab, fall back to the /mnt/data files used for automated QA.
try:
    from google.colab import files  # type: ignore

    uploaded = files.upload()
    uploaded_files = list(uploaded.keys())

except ModuleNotFoundError:
    uploaded = {}
    uploaded_files = [
        str(path)
        for path in Path("/mnt/data").glob("*.xlsx")
        if (
            "chemical_pair_risk_dataset_public" in path.name.lower()
            or "gold_set_audit" in path.name.lower()
        )
    ]

print("\nUploaded / discovered files:")
for filename in uploaded_files:
    print(f"  - {filename}")

if len(uploaded_files) < 2:
    raise ValueError(
        "Upload both the public-data workbook and the gold-set audit workbook."
    )

In [ ]:
# ============================================================
# 2.2 Identify the uploaded source workbooks
# ============================================================

excel_files = [Path(name) for name in uploaded_files if name.lower().endswith(".xlsx")]

if len(excel_files) != 2:
    raise ValueError(
        f"Expected exactly 2 Excel workbooks, but found {len(excel_files)}: "
        f"{[file.name for file in excel_files]}"
    )

public_candidates = [
    file for file in excel_files
    if "chemical_pair_risk_dataset_public" in file.name.lower()
]

gold_candidates = [
    file for file in excel_files
    if "gold_set_audit" in file.name.lower()
]

if len(public_candidates) != 1:
    raise ValueError(
        "Could not uniquely identify the public-data workbook. "
        "Its filename should contain 'chemical_pair_risk_dataset_public'."
    )

if len(gold_candidates) != 1:
    raise ValueError(
        "Could not uniquely identify the gold-set workbook. "
        "Its filename should contain 'gold_set_audit'."
    )

PUBLIC_WORKBOOK = public_candidates[0]
GOLD_WORKBOOK = gold_candidates[0]

print(f"Public workbook:   {PUBLIC_WORKBOOK.name}")
print(f"Gold-set workbook: {GOLD_WORKBOOK.name}")

In [ ]:
# ============================================================
# 2.3 Inspect workbook sheet names
# ============================================================

public_excel = pd.ExcelFile(PUBLIC_WORKBOOK)
gold_excel = pd.ExcelFile(GOLD_WORKBOOK)

print("Public workbook sheets:")
for sheet in public_excel.sheet_names:
    print(f"  - {sheet}")

print("\nGold-set workbook sheets:")
for sheet in gold_excel.sheet_names:
    print(f"  - {sheet}")

In [ ]:
# ============================================================
# 2.4 Load the public dataset and gold-set audit
# ============================================================

PUBLIC_SHEET = "Public_Colab_Values"
GOLD_SHEET = "Gold_Set_Audit_50"

if PUBLIC_SHEET not in public_excel.sheet_names:
    raise KeyError(
        f"Required sheet '{PUBLIC_SHEET}' was not found in "
        f"{PUBLIC_WORKBOOK.name}."
    )

if GOLD_SHEET not in gold_excel.sheet_names:
    raise KeyError(
        f"Required sheet '{GOLD_SHEET}' was not found in "
        f"{GOLD_WORKBOOK.name}."
    )

public_df = pd.read_excel(
    PUBLIC_WORKBOOK,
    sheet_name=PUBLIC_SHEET
)

gold_df = pd.read_excel(
    GOLD_WORKBOOK,
    sheet_name=GOLD_SHEET
)

# Remove fully blank rows and columns
public_df = public_df.dropna(axis=0, how="all").dropna(axis=1, how="all")
gold_df = gold_df.dropna(axis=0, how="all").dropna(axis=1, how="all")

print(f"Public dataset shape: {public_df.shape}")
print(f"Gold-set sheet shape: {gold_df.shape}")

display(public_df.head(3))
display(gold_df.head(3))

# 3. Schema Validation and Leakage Control

Before reproducing the labeling functions or training a model, this section verifies that:

1. Required columns are present.
2. Column names and categorical values are standardized.
3. The completed audited rows are identified correctly.
4. Audited pair IDs are removed from the weak-label training pool.
5. Gold-set-only fields and downstream outputs are excluded from predictive features.

The 21 completed audits were used during labeling-function development, so they are treated as a development/calibration set rather than an untouched final holdout.

In [ ]:
# ============================================================
# 3.1 Normalize column names
# ============================================================

def normalize_column_name(column: object) -> str:
    """Convert a column name to a consistent snake_case format."""
    name = str(column).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")


def validate_unique_columns(df: pd.DataFrame, dataframe_name: str) -> None:
    """Raise an error when normalization creates duplicate column names."""
    duplicates = df.columns[df.columns.duplicated()].tolist()

    if duplicates:
        raise ValueError(
            f"{dataframe_name} contains duplicate columns after normalization: "
            f"{duplicates}"
        )


public_df.columns = [normalize_column_name(col) for col in public_df.columns]
gold_df.columns = [normalize_column_name(col) for col in gold_df.columns]

validate_unique_columns(public_df, "public_df")
validate_unique_columns(gold_df, "gold_df")

print(f"Public columns: {len(public_df.columns)}")
print(f"Gold-set columns: {len(gold_df.columns)}")

print("\nPublic-data columns:")
print(public_df.columns.tolist())

print("\nGold-set columns:")
print(gold_df.columns.tolist())

In [ ]:
# ============================================================
# 3.2 Standardize blank values and text fields
# ============================================================

BLANK_STRINGS = {
    "",
    " ",
    "nan",
    "none",
    "null",
    "n/a",
    "na",
    "-"
}


def clean_object_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Trim text, replace blank-like strings with missing values."""
    cleaned = df.copy()

    for column in cleaned.select_dtypes(include="object").columns:
        cleaned[column] = cleaned[column].astype("string").str.strip()

        blank_mask = cleaned[column].str.lower().isin(BLANK_STRINGS)
        cleaned.loc[blank_mask, column] = pd.NA

    return cleaned


public_df = clean_object_columns(public_df)
gold_df = clean_object_columns(gold_df)

# Pair IDs should remain strings so formatting is preserved.
for df in (public_df, gold_df):
    if "pair_id" in df.columns:
        df["pair_id"] = df["pair_id"].astype("string").str.strip().str.upper()

print("Blank values and string fields standardized.")

In [ ]:
# ============================================================
# 3.3 Validate required columns
# ============================================================

PUBLIC_REQUIRED_INPUTS = [
    "pair_id",
    "chemical_a_class",
    "chemical_b_class",
    "reaction_type",
    "reaction_speed",
    "setting",
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "corrosive",
    "final_severity_num",
]

PUBLIC_REFERENCE_OUTPUTS = [
    "weak_vote_prob",
    "weak_vote_count",
    "weak_rtd_label",
    "lf_conflict_flag",
]

GOLD_REQUIRED_COLUMNS = [
    "pair_id",
    "rtd_required",
    "control_failure_mode",
]

GOLD_OPTIONAL_AUDIT_COLUMNS = [
    "gold_severity_num",
    "gold_lt_c5",
    "exotherm_present",
    "exotherm_hazardous",
    "runaway_credible",
    "pressure_hazard_credible",
    "rtd_detectable",
]


def report_schema(
    df: pd.DataFrame,
    required_columns: list[str],
    dataframe_name: str
) -> list[str]:
    """Report required-column availability and return missing columns."""
    missing = [col for col in required_columns if col not in df.columns]
    present = [col for col in required_columns if col in df.columns]

    print(f"\n{dataframe_name}")
    print("-" * len(dataframe_name))
    print(f"Required columns present: {len(present)}/{len(required_columns)}")

    if missing:
        print("Missing required columns:")
        for column in missing:
            print(f"  - {column}")
    else:
        print("All required columns are present.")

    return missing


missing_public_inputs = report_schema(
    public_df,
    PUBLIC_REQUIRED_INPUTS,
    "Public dataset — raw inputs"
)

missing_public_outputs = report_schema(
    public_df,
    PUBLIC_REFERENCE_OUTPUTS,
    "Public dataset — spreadsheet reference outputs"
)

missing_gold = report_schema(
    gold_df,
    GOLD_REQUIRED_COLUMNS,
    "Gold-set audit"
)

if missing_public_inputs or missing_gold:
    raise KeyError(
        "One or more essential columns are missing. "
        "Review the normalized column list above before continuing."
    )

In [ ]:
# ============================================================
# 3.4 Identify completed audits and create leakage-safe pools
# ============================================================

# Convert audited target to numeric while preserving missing rows.
gold_df["rtd_required"] = pd.to_numeric(
    gold_df["rtd_required"],
    errors="coerce"
)

# A completed audit requires:
# - a pair ID
# - an audited RTD target
# - a control failure mode
audited_mask = (
    gold_df["pair_id"].notna()
    & gold_df["rtd_required"].isin([0, 1])
    & gold_df["control_failure_mode"].notna()
)

audited_gold_df = gold_df.loc[audited_mask].copy()
unaudited_gold_df = gold_df.loc[
    gold_df["pair_id"].notna() & ~audited_mask
].copy()

audited_pair_ids = set(audited_gold_df["pair_id"].dropna())

public_pair_ids = set(public_df["pair_id"].dropna())

matched_audited_ids = audited_pair_ids & public_pair_ids
missing_from_public = audited_pair_ids - public_pair_ids

# Remove audited development rows from model-training data.
training_pool_df = public_df.loc[
    ~public_df["pair_id"].isin(audited_pair_ids)
].copy()

# Public versions of the audited rows for parity checks.
public_audited_df = public_df.loc[
    public_df["pair_id"].isin(audited_pair_ids)
].copy()

print("Gold-set audit status")
print("---------------------")
print(f"Completed audited rows:       {len(audited_gold_df)}")
print(f"Remaining unaudited rows:     {len(unaudited_gold_df)}")
print(f"Unique audited pair IDs:      {len(audited_pair_ids)}")
print(f"Audited IDs found in public:  {len(matched_audited_ids)}")
print(f"Audited IDs missing publicly: {len(missing_from_public)}")

print("\nLeakage-safe data pools")
print("-----------------------")
print(f"Original public rows:         {len(public_df)}")
print(f"Audited public rows removed:  {len(public_audited_df)}")
print(f"Remaining training-pool rows: {len(training_pool_df)}")

if missing_from_public:
    print("\nWarning — audited pair IDs not found in public data:")
    for pair_id in sorted(missing_from_public):
        print(f"  - {pair_id}")

if len(audited_gold_df) != 21:
    print(
        f"\nReview required: expected 21 completed audits, "
        f"but identified {len(audited_gold_df)}."
    )

if training_pool_df["pair_id"].isin(audited_pair_ids).any():
    raise AssertionError("Leakage detected: audited rows remain in the training pool.")

print("\nLeakage check passed: no audited pair IDs remain in training_pool_df.")

In [ ]:
# ============================================================
# 3.5 Inspect audited target and failure-mode distributions
# ============================================================

audit_target_summary = (
    audited_gold_df["rtd_required"]
    .value_counts(dropna=False)
    .rename_axis("rtd_required")
    .reset_index(name="row_count")
)

failure_mode_summary = (
    audited_gold_df["control_failure_mode"]
    .value_counts(dropna=False)
    .rename_axis("control_failure_mode")
    .reset_index(name="row_count")
)

print("Audited RTD target distribution:")
display(audit_target_summary)

print("\nAudited control failure modes:")
display(failure_mode_summary)

print("\nCompleted audited rows:")
display(
    audited_gold_df[
        [
            "pair_id",
            "rtd_required",
            "control_failure_mode",
        ]
    ].sort_values("pair_id")
)

# 4. Data Quality, Pair Integrity, and Legacy RTD Baseline

This section performs the final data checks needed before recreating the
weak-supervision framework.

It will:

1. Convert hazard indicators to consistent numeric values.
2. Measure missingness and inspect categorical values.
3. Check pair IDs and unordered chemical-pair duplicates.
4. Quantify the legacy RTD recommendation rule:
   high severity (C5/C6) plus heat generation.
5. Compare that broad baseline with the 21 audited engineering decisions.

The baseline establishes the decision and business problem that the
mechanism-aware system is intended to improve.

In [ ]:
# ============================================================
# 4.1 Validate and standardize data types
# ============================================================

BINARY_COLUMNS = [
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "corrosive",
]

NUMERIC_COLUMNS = [
    "final_severity_num",
]


def coerce_binary_column(
    series: pd.Series,
    column_name: str
) -> pd.Series:
    """
    Convert common binary representations to nullable integers:
    1 = Yes/True
    0 = No/False
    <NA> = missing or unrecognized
    """
    text = series.astype("string").str.strip().str.lower()

    mapping = {
        "1": 1,
        "yes": 1,
        "y": 1,
        "true": 1,
        "present": 1,

        "0": 0,
        "no": 0,
        "n": 0,
        "false": 0,
        "absent": 0,
    }

    mapped = text.map(mapping)
    numeric = pd.to_numeric(series, errors="coerce")

    result = mapped.fillna(numeric)

    invalid_mask = (
        series.notna()
        & result.isna()
    )

    if invalid_mask.any():
        invalid_values = sorted(
            series.loc[invalid_mask]
            .astype("string")
            .dropna()
            .unique()
            .tolist()
        )

        print(
            f"Warning: {column_name} contains unrecognized values: "
            f"{invalid_values}"
        )

    return result.astype("Int64")


for column in BINARY_COLUMNS:
    public_df[column] = coerce_binary_column(
        public_df[column],
        column
    )

for column in NUMERIC_COLUMNS:
    public_df[column] = pd.to_numeric(
        public_df[column],
        errors="coerce"
    )

# Standardize important categorical columns.
CATEGORICAL_COLUMNS = [
    "chemical_a_class",
    "chemical_b_class",
    "reaction_type",
    "reaction_speed",
    "setting",
]

for column in CATEGORICAL_COLUMNS:
    public_df[column] = (
        public_df[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

print("Data types standardized.")

display(
    public_df[
        BINARY_COLUMNS
        + NUMERIC_COLUMNS
        + CATEGORICAL_COLUMNS
    ].dtypes.to_frame("dtype")
)

In [ ]:
# ============================================================
# 4.2 Inspect missingness and categorical values
# ============================================================

qa_columns = (
    ["pair_id"]
    + CATEGORICAL_COLUMNS
    + BINARY_COLUMNS
    + NUMERIC_COLUMNS
)

missingness_summary = pd.DataFrame({
    "column": qa_columns,
    "missing_count": [
        public_df[column].isna().sum()
        for column in qa_columns
    ],
    "missing_percent": [
        public_df[column].isna().mean() * 100
        for column in qa_columns
    ],
    "unique_non_null": [
        public_df[column].nunique(dropna=True)
        for column in qa_columns
    ],
}).sort_values(
    ["missing_percent", "column"],
    ascending=[False, True]
)

print("Missingness summary:")
display(missingness_summary)

for column in [
    "reaction_type",
    "reaction_speed",
    "setting",
]:
    print(f"\nValue counts — {column}:")
    display(
        public_df[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="row_count")
    )

In [ ]:
# ============================================================
# 4.3 Check pair IDs and unordered chemical-pair duplicates
# ============================================================

duplicate_pair_ids = public_df.loc[
    public_df["pair_id"].duplicated(keep=False),
    "pair_id"
].sort_values()

print(f"Duplicate pair_id rows: {len(duplicate_pair_ids)}")

if not duplicate_pair_ids.empty:
    display(
        public_df.loc[
            public_df["pair_id"].isin(duplicate_pair_ids)
        ].sort_values("pair_id")
    )


def select_pair_identity_columns(
    df: pd.DataFrame
) -> tuple[str, str]:
    """
    Use the most specific available pair-identity columns.
    Fall back to chemical classes when public labels are unavailable.
    """
    candidate_pairs = [
        ("chemical_a_public_label", "chemical_b_public_label"),
        ("chemical_a_label", "chemical_b_label"),
        ("chemical_a_name", "chemical_b_name"),
        ("chemical_a_class", "chemical_b_class"),
    ]

    for chemical_a_col, chemical_b_col in candidate_pairs:
        if (
            chemical_a_col in df.columns
            and chemical_b_col in df.columns
        ):
            return chemical_a_col, chemical_b_col

    raise KeyError(
        "No suitable Chemical A / Chemical B identity columns were found."
    )


CHEMICAL_A_COLUMN, CHEMICAL_B_COLUMN = (
    select_pair_identity_columns(public_df)
)

print(
    "Pair identity columns selected: "
    f"{CHEMICAL_A_COLUMN}, {CHEMICAL_B_COLUMN}"
)


def make_canonical_pair_key(
    row: pd.Series,
    chemical_a_col: str,
    chemical_b_col: str
) -> str:
    """
    Build an order-independent pair key so A+B and B+A match.
    """
    chemical_a = str(row[chemical_a_col]).strip().lower()
    chemical_b = str(row[chemical_b_col]).strip().lower()

    return " || ".join(sorted([chemical_a, chemical_b]))


public_df["canonical_pair_key"] = public_df.apply(
    make_canonical_pair_key,
    axis=1,
    chemical_a_col=CHEMICAL_A_COLUMN,
    chemical_b_col=CHEMICAL_B_COLUMN,
)

canonical_pair_counts = (
    public_df["canonical_pair_key"]
    .value_counts()
)

repeated_canonical_keys = canonical_pair_counts[
    canonical_pair_counts > 1
].index

print(
    "Repeated unordered chemical-pair keys: "
    f"{len(repeated_canonical_keys)}"
)

if len(repeated_canonical_keys) > 0:
    repeated_pairs_df = public_df.loc[
        public_df["canonical_pair_key"].isin(
            repeated_canonical_keys
        ),
        [
            "pair_id",
            CHEMICAL_A_COLUMN,
            CHEMICAL_B_COLUMN,
            "canonical_pair_key",
            "reaction_type",
            "generates_heat",
            "generates_gas",
            "toxic_gas",
            "explosive_potential",
            "final_severity_num",
        ],
    ].sort_values("canonical_pair_key")

    display(repeated_pairs_df)

In [ ]:
# ============================================================
# 4.4 Check conflicting data among repeated pair keys
# ============================================================

PAIR_COMPARISON_COLUMNS = [
    "reaction_type",
    "reaction_speed",
    "setting",
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "corrosive",
    "final_severity_num",
]

pair_conflict_records = []

for pair_key, group in public_df.groupby("canonical_pair_key"):
    if len(group) <= 1:
        continue

    conflicting_columns = [
        column
        for column in PAIR_COMPARISON_COLUMNS
        if group[column].nunique(dropna=False) > 1
    ]

    if conflicting_columns:
        pair_conflict_records.append({
            "canonical_pair_key": pair_key,
            "row_count": len(group),
            "pair_ids": ", ".join(
                group["pair_id"].astype(str).tolist()
            ),
            "conflicting_columns": ", ".join(
                conflicting_columns
            ),
        })

pair_conflict_summary = pd.DataFrame(pair_conflict_records)

print(
    "Repeated pair keys with conflicting feature values: "
    f"{len(pair_conflict_summary)}"
)

if not pair_conflict_summary.empty:
    display(pair_conflict_summary)

In [ ]:
# ============================================================
# 4.5 Rebuild the legacy severe-plus-heat RTD baseline
# ============================================================

baseline_required_fields = (
    public_df["final_severity_num"].notna()
    & public_df["generates_heat"].notna()
)

public_df["legacy_rtd_baseline"] = pd.Series(
    pd.NA,
    index=public_df.index,
    dtype="Int64"
)

public_df.loc[
    baseline_required_fields,
    "legacy_rtd_baseline"
] = (
    (
        public_df.loc[
            baseline_required_fields,
            "final_severity_num"
        ] >= 5
    )
    & (
        public_df.loc[
            baseline_required_fields,
            "generates_heat"
        ] == 1
    )
).astype("int64")

baseline_summary = (
    public_df["legacy_rtd_baseline"]
    .value_counts(dropna=False)
    .rename_axis("legacy_rtd_baseline")
    .reset_index(name="row_count")
)

print(
    "Legacy rule: recommend RTD when "
    "final severity is C5/C6 and heat is generated."
)

display(baseline_summary)

baseline_positive_rate = (
    public_df["legacy_rtd_baseline"]
    .eq(1)
    .mean()
)

print(
    f"Legacy RTD recommendation rate: "
    f"{baseline_positive_rate:.1%}"
)

In [ ]:
# ============================================================
# 4.6 Verify parity with the spreadsheet LF7 baseline
# ============================================================

lf7_candidates = [
    column
    for column in public_df.columns
    if column.startswith("lf7_")
]

print(f"LF7 reference columns found: {lf7_candidates}")

if lf7_candidates:
    LF7_REFERENCE_COLUMN = lf7_candidates[0]

    spreadsheet_lf7 = pd.to_numeric(
        public_df[LF7_REFERENCE_COLUMN],
        errors="coerce"
    ).astype("Int64")

    parity_mask = (
        public_df["legacy_rtd_baseline"].notna()
        & spreadsheet_lf7.notna()
    )

    mismatch_mask = (
        parity_mask
        & (
            public_df["legacy_rtd_baseline"]
            != spreadsheet_lf7
        )
    )

    print(
        f"Rows compared: {parity_mask.sum()}"
    )
    print(
        f"Python vs spreadsheet LF7 mismatches: "
        f"{mismatch_mask.sum()}"
    )

    if mismatch_mask.any():
        display(
            public_df.loc[
                mismatch_mask,
                [
                    "pair_id",
                    "final_severity_num",
                    "generates_heat",
                    "legacy_rtd_baseline",
                    LF7_REFERENCE_COLUMN,
                ],
            ]
        )
else:
    print(
        "No LF7 reference column was found. "
        "The Python baseline will be treated as the source calculation."
    )

In [ ]:
# ============================================================
# 4.7 Evaluate the legacy baseline against audited decisions
# ============================================================

audit_evaluation_df = public_df.merge(
    audited_gold_df[
        [
            "pair_id",
            "rtd_required",
            "control_failure_mode",
        ]
    ],
    on="pair_id",
    how="inner",
    validate="one_to_one",
)

audit_evaluation_df["rtd_required"] = (
    pd.to_numeric(
        audit_evaluation_df["rtd_required"],
        errors="coerce"
    )
    .astype("Int64")
)

audit_evaluation_df["legacy_rtd_baseline"] = (
    pd.to_numeric(
        audit_evaluation_df["legacy_rtd_baseline"],
        errors="coerce"
    )
    .astype("Int64")
)

valid_evaluation_mask = (
    audit_evaluation_df["rtd_required"].isin([0, 1])
    & audit_evaluation_df[
        "legacy_rtd_baseline"
    ].isin([0, 1])
)

baseline_eval_df = audit_evaluation_df.loc[
    valid_evaluation_mask
].copy()

actual = baseline_eval_df["rtd_required"]
predicted = baseline_eval_df["legacy_rtd_baseline"]

true_positive = int(
    ((actual == 1) & (predicted == 1)).sum()
)
false_positive = int(
    ((actual == 0) & (predicted == 1)).sum()
)
true_negative = int(
    ((actual == 0) & (predicted == 0)).sum()
)
false_negative = int(
    ((actual == 1) & (predicted == 0)).sum()
)


def safe_divide(
    numerator: float,
    denominator: float
) -> float:
    return (
        numerator / denominator
        if denominator != 0
        else np.nan
    )


baseline_precision = safe_divide(
    true_positive,
    true_positive + false_positive
)

baseline_recall = safe_divide(
    true_positive,
    true_positive + false_negative
)

baseline_specificity = safe_divide(
    true_negative,
    true_negative + false_positive
)

baseline_accuracy = safe_divide(
    true_positive + true_negative,
    len(baseline_eval_df)
)

overprescription_rate = safe_divide(
    false_positive,
    true_positive + false_positive
)

confusion_table = pd.DataFrame(
    [
        [true_negative, false_positive],
        [false_negative, true_positive],
    ],
    index=[
        "Audited: No RTD",
        "Audited: RTD Required",
    ],
    columns=[
        "Baseline: No RTD",
        "Baseline: RTD",
    ],
)

metric_table = pd.DataFrame({
    "metric": [
        "Accuracy",
        "RTD precision",
        "RTD recall",
        "No-RTD specificity",
        "RTD over-prescription rate",
    ],
    "value": [
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_specificity,
        overprescription_rate,
    ],
})

print("Legacy baseline confusion table:")
display(confusion_table)

print("\nLegacy baseline metrics:")
display(
    metric_table.style.format({
        "value": "{:.1%}"
    })
)

In [ ]:
# ============================================================
# 4.8 Diagnose baseline errors by control failure mode
# ============================================================

baseline_eval_df["baseline_false_positive"] = (
    (baseline_eval_df["legacy_rtd_baseline"] == 1)
    & (baseline_eval_df["rtd_required"] == 0)
).astype(int)

failure_mode_baseline_summary = (
    baseline_eval_df
    .groupby(
        "control_failure_mode",
        dropna=False
    )
    .agg(
        audited_rows=("pair_id", "count"),
        audited_rtd_required=("rtd_required", "sum"),
        baseline_rtd_recommendations=(
            "legacy_rtd_baseline",
            "sum"
        ),
        false_positive_rtd_recommendations=(
            "baseline_false_positive",
            "sum"
        ),
    )
    .reset_index()
    .sort_values(
        "false_positive_rtd_recommendations",
        ascending=False
    )
)

display(failure_mode_baseline_summary)

print("\nBaseline false-positive rows:")

display(
    baseline_eval_df.loc[
        baseline_eval_df["baseline_false_positive"] == 1,
        [
            "pair_id",
            "chemical_a_class",
            "chemical_b_class",
            "reaction_type",
            "generates_heat",
            "toxic_gas",
            "generates_gas",
            "explosive_potential",
            "final_severity_num",
            "control_failure_mode",
        ],
    ].sort_values(
        [
            "control_failure_mode",
            "pair_id",
        ]
    )
)

In [ ]:
# ============================================================
# 4.4A Strengthen leakage control using chemical-pair groups
# ============================================================

# Rebuild the public audited subset now that canonical_pair_key exists.
public_audited_df = public_df.loc[
    public_df["pair_id"].isin(audited_pair_ids)
].copy()

audited_canonical_pair_keys = set(
    public_audited_df["canonical_pair_key"].dropna()
)

# Identify non-audited rows that share chemistry with an audited row.
related_training_rows = public_df.loc[
    ~public_df["pair_id"].isin(audited_pair_ids)
    & public_df["canonical_pair_key"].isin(
        audited_canonical_pair_keys
    )
].copy()

# Stronger leakage-safe pool:
# exclude the audited rows and all repeated versions of their chemical pairs.
group_safe_training_pool_df = public_df.loc[
    ~public_df["canonical_pair_key"].isin(
        audited_canonical_pair_keys
    )
].copy()

print("Pair-group leakage assessment")
print("-----------------------------")
print(f"Audited pair IDs:                         {len(audited_pair_ids)}")
print(f"Unique audited canonical pair keys:       {len(audited_canonical_pair_keys)}")
print(f"Related non-audited rows also excluded:   {len(related_training_rows)}")
print(f"Original public rows:                     {len(public_df)}")
print(f"Pair-ID-only training pool:               {len(training_pool_df)}")
print(f"Group-safe training pool:                 {len(group_safe_training_pool_df)}")

if not related_training_rows.empty:
    print("\nNon-audited rows sharing chemistry with audited rows:")
    display(
        related_training_rows[
            [
                "pair_id",
                CHEMICAL_A_COLUMN,
                CHEMICAL_B_COLUMN,
                "canonical_pair_key",
                "reaction_type",
                "setting",
                "final_severity_num",
            ]
        ].sort_values(
            ["canonical_pair_key", "pair_id"]
        )
    )

assert not group_safe_training_pool_df[
    "canonical_pair_key"
].isin(audited_canonical_pair_keys).any()

print("\nGroup leakage check passed.")

In [ ]:
# ============================================================
# 4.4B Classify repeated-pair differences
# ============================================================

CONTEXT_COLUMNS = [
    "setting",
]

MECHANISM_AND_HAZARD_COLUMNS = [
    "reaction_type",
    "reaction_speed",
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "corrosive",
    "final_severity_num",
]

repeated_pair_diagnostics = []

for pair_key, group in public_df.groupby("canonical_pair_key"):
    if len(group) <= 1:
        continue

    context_conflicts = [
        column
        for column in CONTEXT_COLUMNS
        if group[column].nunique(dropna=False) > 1
    ]

    mechanism_conflicts = [
        column
        for column in MECHANISM_AND_HAZARD_COLUMNS
        if group[column].nunique(dropna=False) > 1
    ]

    if mechanism_conflicts:
        classification = "Mechanism/hazard inconsistency"
    elif context_conflicts:
        classification = "Context-only variant"
    else:
        classification = "Same modeled inputs"

    repeated_pair_diagnostics.append({
        "canonical_pair_key": pair_key,
        "row_count": len(group),
        "pair_ids": ", ".join(
            group["pair_id"].astype(str).tolist()
        ),
        "classification": classification,
        "context_conflicts": ", ".join(context_conflicts),
        "mechanism_conflicts": ", ".join(mechanism_conflicts),
    })

repeated_pair_diagnostics_df = pd.DataFrame(
    repeated_pair_diagnostics
)

print("Repeated-pair classification:")
display(
    repeated_pair_diagnostics_df[
        "classification"
    ]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="pair_group_count")
)

display(
    repeated_pair_diagnostics_df.sort_values(
        ["classification", "canonical_pair_key"]
    )
)

## Repeated-Pair Data Integrity Decision

Twenty repeated unordered chemical-pair groups contain differences in one or
more mechanism, hazard, context, or severity fields. These records are retained
as separate scenario rows because the differences may reflect directional,
site, exposure, or screening-context assumptions rather than simple duplicates.

The repeated-pair conflict flag is retained for quality assurance but is not
used as a predictive feature. Rows sharing the same canonical chemical-pair key
will remain in the same cross-validation fold. A later sensitivity analysis
will compare model results with and without the conflicting groups before any
manual consolidation or conservative aggregation is considered.

# 5. Spreadsheet Helper-Feature and Labeling-Function Inventory

The public workbook contains reference values calculated in Excel. Before
reimplementing those calculations in Python, this section inventories the
available helper features, labeling functions, and weak-label outputs.

The Excel-derived columns will be used only for parity testing. The final
Python pipeline will rebuild these features from the original structured
inputs so that the notebook does not depend on spreadsheet formulas.

In [ ]:
# ============================================================
# 5.1 Add pair-group QA metadata
# ============================================================

pair_group_size_df = (
    public_df
    .groupby("canonical_pair_key", as_index=False)
    .agg(pair_group_size=("pair_id", "size"))
)

pair_group_classification_df = (
    repeated_pair_diagnostics_df[
        [
            "canonical_pair_key",
            "classification",
        ]
    ]
    .rename(
        columns={
            "classification": "pair_group_classification"
        }
    )
)

# Avoid duplicating columns if this cell is rerun.
columns_to_remove = [
    column
    for column in [
        "pair_group_size",
        "pair_group_classification",
        "pair_group_conflict_flag",
    ]
    if column in public_df.columns
]

if columns_to_remove:
    public_df = public_df.drop(
        columns=columns_to_remove
    )

public_df = (
    public_df
    .merge(
        pair_group_size_df,
        on="canonical_pair_key",
        how="left",
        validate="many_to_one",
    )
    .merge(
        pair_group_classification_df,
        on="canonical_pair_key",
        how="left",
        validate="many_to_one",
    )
)

public_df["pair_group_classification"] = (
    public_df["pair_group_classification"]
    .fillna("Unique chemical pair")
)

public_df["pair_group_conflict_flag"] = (
    public_df["pair_group_classification"]
    .eq("Mechanism/hazard inconsistency")
    .astype("Int64")
)

# Recreate the leakage-safe pool so it includes the QA metadata.
group_safe_training_pool_df = public_df.loc[
    ~public_df["canonical_pair_key"].isin(
        audited_canonical_pair_keys
    )
].copy()

pair_group_row_summary = (
    public_df[
        [
            "pair_group_classification",
            "pair_group_conflict_flag",
        ]
    ]
    .value_counts(dropna=False)
    .rename("row_count")
    .reset_index()
)

display(pair_group_row_summary)

print(
    f"Training-pool rows after metadata update: "
    f"{len(group_safe_training_pool_df)}"
)

In [ ]:
# ============================================================
# 5.2 Inventory spreadsheet-derived columns
# ============================================================

def natural_sort_key(value: str) -> list[object]:
    """Sort strings such as LF2 before LF10."""
    return [
        int(token) if token.isdigit() else token
        for token in re.split(r"(\d+)", value.lower())
    ]


helper_feature_columns = sorted(
    [
        column
        for column in public_df.columns
        if (
            column.startswith("has_")
            or column.startswith("is_")
            or column.endswith("_proxy")
        )
    ],
    key=natural_sort_key,
)

rtd_lf_columns = sorted(
    [
        column
        for column in public_df.columns
        if re.match(r"^lf\d+(?:_|$)", column)
    ],
    key=natural_sort_key,
)

severity_lf_columns = sorted(
    [
        column
        for column in public_df.columns
        if re.match(r"^slf\d+(?:_|$)", column)
    ],
    key=natural_sort_key,
)

weak_output_columns = sorted(
    [
        column
        for column in public_df.columns
        if (
            column.startswith("weak_")
            or "vote_prob" in column
            or "vote_count" in column
            or "conflict_flag" in column
        )
    ],
    key=natural_sort_key,
)

print("Helper-feature columns:")
for column in helper_feature_columns:
    print(f"  - {column}")

print("\nRTD labeling-function columns:")
for column in rtd_lf_columns:
    print(f"  - {column}")

print("\nSeverity review-flag columns:")
for column in severity_lf_columns:
    print(f"  - {column}")

print("\nWeak-label / aggregation output columns:")
for column in weak_output_columns:
    print(f"  - {column}")

In [ ]:
# ============================================================
# 5.3 Summarize spreadsheet LF reference values
# ============================================================

def summarize_lf_columns(
    df: pd.DataFrame,
    columns: list[str]
) -> pd.DataFrame:
    """Summarize votes, abstentions, and unexpected values."""
    records = []

    for column in columns:
        numeric_values = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        nonblank_count = int(
            numeric_values.notna().sum()
        )

        zero_votes = int(
            numeric_values.eq(0).sum()
        )

        one_votes = int(
            numeric_values.eq(1).sum()
        )

        unexpected_count = int(
            (
                numeric_values.notna()
                & ~numeric_values.isin([0, 1])
            ).sum()
        )

        records.append({
            "labeling_function": column,
            "row_count": len(df),
            "vote_count": nonblank_count,
            "abstention_count": len(df) - nonblank_count,
            "no_rtd_votes_0": zero_votes,
            "rtd_votes_1": one_votes,
            "unexpected_numeric_values": unexpected_count,
            "coverage_percent": (
                nonblank_count / len(df) * 100
                if len(df) > 0
                else np.nan
            ),
        })

    return pd.DataFrame(records)


public_rtd_lf_summary = summarize_lf_columns(
    public_df,
    rtd_lf_columns
)

public_severity_lf_summary = summarize_lf_columns(
    public_df,
    severity_lf_columns
)

print("RTD LF reference summary:")
display(
    public_rtd_lf_summary.style.format({
        "coverage_percent": "{:.1f}%"
    })
)

print("\nSeverity-review LF reference summary:")
display(
    public_severity_lf_summary.style.format({
        "coverage_percent": "{:.1f}%"
    })
)

In [ ]:
# ============================================================
# 5.4 Create a stable LF-column map
# ============================================================

def build_numbered_column_map(
    columns: list[str],
    prefix: str
) -> dict[int, str]:
    """Map LF number to the exact normalized dataframe column."""
    result = {}

    pattern = re.compile(
        rf"^{re.escape(prefix)}(\d+)(?:_|$)"
    )

    for column in columns:
        match = pattern.match(column)

        if match:
            number = int(match.group(1))

            if number in result:
                raise ValueError(
                    f"More than one {prefix.upper()}{number} "
                    f"column was found: "
                    f"{result[number]} and {column}"
                )

            result[number] = column

    return result


RTD_LF_COLUMN_MAP = build_numbered_column_map(
    rtd_lf_columns,
    prefix="lf"
)

SEVERITY_LF_COLUMN_MAP = build_numbered_column_map(
    severity_lf_columns,
    prefix="slf"
)

print("RTD LF column map:")
for number, column in sorted(
    RTD_LF_COLUMN_MAP.items()
):
    print(f"  LF{number}: {column}")

print("\nSeverity LF column map:")
for number, column in sorted(
    SEVERITY_LF_COLUMN_MAP.items()
):
    print(f"  SLF{number}: {column}")

In [ ]:
# ============================================================
# 5.5 Define columns that cannot be used as model predictors
# ============================================================

IDENTIFIER_COLUMNS = [
    "pair_id",
    "canonical_pair_key",
]

GROUP_QA_COLUMNS = [
    "pair_group_size",
    "pair_group_classification",
    "pair_group_conflict_flag",
]

SPREADSHEET_DERIVED_COLUMNS = sorted(
    set(
        helper_feature_columns
        + rtd_lf_columns
        + severity_lf_columns
        + weak_output_columns
        + [
            "legacy_rtd_baseline",
        ]
    )
)

MODEL_EXCLUSION_COLUMNS = sorted(
    set(
        IDENTIFIER_COLUMNS
        + GROUP_QA_COLUMNS
        + SPREADSHEET_DERIVED_COLUMNS
    )
)

print(
    f"Columns currently excluded from direct model inputs: "
    f"{len(MODEL_EXCLUSION_COLUMNS)}"
)

for column in MODEL_EXCLUSION_COLUMNS:
    if column in public_df.columns:
        print(f"  - {column}")

# 6. Python Recreation of Helper Features and Labeling Functions

The Section 5 inventory provides the reference targets for the Python implementation.

## Labeling-function roles

### Positive RTD evidence

- **LF2 — Bulk-Liquid Neutralization:** Votes for RTD when a severe, heat-generating acid/base reaction resembles a clean bulk-liquid exotherm.
- **LF9 — Controlled Thermal Default:** Votes for RTD when severe heat generation is present and no active failure-mode proxy indicates that RTD would be ineffective.

### Negative RTD evidence

- **LF1 — No Heat or Lower Severity:** Votes against RTD when the scenario is below C5 or does not generate heat.
- **LF3 — Toxic-Gas Dominant:** Votes against RTD when toxic gas or vapor release is the dominant hazard.
- **LF4 — Detectability Failure:** Votes against RTD when gel, sludge, precipitation, phase separation, or fouling may prevent reliable detection.
- **LF5 — Fast Pressure or Decomposition:** Votes against RTD when the event may progress too quickly or is dominated by gas or pressure generation.
- **LF6 — Oxidizer-Organic Candidate:** Candidate no-RTD vote for fire- or decomposition-dominant oxidizer/organic scenarios.
- **LF8 — Diluent or Lower-Severity Candidate:** Candidate no-RTD vote for lower-severity or diluent-containing scenarios.

### Baseline comparator

- **LF7 — Severe + Heat Baseline:** Represents the legacy rule that recommends RTD whenever severity is C5/C6 and heat is generated. It is retained for comparison but excluded from the final mechanism-aware vote.

## Spreadsheet reference coverage

| Labeling Function | Vote Type | Rows Receiving a Vote | Coverage |
|---|---|---:|---:|
| LF1 | No RTD | 196 | 62.6% |
| LF2 | RTD | 4 | 1.3% |
| LF3 | No RTD | 19 | 6.1% |
| LF4 | No RTD | 33 | 10.5% |
| LF5 | No RTD | 6 | 1.9% |
| LF6 | No RTD candidate | 38 | 12.1% |
| LF7 | RTD baseline only | 117 | 37.4% |
| LF8 | No RTD candidate | 193 | 61.7% |
| LF9 | RTD | 41 | 13.1% |

## Key observations

- LF7 matches the Python legacy baseline exactly: **117 RTD recommendations**.
- LF9 provides **41 controlled positive RTD votes**, compared with LF7's broader 117 recommendations.
- LF2 is intentionally narrow and provides four positive votes.
- LF1 and LF8 overlap substantially. Their combined influence will later be tested to avoid counting similar negative evidence twice.
- The severity-demotion review flags are sparse:
  - SLF1 votes on 6 rows.
  - SLF2 votes on 5 rows.
- No RTD labeling function contains unexpected numeric values.
- The severity-demotion columns remain engineering-review flags, not ordinary predictive features.

## Purpose of this section

The spreadsheet-derived helper features and LF outputs are used only as quality-assurance references.

The Python pipeline will:

1. Recreate helper features from the original structured inputs.
2. Recreate LF1 through LF9 using explicit Python logic.
3. Preserve abstentions as missing votes rather than treating them as zero.
4. Compare every Python-generated result with the spreadsheet reference.
5. Resolve any parity differences before weak-label aggregation or model training begins.

In [ ]:
# ============================================================
# 6.1 Inspect class vocabulary
# ============================================================

available_class_values = sorted(
    set(
        public_df["chemical_a_class"].dropna().tolist()
        + public_df["chemical_b_class"].dropna().tolist()
    )
)

print(f"Unique public chemical classes: {len(available_class_values)}")

for chemical_class in available_class_values:
    print(f"  - {chemical_class}")

In [ ]:
# ============================================================
# 6.1 Inspect class vocabulary
# ============================================================

available_class_values = sorted(
    set(
        public_df["chemical_a_class"].dropna().tolist()
        + public_df["chemical_b_class"].dropna().tolist()
    )
)

print(f"Unique public chemical classes: {len(available_class_values)}")

for chemical_class in available_class_values:
    print(f"  - {chemical_class}")

In [ ]:
# ============================================================
# 6.2 Define class groups used by the engineered proxies
# ============================================================

ACID_CLASSES = {
    "mineral_acid",
    "oxidizing_acid",
}

BASE_CLASSES = {
    "alkali_base",
    "ammoniacal_base",
    "alkanolamine",
}

ACID_OR_ACIDIC_CLASSES = {
    "mineral_acid",
    "oxidizing_acid",
    "acidic_anionic_surfactant",
}

# Match the spreadsheet helper formula exactly.  Quaternary ammonium remains in
# toxic-hypochlorite and oxidizer-organic logic, but it is not included in the
# has_ammoniacal_or_amine helper column in the workbook snapshot.
AMMONIACAL_OR_AMINE_CLASSES = {
    "ammoniacal_base",
    "alkanolamine",
    "amine_oxide_surfactant",
    "metal_ammonium_complex",
}

TOXIC_HYPOCHLORITE_PARTNER_CLASSES = {
    "mineral_acid",
    "oxidizing_acid",
    "acidic_anionic_surfactant",
    "ammoniacal_base",
    "alkanolamine",
    "amine_oxide_surfactant",
    "quaternary_ammonium",
    "metal_ammonium_complex",
    "metal_salt",
}

METAL_OR_COMPLEX_CLASSES = {
    "metal_salt",
    "metal_ammonium_complex",
}

POLYMER_OR_EMULSION_WAX_CLASSES = {
    "acrylic_polymer_dispersant",
    "proprietary_polymer_additive",
    "proprietary_polymer_dispersion",
    "proprietary_polymer",
    "polymer_emulsion",
    "polypropylene_emulsion",
    "nonionic_wax_emulsion",
}

STRONGLY_REACTIVE_CLASSES = {
    "mineral_acid",
    "oxidizing_acid",
    "acidic_anionic_surfactant",
    "alkali_base",
    "ammoniacal_base",
    "alkanolamine",
    "chlorinated_oxidizer",
    "inorganic_oxidizer",
}

CLEAR_BULK_EXOTHERM_EXCLUSIONS = {
    "alkaline_silicate",
    "metal_salt",
    "metal_ammonium_complex",
    "chelating_agent",
    "acrylic_polymer_dispersant",
    "proprietary_polymer_additive",
    "proprietary_polymer_dispersion",
    "proprietary_polymer",
    "polymer_emulsion",
    "polypropylene_emulsion",
    "nonionic_wax_emulsion",
    "chlorinated_oxidizer",
    "inorganic_oxidizer",
    "water_diluent",
    "acidic_anionic_surfactant",
}

OXIDIZER_CLASSES = {
    "chlorinated_oxidizer",
    "inorganic_oxidizer",
    "oxidizing_acid",
}

ORGANIC_FUEL_CLASSES = {
    "glycol_ether_solvent",
    "aromatic_alcohol",
    "phosphate_ester",
    "nonionic_surfactant",
    "anionic_surfactant",
    "amphoteric_surfactant",
    "amine_oxide_surfactant",
    "aromatic_sulfonate_hydrotrope",
    "acidic_anionic_surfactant",
    "chelating_agent",
    "acrylic_polymer_dispersant",
    "proprietary_polymer_additive",
    "proprietary_polymer_dispersion",
    "proprietary_polymer",
    "polymer_emulsion",
    "polypropylene_emulsion",
    "nonionic_wax_emulsion",
    "quaternary_ammonium",
}

ANIONIC_OR_ACIDIC_SURFACTANT_CLASSES = {
    "acidic_anionic_surfactant",
    "anionic_surfactant",
}

DEFINED_CLASS_VALUES = (
    ACID_CLASSES
    | BASE_CLASSES
    | ACID_OR_ACIDIC_CLASSES
    | AMMONIACAL_OR_AMINE_CLASSES
    | TOXIC_HYPOCHLORITE_PARTNER_CLASSES
    | METAL_OR_COMPLEX_CLASSES
    | POLYMER_OR_EMULSION_WAX_CLASSES
    | STRONGLY_REACTIVE_CLASSES
    | CLEAR_BULK_EXOTHERM_EXCLUSIONS
    | OXIDIZER_CLASSES
    | ORGANIC_FUEL_CLASSES
    | ANIONIC_OR_ACIDIC_SURFACTANT_CLASSES
    | {
        "alkaline_silicate",
        "chelating_agent",
        "water_diluent",
    }
)

unknown_constants = sorted(
    DEFINED_CLASS_VALUES - set(available_class_values)
)

print("Class groups defined.")

if unknown_constants:
    print(
        "\nDefined values not currently present in the public dataset "
        "(retained for future pipeline robustness):"
    )

    for value in unknown_constants:
        print(f"  - {value}")

In [ ]:
# ============================================================
# 6.3 Recreate engineered helper features
# ============================================================

def contains_any_class(
    df: pd.DataFrame,
    class_values: set[str],
) -> pd.Series:
    """
    Return True when Chemical A or Chemical B belongs to
    any class in class_values.
    """
    return (
        df["chemical_a_class"].isin(class_values)
        | df["chemical_b_class"].isin(class_values)
    )


def pair_matches_class_sets(
    df: pd.DataFrame,
    left_classes: set[str],
    right_classes: set[str],
) -> pd.Series:
    """
    Match an unordered chemical pair against two class sets.

    A+B and B+A are treated as the same class pairing.
    """
    forward = (
        df["chemical_a_class"].isin(left_classes)
        & df["chemical_b_class"].isin(right_classes)
    )

    reverse = (
        df["chemical_a_class"].isin(right_classes)
        & df["chemical_b_class"].isin(left_classes)
    )

    return forward | reverse


python_df = public_df.copy()

# ------------------------------------------------------------
# Simple chemical-class presence indicators
# ------------------------------------------------------------

python_df["py_has_alkali_base"] = (
    contains_any_class(
        python_df,
        {"alkali_base"},
    ).astype("Int64")
)

python_df["py_has_ammoniacal_or_amine"] = (
    contains_any_class(
        python_df,
        AMMONIACAL_OR_AMINE_CLASSES,
    ).astype("Int64")
)

python_df["py_has_chelant"] = (
    contains_any_class(
        python_df,
        {"chelating_agent"},
    ).astype("Int64")
)

python_df["py_has_chlorinated_oxidizer"] = (
    contains_any_class(
        python_df,
        {"chlorinated_oxidizer"},
    ).astype("Int64")
)

python_df["py_has_diluent"] = (
    contains_any_class(
        python_df,
        {"water_diluent"},
    ).astype("Int64")
)

python_df["py_has_inorganic_oxidizer"] = (
    contains_any_class(
        python_df,
        {"inorganic_oxidizer"},
    ).astype("Int64")
)

python_df["py_has_metal_or_complex"] = (
    contains_any_class(
        python_df,
        METAL_OR_COMPLEX_CLASSES,
    ).astype("Int64")
)

python_df["py_has_mineral_acid"] = (
    contains_any_class(
        python_df,
        {"mineral_acid"},
    ).astype("Int64")
)

python_df["py_has_oxidizing_acid"] = (
    contains_any_class(
        python_df,
        {"oxidizing_acid"},
    ).astype("Int64")
)

python_df["py_has_polymer_or_emulsion_wax"] = (
    contains_any_class(
        python_df,
        POLYMER_OR_EMULSION_WAX_CLASSES,
    ).astype("Int64")
)

python_df["py_has_silicate"] = (
    contains_any_class(
        python_df,
        {"alkaline_silicate"},
    ).astype("Int64")
)

# ------------------------------------------------------------
# Broad context indicators
# ------------------------------------------------------------

python_df["py_is_high_severity"] = (
    python_df["final_severity_num"]
    .ge(5)
    .astype("Int64")
)

python_df["py_is_acid_base"] = (
    python_df["reaction_type"]
    .eq("acid_base_neutralization")
    .astype("Int64")
)

oxidizer_organic_class_pair = pair_matches_class_sets(
    python_df,
    OXIDIZER_CLASSES,
    ORGANIC_FUEL_CLASSES,
)

python_df["py_is_oxidizer_organic"] = (
    python_df["reaction_type"].eq(
        "oxidizer_plus_organic_reducer"
    )
    & oxidizer_organic_class_pair
).astype("Int64")

# ------------------------------------------------------------
# Clear bulk-liquid exotherm proxy
# ------------------------------------------------------------

acid_base_class_pair = pair_matches_class_sets(
    python_df,
    ACID_CLASSES,
    BASE_CLASSES,
)

has_clear_bulk_exclusion = contains_any_class(
    python_df,
    CLEAR_BULK_EXOTHERM_EXCLUSIONS,
)

python_df["py_has_clear_bulk_exotherm_proxy"] = (
    acid_base_class_pair
    & python_df["reaction_type"].eq(
        "acid_base_neutralization"
    )
    & ~has_clear_bulk_exclusion
).astype("Int64")

# ------------------------------------------------------------
# Toxic hypochlorite or chloramine pattern
# ------------------------------------------------------------

chlorinated_toxic_pair = pair_matches_class_sets(
    python_df,
    {"chlorinated_oxidizer"},
    TOXIC_HYPOCHLORITE_PARTNER_CLASSES,
)

python_df["py_is_hypochlorite_toxic_pattern"] = (
    python_df["reaction_type"].isin(
        {
            "hypochlorite_plus_acid",
            "oxidizer_plus_ammonia_amine",
        }
    )
    | chlorinated_toxic_pair
).astype("Int64")

# ------------------------------------------------------------
# Detectability-failure proxy
# ------------------------------------------------------------

silicate_acid_pair = pair_matches_class_sets(
    python_df,
    {"alkaline_silicate"},
    ACID_OR_ACIDIC_CLASSES,
)

metal_chelant_pair = pair_matches_class_sets(
    python_df,
    METAL_OR_COMPLEX_CLASSES,
    {"chelating_agent"},
)

metal_base_pair = pair_matches_class_sets(
    python_df,
    METAL_OR_COMPLEX_CLASSES,
    BASE_CLASSES,
)

polymer_reactive_pair = pair_matches_class_sets(
    python_df,
    POLYMER_OR_EMULSION_WAX_CLASSES,
    STRONGLY_REACTIVE_CLASSES,
)

anionic_base_pair = pair_matches_class_sets(
    python_df,
    ANIONIC_OR_ACIDIC_SURFACTANT_CLASSES,
    BASE_CLASSES,
)

python_df["py_has_detectability_failure_proxy"] = (
    silicate_acid_pair
    | metal_chelant_pair
    | metal_base_pair
    | polymer_reactive_pair
    | anionic_base_pair
).astype("Int64")

# ------------------------------------------------------------
# Fast pressure or decomposition proxy
# ------------------------------------------------------------

peroxide_base_pair = pair_matches_class_sets(
    python_df,
    {"inorganic_oxidizer"},
    {"alkali_base"},
)

oxidizer_oxidizing_acid_pair = pair_matches_class_sets(
    python_df,
    {"inorganic_oxidizer"},
    {"oxidizing_acid"},
)

hypochlorite_metal_pair = pair_matches_class_sets(
    python_df,
    {"chlorinated_oxidizer"},
    METAL_OR_COMPLEX_CLASSES,
)

python_df["py_has_fast_pressure_decomp_proxy"] = (
    peroxide_base_pair
    | oxidizer_oxidizing_acid_pair
    | hypochlorite_metal_pair
).astype("Int64")

# ------------------------------------------------------------
# Oxidizer-organic fire or decomposition proxy
# ------------------------------------------------------------

python_df["py_has_oxidizer_organic_fire_proxy"] = (
    python_df["reaction_type"].eq(
        "oxidizer_plus_organic_reducer"
    )
    & oxidizer_organic_class_pair
    & (
        python_df["flammable"].eq(1)
        | python_df["explosive_potential"].eq(1)
    )
).astype("Int64")

print("Python helper features created.")

In [ ]:
# ============================================================
# 6.4 Compare Python helper features with Excel references
# ============================================================

HELPER_PARITY_MAP = {
    "has_alkali_base":
        "py_has_alkali_base",

    "has_ammoniacal_or_amine":
        "py_has_ammoniacal_or_amine",

    "has_chelant":
        "py_has_chelant",

    "has_chlorinated_oxidizer":
        "py_has_chlorinated_oxidizer",

    "has_clear_bulk_exotherm_proxy":
        "py_has_clear_bulk_exotherm_proxy",

    "has_detectability_failure_proxy":
        "py_has_detectability_failure_proxy",

    "has_diluent":
        "py_has_diluent",

    "has_fast_pressure_decomp_proxy":
        "py_has_fast_pressure_decomp_proxy",

    "has_inorganic_oxidizer":
        "py_has_inorganic_oxidizer",

    "has_metal_or_complex":
        "py_has_metal_or_complex",

    "has_mineral_acid":
        "py_has_mineral_acid",

    "has_oxidizer_organic_fire_proxy":
        "py_has_oxidizer_organic_fire_proxy",

    "has_oxidizing_acid":
        "py_has_oxidizing_acid",

    "has_polymer_or_emulsion_wax":
        "py_has_polymer_or_emulsion_wax",

    "has_silicate":
        "py_has_silicate",

    "is_acid_base":
        "py_is_acid_base",

    "is_high_severity":
        "py_is_high_severity",

    "is_hypochlorite_toxic_pattern":
        "py_is_hypochlorite_toxic_pattern",

    "is_oxidizer_organic":
        "py_is_oxidizer_organic",
}

helper_parity_records = []

for excel_column, python_column in HELPER_PARITY_MAP.items():

    if excel_column not in python_df.columns:
        helper_parity_records.append({
            "excel_column": excel_column,
            "python_column": python_column,
            "rows_compared": 0,
            "mismatch_count": np.nan,
            "match_percent": np.nan,
            "status": "Excel reference missing",
        })

        continue

    excel_values = pd.to_numeric(
        python_df[excel_column],
        errors="coerce",
    ).astype("Int64")

    python_values = pd.to_numeric(
        python_df[python_column],
        errors="coerce",
    ).astype("Int64")

    comparison_mask = (
        excel_values.notna()
        & python_values.notna()
    )

    mismatch_mask = (
        comparison_mask
        & excel_values.ne(python_values)
    )

    rows_compared = int(comparison_mask.sum())
    mismatch_count = int(mismatch_mask.sum())

    match_percent = (
        (rows_compared - mismatch_count) / rows_compared
        if rows_compared > 0
        else np.nan
    )

    helper_parity_records.append({
        "excel_column": excel_column,
        "python_column": python_column,
        "rows_compared": rows_compared,
        "mismatch_count": mismatch_count,
        "match_percent": match_percent,
        "status": (
            "PASS"
            if mismatch_count == 0
            else "REVIEW"
        ),
    })

helper_parity_summary = pd.DataFrame(
    helper_parity_records
)

display(
    helper_parity_summary.style.format({
        "match_percent": "{:.1%}"
    })
)

total_helper_mismatches = int(
    helper_parity_summary[
        "mismatch_count"
    ].fillna(0).sum()
)

print(
    f"Total helper-feature mismatches: "
    f"{total_helper_mismatches}"
)

In [ ]:
# ============================================================
# 6.5 Define labeling-function vote utility
# ============================================================

ABSTAIN = pd.NA


def make_lf_vote(
    condition: pd.Series,
    vote: int,
) -> pd.Series:
    """
    Return a nullable integer labeling-function vote.

    Values:
      1    = RTD required
      0    = RTD not required
      <NA> = abstain
    """
    if vote not in {0, 1}:
        raise ValueError(
            "LF vote must be either 0 or 1."
        )

    result = pd.Series(
        pd.NA,
        index=condition.index,
        dtype="Int64",
    )

    result.loc[
        condition.fillna(False)
    ] = vote

    return result

In [ ]:
# ============================================================
# 6.6 Recreate RTD labeling functions
# ============================================================

high_severity = (
    python_df["final_severity_num"].ge(5)
)

low_severity = (
    python_df["final_severity_num"].lt(5)
)

heat_present = (
    python_df["generates_heat"].eq(1)
)

no_heat = (
    python_df["generates_heat"].eq(0)
)

# LF1 — No heat or lower severity
python_df["py_lf1_noheat_or_lowseverity_nortd"] = (
    make_lf_vote(
        low_severity | no_heat,
        vote=0,
    )
)

# LF2 — Clean bulk-liquid acid/base exotherm
python_df["py_lf2_bulkliquid_neutralization_rtd"] = (
    make_lf_vote(
        high_severity
        & heat_present
        & python_df[
            "py_has_clear_bulk_exotherm_proxy"
        ].eq(1),
        vote=1,
    )
)

# LF3 — Toxic-gas or hypochlorite-dominant mechanism
python_df["py_lf3_toxicgas_dominant_nortd"] = (
    make_lf_vote(
        high_severity
        & python_df["toxic_gas"].eq(1)
        & python_df[
            "py_is_hypochlorite_toxic_pattern"
        ].eq(1),
        vote=0,
    )
)

# LF4 — Gel, sludge, precipitation, or fouling proxy
python_df["py_lf4_detectabilityfailure_nortd"] = (
    make_lf_vote(
        high_severity
        & heat_present
        & python_df[
            "py_has_detectability_failure_proxy"
        ].eq(1),
        vote=0,
    )
)

# LF5 — Fast gas, pressure, or decomposition mechanism
python_df["py_lf5_fastpressure_decomp_nortd"] = (
    make_lf_vote(
        high_severity
        & python_df["generates_gas"].eq(1)
        & python_df["explosive_potential"].eq(1)
        & python_df[
            "py_has_fast_pressure_decomp_proxy"
        ].eq(1),
        vote=0,
    )
)

# LF6 — Oxidizer-organic fire/decomposition candidate
python_df["py_lf6_oxidizerorganic_nortd_candidate"] = (
    make_lf_vote(
        high_severity
        & python_df[
            "py_has_oxidizer_organic_fire_proxy"
        ].eq(1)
        & (
            python_df["flammable"].eq(1)
            | python_df["explosive_potential"].eq(1)
        ),
        vote=0,
    )
)

# LF7 — Legacy severe-plus-heat comparator only
python_df["py_lf7_baseline_severeheat_rtd"] = (
    make_lf_vote(
        high_severity & heat_present,
        vote=1,
    )
)

# LF8 — Lower severity or diluent candidate
python_df["py_lf8_diluent_nortd_candidate"] = (
    make_lf_vote(
        low_severity
        | (
            python_df["py_has_diluent"].eq(1)
            & python_df["toxic_gas"].eq(0)
            & python_df["explosive_potential"].eq(0)
        ),
        vote=0,
    )
)

# LF9 — Controlled thermal default
negative_failure_votes_absent = (
    python_df[
        "py_lf3_toxicgas_dominant_nortd"
    ].isna()
    & python_df[
        "py_lf4_detectabilityfailure_nortd"
    ].isna()
    & python_df[
        "py_lf5_fastpressure_decomp_nortd"
    ].isna()
    & python_df[
        "py_lf6_oxidizerorganic_nortd_candidate"
    ].isna()
    & python_df[
        "py_lf8_diluent_nortd_candidate"
    ].isna()
)

python_df["py_lf9_thermal_default_rtd"] = (
    make_lf_vote(
        high_severity
        & heat_present
        & negative_failure_votes_absent,
        vote=1,
    )
)

print("Python RTD labeling functions created.")

In [ ]:
# ============================================================
# 6.7 Compare Python LF votes with Excel reference values
# ============================================================

LF_PARITY_MAP = {
    RTD_LF_COLUMN_MAP[1]:
        "py_lf1_noheat_or_lowseverity_nortd",

    RTD_LF_COLUMN_MAP[2]:
        "py_lf2_bulkliquid_neutralization_rtd",

    RTD_LF_COLUMN_MAP[3]:
        "py_lf3_toxicgas_dominant_nortd",

    RTD_LF_COLUMN_MAP[4]:
        "py_lf4_detectabilityfailure_nortd",

    RTD_LF_COLUMN_MAP[5]:
        "py_lf5_fastpressure_decomp_nortd",

    RTD_LF_COLUMN_MAP[6]:
        "py_lf6_oxidizerorganic_nortd_candidate",

    RTD_LF_COLUMN_MAP[7]:
        "py_lf7_baseline_severeheat_rtd",

    RTD_LF_COLUMN_MAP[8]:
        "py_lf8_diluent_nortd_candidate",

    RTD_LF_COLUMN_MAP[9]:
        "py_lf9_thermal_default_rtd",
}

lf_parity_records = []
lf_mismatch_frames = []

for excel_column, python_column in LF_PARITY_MAP.items():

    excel_values = pd.to_numeric(
        python_df[excel_column],
        errors="coerce",
    ).astype("Int64")

    python_values = pd.to_numeric(
        python_df[python_column],
        errors="coerce",
    ).astype("Int64")

    mismatch_mask = (
        excel_values.fillna(-1)
        != python_values.fillna(-1)
    )

    mismatch_count = int(
        mismatch_mask.sum()
    )

    match_percent = (
        1 - mismatch_count / len(python_df)
    )

    lf_parity_records.append({
        "excel_lf": excel_column,
        "python_lf": python_column,
        "excel_vote_count": int(
            excel_values.notna().sum()
        ),
        "python_vote_count": int(
            python_values.notna().sum()
        ),
        "mismatch_count": mismatch_count,
        "match_percent": match_percent,
        "status": (
            "PASS"
            if mismatch_count == 0
            else "REVIEW"
        ),
    })

    if mismatch_count > 0:

        mismatch_df = python_df.loc[
            mismatch_mask,
            [
                "pair_id",
                "chemical_a_class",
                "chemical_b_class",
                "reaction_type",
                "final_severity_num",
                "generates_heat",
                "generates_gas",
                "toxic_gas",
                "explosive_potential",
                excel_column,
                python_column,
            ],
        ].copy()

        mismatch_df.insert(
            0,
            "lf_name",
            excel_column,
        )

        lf_mismatch_frames.append(
            mismatch_df
        )

lf_parity_summary = pd.DataFrame(
    lf_parity_records
)

display(
    lf_parity_summary.style.format({
        "match_percent": "{:.1%}"
    })
)

total_lf_mismatches = int(
    lf_parity_summary[
        "mismatch_count"
    ].sum()
)

print(
    f"Total RTD LF mismatches: "
    f"{total_lf_mismatches}"
)

if lf_mismatch_frames:

    lf_mismatch_detail = pd.concat(
        lf_mismatch_frames,
        ignore_index=True,
    )

    print("\nFirst 30 LF mismatches:")

    display(
        lf_mismatch_detail.head(30)
    )

else:

    lf_mismatch_detail = pd.DataFrame()

    print(
        "All Python LF outputs match "
        "the Excel reference values."
    )

# 7. Weak RTD Label Aggregation

Sections 5 and 6 reproduced the spreadsheet helper features and labeling-function votes in Python. This section converts those individual LF votes into a single weak RTD recommendation.

The aggregation logic follows the Phase 1 MVP framing:

- Active labeling functions vote only when their engineering conditions apply.
- Abstentions are ignored rather than treated as negative votes.
- LF7, the broad severe-plus-heat baseline, is retained only as a comparator because it over-recommends RTDs.
- Conflicting positive and negative votes are routed to Review rather than forced into a binary decision.
- The output is a weak label for model development, not a final engineering authorization.

The goal is to create a reproducible Python-side weak target that can later be used for tabular model training while preserving conservative review routing for ambiguous cases.

In [ ]:
# ============================================================
# 7. Weak RTD Label Aggregation
# ============================================================

# Active v1 voting LFs:
#   LF1-LF6, LF8, LF9
#
# Comparator only:
#   LF7 legacy severe + heat baseline
#
# Abstentions are ignored. LF7 is deliberately excluded from
# the final weak label because it over-recommends RTDs.

ACTIVE_RTD_LF_COLUMNS = [
    "py_lf1_noheat_or_lowseverity_nortd",
    "py_lf2_bulkliquid_neutralization_rtd",
    "py_lf3_toxicgas_dominant_nortd",
    "py_lf4_detectabilityfailure_nortd",
    "py_lf5_fastpressure_decomp_nortd",
    "py_lf6_oxidizerorganic_nortd_candidate",
    "py_lf8_diluent_nortd_candidate",
    "py_lf9_thermal_default_rtd",
]

COMPARATOR_RTD_LF_COLUMNS = [
    "py_lf7_baseline_severeheat_rtd",
]

RTD_LABEL_POSITIVE_THRESHOLD = 0.67
RTD_LABEL_NEGATIVE_THRESHOLD = 0.33

missing_active_lfs = [
    column for column in ACTIVE_RTD_LF_COLUMNS
    if column not in python_df.columns
]

if missing_active_lfs:
    raise KeyError(
        "Missing expected active RTD LF columns: "
        f"{missing_active_lfs}"
    )

lf_role_table = pd.DataFrame({
    "lf_column": ACTIVE_RTD_LF_COLUMNS + COMPARATOR_RTD_LF_COLUMNS,
    "role": ["active_vote"] * len(ACTIVE_RTD_LF_COLUMNS) + ["comparator_only"],
    "included_in_weak_label": [True] * len(ACTIVE_RTD_LF_COLUMNS) + [False],
})

print("Frozen RTD LF set for v1 aggregation:")
display(lf_role_table)

active_vote_df = python_df[ACTIVE_RTD_LF_COLUMNS].apply(
    pd.to_numeric,
    errors="coerce",
)

python_df["py_weak_vote_count"] = (
    active_vote_df.notna().sum(axis=1).astype("Int64")
)

python_df["py_weak_rtd_positive_votes"] = (
    active_vote_df.eq(1).sum(axis=1).astype("Int64")
)

python_df["py_weak_rtd_negative_votes"] = (
    active_vote_df.eq(0).sum(axis=1).astype("Int64")
)

python_df["py_weak_vote_prob"] = pd.Series(
    np.nan,
    index=python_df.index,
    dtype="float64",
)

voted_mask = python_df["py_weak_vote_count"].gt(0)

python_df.loc[voted_mask, "py_weak_vote_prob"] = (
    python_df.loc[
        voted_mask,
        "py_weak_rtd_positive_votes",
    ].astype(float)
    / python_df.loc[
        voted_mask,
        "py_weak_vote_count",
    ].astype(float)
)

python_df["py_lf_conflict_flag"] = (
    python_df["py_weak_rtd_positive_votes"].gt(0)
    & python_df["py_weak_rtd_negative_votes"].gt(0)
).astype("Int64")

python_df["py_weak_rtd_label"] = pd.Series(
    pd.NA,
    index=python_df.index,
    dtype="Int64",
)

confident_positive_mask = (
    python_df["py_weak_vote_count"].gt(0)
    & python_df["py_lf_conflict_flag"].eq(0)
    & python_df["py_weak_vote_prob"].ge(RTD_LABEL_POSITIVE_THRESHOLD)
)

confident_negative_mask = (
    python_df["py_weak_vote_count"].gt(0)
    & python_df["py_lf_conflict_flag"].eq(0)
    & python_df["py_weak_vote_prob"].le(RTD_LABEL_NEGATIVE_THRESHOLD)
)

python_df.loc[
    confident_positive_mask,
    "py_weak_rtd_label",
] = 1

python_df.loc[
    confident_negative_mask,
    "py_weak_rtd_label",
] = 0

python_df["py_weak_rtd_decision"] = (
    "Review - mixed or insufficient LF evidence"
)

python_df.loc[
    python_df["py_weak_vote_count"].eq(0),
    "py_weak_rtd_decision",
] = "Review - no active LF votes"

python_df.loc[
    python_df["py_lf_conflict_flag"].eq(1),
    "py_weak_rtd_decision",
] = "Review - active LF conflict"

python_df.loc[
    confident_positive_mask,
    "py_weak_rtd_decision",
] = "RTD required"

python_df.loc[
    confident_negative_mask,
    "py_weak_rtd_decision",
] = "RTD not required"

weak_label_summary = (
    python_df[
        [
            "py_weak_rtd_decision",
            "py_weak_rtd_label",
        ]
    ]
    .value_counts(dropna=False)
    .rename("row_count")
    .reset_index()
    .sort_values(
        ["py_weak_rtd_decision", "py_weak_rtd_label"],
        na_position="last",
    )
)

vote_count_summary = (
    python_df["py_weak_vote_count"]
    .value_counts(dropna=False)
    .rename_axis("active_lf_vote_count")
    .reset_index(name="row_count")
    .sort_values("active_lf_vote_count")
)

print("Weak RTD decision distribution:")
display(weak_label_summary)

print("\nActive LF vote-count distribution:")
display(vote_count_summary)

print(
    "\nRows routed to Review: "
    f"{python_df['py_weak_rtd_label'].isna().sum()} "
    f"of {len(python_df)}"
)

## 7.2 Weak-Label Aggregation QA Against Excel Snapshot

Before using the Python-generated weak labels for modeling, this section compares the Python aggregation outputs against the Excel workbook snapshot.

This check is important because the spreadsheet served as the original engineering rule-development environment. The Python notebook should either:

1. match the workbook exactly, or  
2. clearly document any intentional logic changes.

At this stage of the project, exact parity is preferred so that any later model results are traceable to the audited spreadsheet logic.

In [ ]:
# ============================================================
# 7.2 Compare Python weak-label aggregation with Excel snapshot
# ============================================================

WEAK_AGGREGATION_PARITY_MAP = {
    "weak_vote_prob": "py_weak_vote_prob",
    "weak_vote_count": "py_weak_vote_count",
    "weak_rtd_label": "py_weak_rtd_label",
    "lf_conflict_flag": "py_lf_conflict_flag",
}

weak_parity_records = []
weak_mismatch_frames = []

for excel_column, python_column in WEAK_AGGREGATION_PARITY_MAP.items():

    if excel_column not in python_df.columns:
        print(
            f"Skipping parity check for missing Excel column: "
            f"{excel_column}"
        )
        continue

    excel_values = pd.to_numeric(
        python_df[excel_column],
        errors="coerce",
    )

    python_values = pd.to_numeric(
        python_df[python_column],
        errors="coerce",
    )

    # The workbook snapshot leaves lf_conflict_flag blank in many
    # no-conflict rows. Treat blanks as zero for parity checking.
    if excel_column == "lf_conflict_flag":
        excel_compare = excel_values.fillna(0).astype("float64")
        python_compare = python_values.fillna(0).astype("float64")
    else:
        excel_compare = excel_values.fillna(-999999).astype("float64")
        python_compare = python_values.fillna(-999999).astype("float64")

    mismatch_mask = ~np.isclose(
        excel_compare,
        python_compare,
        equal_nan=True,
    )

    mismatch_count = int(mismatch_mask.sum())
    match_percent = 1 - mismatch_count / len(python_df)

    weak_parity_records.append({
        "excel_column": excel_column,
        "python_column": python_column,
        "mismatch_count": mismatch_count,
        "match_percent": match_percent,
        "status": "PASS" if mismatch_count == 0 else "REVIEW",
    })

    if mismatch_count > 0:
        mismatch_detail = python_df.loc[
            mismatch_mask,
            [
                "pair_id",
                "chemical_a_class",
                "chemical_b_class",
                "reaction_type",
                excel_column,
                python_column,
                "py_weak_vote_count",
                "py_weak_rtd_positive_votes",
                "py_weak_rtd_negative_votes",
                "py_weak_rtd_decision",
            ],
        ].copy()

        mismatch_detail.insert(
            0,
            "checked_column",
            excel_column,
        )

        weak_mismatch_frames.append(mismatch_detail)

weak_aggregation_parity_summary = pd.DataFrame(
    weak_parity_records
)

print("Weak-label aggregation parity summary:")
display(
    weak_aggregation_parity_summary.style.format({
        "match_percent": "{:.1%}",
    })
)

weak_aggregation_mismatch_count = int(
    weak_aggregation_parity_summary["mismatch_count"].sum()
)

print(
    "Total weak-aggregation mismatches: "
    f"{weak_aggregation_mismatch_count}"
)

if weak_mismatch_frames:
    weak_aggregation_mismatch_detail = pd.concat(
        weak_mismatch_frames,
        ignore_index=True,
    )

    print("\nFirst 30 weak-aggregation mismatches:")
    display(weak_aggregation_mismatch_detail.head(30))

else:
    weak_aggregation_mismatch_detail = pd.DataFrame()
    print("Python weak-label aggregation matches the Excel snapshot.")

# 8. Gold-Set Diagnostics for Weak RTD Labels

This section evaluates the aggregated weak RTD label against the completed audited gold-set rows.

The audited rows are not treated as an independent final holdout because they were used during labeling-function development and calibration. Instead, they are used as a development diagnostic to answer practical engineering questions:

- Does the weak-label system reduce the over-recommendation seen in the broad severe-plus-heat baseline?
- Are no-RTD decisions aligned with audited control failure modes such as toxic gas dominance, fast pressure/decomposition, or detectability failure?
- Are any RTD-required cases missed?
- Which cases should be routed to Review rather than forced into a binary recommendation?

For process safety credibility, false negatives are the highest-concern error type. False positives may over-scope RTDs, but false negatives could under-recommend a potentially useful safeguard.

In [ ]:
# ============================================================
# 8. Gold-Set Diagnostics for Weak RTD Labels
# ============================================================

# This is a development/calibration diagnostic, not an independent final holdout.

if "safe_divide" not in globals():
    def safe_divide(numerator: float, denominator: float) -> float:
        return numerator / denominator if denominator != 0 else np.nan

audit_columns_from_gold = [
    column for column in [
        "pair_id",
        "rtd_required",
        "control_failure_mode",
        "gold_severity_num",
        "gold_lt_c5",
        "exotherm_present",
        "exotherm_hazardous",
        "runaway_credible",
        "pressure_hazard_credible",
        "rtd_detectable",
        "audit_confidence",
        "audit_notes",
    ]
    if column in audited_gold_df.columns
]

audit_eval_df = python_df.merge(
    audited_gold_df[audit_columns_from_gold],
    on="pair_id",
    how="inner",
    validate="one_to_one",
)

audit_eval_df["rtd_required"] = pd.to_numeric(
    audit_eval_df["rtd_required"],
    errors="coerce",
).astype("Int64")

audit_eval_df["weak_label_matches_audit"] = (
    audit_eval_df["py_weak_rtd_label"].eq(
        audit_eval_df["rtd_required"]
    )
).astype("Int64")

audit_eval_df["weak_false_positive"] = (
    audit_eval_df["py_weak_rtd_label"].eq(1)
    & audit_eval_df["rtd_required"].eq(0)
).astype("Int64")

audit_eval_df["weak_false_negative"] = (
    audit_eval_df["py_weak_rtd_label"].eq(0)
    & audit_eval_df["rtd_required"].eq(1)
).astype("Int64")

audit_eval_df["weak_review_routed"] = (
    audit_eval_df["py_weak_rtd_label"].isna()
).astype("Int64")

valid_audit_eval_mask = (
    audit_eval_df["rtd_required"].isin([0, 1])
    & audit_eval_df["py_weak_rtd_label"].isin([0, 1])
)

classified_audit_eval_df = audit_eval_df.loc[
    valid_audit_eval_mask
].copy()

actual = classified_audit_eval_df["rtd_required"].astype(int)
predicted = classified_audit_eval_df["py_weak_rtd_label"].astype(int)

true_positive = int(((actual == 1) & (predicted == 1)).sum())
false_positive = int(((actual == 0) & (predicted == 1)).sum())
true_negative = int(((actual == 0) & (predicted == 0)).sum())
false_negative = int(((actual == 1) & (predicted == 0)).sum())

weak_confusion_table = pd.DataFrame(
    [
        [true_negative, false_positive],
        [false_negative, true_positive],
    ],
    index=[
        "Audited: No RTD",
        "Audited: RTD Required",
    ],
    columns=[
        "Weak label: No RTD",
        "Weak label: RTD",
    ],
)

weak_metric_table = pd.DataFrame({
    "metric": [
        "Audited rows evaluated",
        "Rows classified by weak label",
        "Rows routed to Review",
        "Accuracy on classified audited rows",
        "RTD precision",
        "RTD recall",
        "No-RTD specificity",
        "False positives",
        "False negatives",
    ],
    "value": [
        len(audit_eval_df),
        len(classified_audit_eval_df),
        int(audit_eval_df["weak_review_routed"].sum()),
        safe_divide(
            true_positive + true_negative,
            len(classified_audit_eval_df),
        ),
        safe_divide(
            true_positive,
            true_positive + false_positive,
        ),
        safe_divide(
            true_positive,
            true_positive + false_negative,
        ),
        safe_divide(
            true_negative,
            true_negative + false_positive,
        ),
        false_positive,
        false_negative,
    ],
})

print(
    "Weak-label audit diagnostic "
    "(development/calibration set, not independent holdout)"
)

print("\nWeak-label confusion table:")
display(weak_confusion_table)

print("\nWeak-label metrics:")
display(weak_metric_table)

rate_metric_names = {
    "Accuracy on classified audited rows",
    "RTD precision",
    "RTD recall",
    "No-RTD specificity",
}

weak_metric_rate_view = weak_metric_table.copy()
weak_metric_rate_view["display_value"] = weak_metric_rate_view.apply(
    lambda row: (
        f"{row['value']:.1%}"
        if row["metric"] in rate_metric_names and pd.notna(row["value"])
        else row["value"]
    ),
    axis=1,
)

display(weak_metric_rate_view[["metric", "display_value"]])

In [ ]:
# ============================================================
# 8.2 Diagnose weak-label behavior by control failure mode
# ============================================================

failure_mode_weak_summary = (
    audit_eval_df
    .groupby("control_failure_mode", dropna=False)
    .agg(
        audited_rows=("pair_id", "count"),
        audited_rtd_required=("rtd_required", "sum"),
        weak_rtd_recommendations=("py_weak_rtd_label", "sum"),
        weak_false_positives=("weak_false_positive", "sum"),
        weak_false_negatives=("weak_false_negative", "sum"),
        weak_review_routed=("weak_review_routed", "sum"),
        mean_weak_vote_prob=("py_weak_vote_prob", "mean"),
    )
    .reset_index()
    .sort_values(
        [
            "weak_false_negatives",
            "weak_false_positives",
            "audited_rows",
        ],
        ascending=[False, False, False],
    )
)

print("Weak-label performance by audited control failure mode:")
display(
    failure_mode_weak_summary.style.format({
        "mean_weak_vote_prob": "{:.2f}",
    })
)

# ============================================================
# 8.3 Row-level audited diagnostic table
# ============================================================

AUDIT_ROW_DISPLAY_COLUMNS = [
    "pair_id",
    CHEMICAL_A_COLUMN,
    "chemical_a_class",
    CHEMICAL_B_COLUMN,
    "chemical_b_class",
    "reaction_type",
    "reaction_speed",
    "setting",
    "final_severity_num",
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "rtd_required",
    "control_failure_mode",
    "py_weak_rtd_label",
    "py_weak_rtd_decision",
    "py_weak_vote_prob",
    "py_weak_vote_count",
    "py_weak_rtd_positive_votes",
    "py_weak_rtd_negative_votes",
    "py_lf1_noheat_or_lowseverity_nortd",
    "py_lf2_bulkliquid_neutralization_rtd",
    "py_lf3_toxicgas_dominant_nortd",
    "py_lf4_detectabilityfailure_nortd",
    "py_lf5_fastpressure_decomp_nortd",
    "py_lf6_oxidizerorganic_nortd_candidate",
    "py_lf8_diluent_nortd_candidate",
    "py_lf9_thermal_default_rtd",
    "weak_label_matches_audit",
    "weak_false_positive",
    "weak_false_negative",
]

AUDIT_ROW_DISPLAY_COLUMNS = [
    column for column in AUDIT_ROW_DISPLAY_COLUMNS
    if column in audit_eval_df.columns
]

audit_row_diagnostics_df = (
    audit_eval_df[AUDIT_ROW_DISPLAY_COLUMNS]
    .sort_values(
        [
            "weak_false_negative",
            "weak_false_positive",
            "control_failure_mode",
            "pair_id",
        ],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

print("Audited row-level weak-label diagnostics:")
display(audit_row_diagnostics_df)

In [ ]:
# ============================================================
# 8.4 Individual LF diagnostics against audited target
# ============================================================

lf_diagnostic_records = []

for lf_column in ACTIVE_RTD_LF_COLUMNS + COMPARATOR_RTD_LF_COLUMNS:

    lf_votes = pd.to_numeric(
        audit_eval_df[lf_column],
        errors="coerce",
    )

    voted = lf_votes.isin([0, 1])
    vote_count = int(voted.sum())

    if vote_count > 0:
        agrees = lf_votes.loc[voted].astype(int).eq(
            audit_eval_df.loc[voted, "rtd_required"].astype(int)
        )
        precision_when_voting = agrees.mean()
        positive_votes = int(lf_votes.loc[voted].eq(1).sum())
        negative_votes = int(lf_votes.loc[voted].eq(0).sum())
    else:
        precision_when_voting = np.nan
        positive_votes = 0
        negative_votes = 0

    lf_diagnostic_records.append({
        "lf_column": lf_column,
        "role": (
            "comparator_only"
            if lf_column in COMPARATOR_RTD_LF_COLUMNS
            else "active_vote"
        ),
        "audited_vote_count": vote_count,
        "audited_coverage": safe_divide(vote_count, len(audit_eval_df)),
        "positive_votes": positive_votes,
        "negative_votes": negative_votes,
        "precision_when_voting": precision_when_voting,
    })

lf_gold_diagnostics_df = pd.DataFrame(lf_diagnostic_records)

print("Individual LF diagnostics against audited RTD target:")
display(
    lf_gold_diagnostics_df.style.format({
        "audited_coverage": "{:.1%}",
        "precision_when_voting": "{:.1%}",
    })
)


## 8.5 Model Development Pool Preparation

The next modeling section should train only on rows that are appropriate for weak supervised model development.

This section prepares three pools:

1. the full leakage safe public training pool,
2. the binary weak labeled model pool, and
3. the Review / excluded pool.

Completed audited rows and their repeated chemical pair groups remain excluded from training. This preserves a cleaner separation between LF development diagnostics and model fitting.

Rows routed to Review are also excluded from the first binary model because the initial model should learn from confident weak labels only. Review cases can later be used for uncertainty analysis, active learning candidate selection, or peer review prioritization.

In [ ]:
# ============================================================
# 8.5 Create model-development pools for the next section
# ============================================================

weak_label_columns_for_model = [
    "pair_id",
    "py_weak_vote_prob",
    "py_weak_vote_count",
    "py_weak_rtd_positive_votes",
    "py_weak_rtd_negative_votes",
    "py_lf_conflict_flag",
    "py_weak_rtd_label",
    "py_weak_rtd_decision",
]

model_pool_df = group_safe_training_pool_df.merge(
    python_df[weak_label_columns_for_model],
    on="pair_id",
    how="left",
    validate="one_to_one",
)

weak_labeled_model_pool_df = model_pool_df.loc[
    model_pool_df["py_weak_rtd_label"].isin([0, 1])
].copy()

review_model_pool_df = model_pool_df.loc[
    model_pool_df["py_weak_rtd_label"].isna()
].copy()

print("Model-development pools prepared:")
print(f"  Group-safe training pool:          {len(model_pool_df)} rows")
print(f"  Weak-labeled binary training rows: {len(weak_labeled_model_pool_df)} rows")
print(f"  Review / excluded rows:            {len(review_model_pool_df)} rows")

print("\nBinary weak-label distribution in model pool:")
display(
    weak_labeled_model_pool_df["py_weak_rtd_label"]
    .value_counts(dropna=False)
    .rename_axis("py_weak_rtd_label")
    .reset_index(name="row_count")
)

assert not weak_labeled_model_pool_df["pair_id"].isin(audited_pair_ids).any()
assert not weak_labeled_model_pool_df["canonical_pair_key"].isin(
    audited_canonical_pair_keys
).any()

print(
    "\nLeakage check passed: no audited pair IDs or audited pair groups "
    "remain in the weak-labeled model pool."
)

## 8.6 LF Calibration Findings and v1.1 Weak Label

The first aggregated weak-label diagnostic improved substantially over the broad LF7 severe-plus-heat baseline, but it still produced several false-positive RTD recommendations and one false negative on the audited development set.

The main finding is that `LF9_Thermal_Default_RTD` remains too permissive. It correctly avoids the blanket behavior of LF7, but it still recommends RTD when no existing negative LF captures a dominant non-thermal failure mode.

The audited rows identify three specific calibration needs:

1. **Oxidizing acid + chelant cases**  
   These rows are high-severity and heat-generating, but the audited failure mode indicates that RTD is not the correct primary safeguard.

2. **Acidic anionic surfactant + glycol ether solvent cases**  
   These rows were also routed to RTD by the thermal default, but the audited failure mode indicates a detectability / wrong safeguard issue.

3. **Acidic anionic surfactant + volatile/ammoniacal base case**  
   The broad detectability-failure proxy incorrectly voted no RTD. The audited judgment indicates that this acid/base neutralization scenario remains RTD-creditable.

This section creates a calibrated v1.1 weak-label layer without overwriting the parity-matched Excel reconstruction. The original `py_lf...` columns remain the spreadsheet-parity baseline. The new `cal_lf...` columns represent the calibrated v1.1 logic to be frozen before model training.

Because these adjustments are informed by the completed audited rows, the 21-row audit set remains a development/calibration set, not an independent final holdout.

In [ ]:
# ============================================================
# 8.6 LF Calibration Findings and v1.1 Weak Label
# ============================================================

# Keep the parity-matched py_lf... columns untouched.
# Create calibrated v1.1 LF columns with a cal_ prefix.

cal_df = python_df.copy()

def class_pair_is(row, class_1, class_2):
    """Order-insensitive class-pair check."""
    pair_classes = {
        str(row.get("chemical_a_class", "")).strip().lower(),
        str(row.get("chemical_b_class", "")).strip().lower(),
    }
    return {
        str(class_1).strip().lower(),
        str(class_2).strip().lower(),
    } == pair_classes


def either_class_in(row, class_values):
    """True if either chemical class is in the supplied class list."""
    normalized_values = {
        str(value).strip().lower()
        for value in class_values
    }

    return (
        str(row.get("chemical_a_class", "")).strip().lower()
        in normalized_values
        or str(row.get("chemical_b_class", "")).strip().lower()
        in normalized_values
    )


# ------------------------------------------------------------
# Start from the parity-matched LFs where possible.
# ------------------------------------------------------------

cal_df["cal_lf1_noheat_or_lowseverity_nortd"] = (
    cal_df["py_lf1_noheat_or_lowseverity_nortd"]
)

cal_df["cal_lf2_bulkliquid_neutralization_rtd"] = (
    cal_df["py_lf2_bulkliquid_neutralization_rtd"]
)

cal_df["cal_lf3_toxicgas_dominant_nortd"] = (
    cal_df["py_lf3_toxicgas_dominant_nortd"]
)

cal_df["cal_lf4_detectabilityfailure_nortd"] = (
    cal_df["py_lf4_detectabilityfailure_nortd"]
)

cal_df["cal_lf5_fastpressure_decomp_nortd"] = (
    cal_df["py_lf5_fastpressure_decomp_nortd"]
)

cal_df["cal_lf6_oxidizerorganic_nortd_candidate"] = (
    cal_df["py_lf6_oxidizerorganic_nortd_candidate"]
)

cal_df["cal_lf8_diluent_nortd_candidate"] = (
    cal_df["py_lf8_diluent_nortd_candidate"]
)


# ------------------------------------------------------------
# Calibration finding 1:
# LF4 was too broad for acidic-anionic surfactant + volatile /
# ammoniacal base neutralization. The audited row indicates this
# pattern can remain RTD-creditable.
# ------------------------------------------------------------

acidic_surfactant_volatile_base_mask = cal_df.apply(
    lambda row: (
        class_pair_is(
            row,
            "acidic_anionic_surfactant",
            "ammoniacal_base",
        )
        and str(row.get("reaction_type", "")).strip().lower()
        == "acid_base_neutralization"
        and int(row.get("generates_heat", 0)) == 1
        and int(row.get("final_severity_num", 0)) >= 5
    ),
    axis=1,
)

cal_df.loc[
    acidic_surfactant_volatile_base_mask,
    "cal_lf4_detectabilityfailure_nortd",
] = pd.NA


# ------------------------------------------------------------
# Calibration finding 2:
# Add no-RTD vote for oxidizing acid + chelant rows where the
# hazard profile indicates non-thermal dominance / wrong
# safeguard type.
# ------------------------------------------------------------

oxidizing_acid_chelant_mask = cal_df.apply(
    lambda row: (
        class_pair_is(
            row,
            "oxidizing_acid",
            "chelating_agent",
        )
        and int(row.get("final_severity_num", 0)) >= 5
        and int(row.get("generates_heat", 0)) == 1
        and (
            int(row.get("toxic_gas", 0)) == 1
            or int(row.get("generates_gas", 0)) == 1
            or int(row.get("explosive_potential", 0)) == 1
            or int(row.get("flammable", 0)) == 1
        )
    ),
    axis=1,
)

cal_df["cal_lf10_oxidizingacid_chelant_nortd"] = pd.Series(
    pd.NA,
    index=cal_df.index,
    dtype="Int64",
)

cal_df.loc[
    oxidizing_acid_chelant_mask,
    "cal_lf10_oxidizingacid_chelant_nortd",
] = 0


# ------------------------------------------------------------
# Calibration finding 3:
# Add no-RTD vote for acidic-anionic surfactant + glycol ether
# solvent cases where the audited failure mode indicates
# detectability / wrong-safeguard concerns.
# ------------------------------------------------------------

acidic_surfactant_glycol_ether_mask = cal_df.apply(
    lambda row: (
        class_pair_is(
            row,
            "acidic_anionic_surfactant",
            "glycol_ether_solvent",
        )
        and int(row.get("final_severity_num", 0)) >= 5
        and int(row.get("generates_heat", 0)) == 1
        and str(row.get("reaction_type", "")).strip().lower()
        in {"other_unknown", "unknown"}
        and (
            int(row.get("toxic_gas", 0)) == 1
            or int(row.get("generates_gas", 0)) == 1
            or int(row.get("explosive_potential", 0)) == 1
            or int(row.get("flammable", 0)) == 1
        )
    ),
    axis=1,
)

cal_df["cal_lf11_acidicsurfactant_glycolether_nortd"] = pd.Series(
    pd.NA,
    index=cal_df.index,
    dtype="Int64",
)

cal_df.loc[
    acidic_surfactant_glycol_ether_mask,
    "cal_lf11_acidicsurfactant_glycolether_nortd",
] = 0


# ------------------------------------------------------------
# Calibration finding 4:
# Add positive RTD vote for severe heat-generating acidic
# anionic surfactant + volatile/ammoniacal base neutralization.
# This is intentionally narrow based on the audited row.
# ------------------------------------------------------------

cal_df["cal_lf12_acidic_surfactant_volatilebase_rtd"] = pd.Series(
    pd.NA,
    index=cal_df.index,
    dtype="Int64",
)

cal_df.loc[
    acidic_surfactant_volatile_base_mask,
    "cal_lf12_acidic_surfactant_volatilebase_rtd",
] = 1


# ------------------------------------------------------------
# Recompute calibrated LF9 thermal default.
#
# Do not simply copy py_lf9 because py_lf9 was calculated using
# the original LF4/LF10/LF11 state. Recompute after calibrated
# negative LFs are known.
# ------------------------------------------------------------

CAL_NEGATIVE_LF_COLUMNS_FOR_LF9 = [
    "cal_lf3_toxicgas_dominant_nortd",
    "cal_lf4_detectabilityfailure_nortd",
    "cal_lf5_fastpressure_decomp_nortd",
    "cal_lf6_oxidizerorganic_nortd_candidate",
    "cal_lf8_diluent_nortd_candidate",
    "cal_lf10_oxidizingacid_chelant_nortd",
    "cal_lf11_acidicsurfactant_glycolether_nortd",
]

cal_negative_vote_df = cal_df[
    CAL_NEGATIVE_LF_COLUMNS_FOR_LF9
].apply(
    pd.to_numeric,
    errors="coerce",
)

has_cal_negative_vote = cal_negative_vote_df.eq(0).any(axis=1)

cal_lf9_positive_mask = (
    cal_df["final_severity_num"].ge(5)
    & cal_df["generates_heat"].eq(1)
    & ~has_cal_negative_vote
)

cal_df["cal_lf9_thermal_default_rtd"] = pd.Series(
    pd.NA,
    index=cal_df.index,
    dtype="Int64",
)

cal_df.loc[
    cal_lf9_positive_mask,
    "cal_lf9_thermal_default_rtd",
] = 1


# ------------------------------------------------------------
# Aggregate calibrated v1.1 LF votes.
# ------------------------------------------------------------

CAL_ACTIVE_RTD_LF_COLUMNS = [
    "cal_lf1_noheat_or_lowseverity_nortd",
    "cal_lf2_bulkliquid_neutralization_rtd",
    "cal_lf3_toxicgas_dominant_nortd",
    "cal_lf4_detectabilityfailure_nortd",
    "cal_lf5_fastpressure_decomp_nortd",
    "cal_lf6_oxidizerorganic_nortd_candidate",
    "cal_lf8_diluent_nortd_candidate",
    "cal_lf10_oxidizingacid_chelant_nortd",
    "cal_lf11_acidicsurfactant_glycolether_nortd",
    "cal_lf12_acidic_surfactant_volatilebase_rtd",
    "cal_lf9_thermal_default_rtd",
]

cal_vote_df = cal_df[CAL_ACTIVE_RTD_LF_COLUMNS].apply(
    pd.to_numeric,
    errors="coerce",
)

cal_df["cal_weak_vote_count"] = (
    cal_vote_df.notna().sum(axis=1).astype("Int64")
)

cal_df["cal_weak_rtd_positive_votes"] = (
    cal_vote_df.eq(1).sum(axis=1).astype("Int64")
)

cal_df["cal_weak_rtd_negative_votes"] = (
    cal_vote_df.eq(0).sum(axis=1).astype("Int64")
)

cal_df["cal_weak_vote_prob"] = pd.Series(
    np.nan,
    index=cal_df.index,
    dtype="float64",
)

cal_voted_mask = cal_df["cal_weak_vote_count"].gt(0)

cal_df.loc[
    cal_voted_mask,
    "cal_weak_vote_prob",
] = (
    cal_df.loc[
        cal_voted_mask,
        "cal_weak_rtd_positive_votes",
    ].astype(float)
    / cal_df.loc[
        cal_voted_mask,
        "cal_weak_vote_count",
    ].astype(float)
)

cal_df["cal_lf_conflict_flag"] = (
    cal_df["cal_weak_rtd_positive_votes"].gt(0)
    & cal_df["cal_weak_rtd_negative_votes"].gt(0)
).astype("Int64")

cal_df["cal_weak_rtd_label"] = pd.Series(
    pd.NA,
    index=cal_df.index,
    dtype="Int64",
)

cal_confident_positive_mask = (
    cal_df["cal_weak_vote_count"].gt(0)
    & cal_df["cal_lf_conflict_flag"].eq(0)
    & cal_df["cal_weak_vote_prob"].ge(RTD_LABEL_POSITIVE_THRESHOLD)
)

cal_confident_negative_mask = (
    cal_df["cal_weak_vote_count"].gt(0)
    & cal_df["cal_lf_conflict_flag"].eq(0)
    & cal_df["cal_weak_vote_prob"].le(RTD_LABEL_NEGATIVE_THRESHOLD)
)

cal_df.loc[
    cal_confident_positive_mask,
    "cal_weak_rtd_label",
] = 1

cal_df.loc[
    cal_confident_negative_mask,
    "cal_weak_rtd_label",
] = 0

cal_df["cal_weak_rtd_decision"] = (
    "Review - mixed or insufficient LF evidence"
)

cal_df.loc[
    cal_df["cal_weak_vote_count"].eq(0),
    "cal_weak_rtd_decision",
] = "Review - no active LF votes"

cal_df.loc[
    cal_df["cal_lf_conflict_flag"].eq(1),
    "cal_weak_rtd_decision",
] = "Review - active LF conflict"

cal_df.loc[
    cal_confident_positive_mask,
    "cal_weak_rtd_decision",
] = "RTD required"

cal_df.loc[
    cal_confident_negative_mask,
    "cal_weak_rtd_decision",
] = "RTD not required"


# ------------------------------------------------------------
# Compare original weak label vs calibrated weak label.
# ------------------------------------------------------------

calibration_change_summary = pd.crosstab(
    cal_df["py_weak_rtd_decision"],
    cal_df["cal_weak_rtd_decision"],
    dropna=False,
)

print("Original weak decision vs calibrated v1.1 weak decision:")
display(calibration_change_summary)

print("\nCalibrated v1.1 weak decision distribution:")
display(
    cal_df[
        [
            "cal_weak_rtd_decision",
            "cal_weak_rtd_label",
        ]
    ]
    .value_counts(dropna=False)
    .rename("row_count")
    .reset_index()
    .sort_values(
        ["cal_weak_rtd_decision", "cal_weak_rtd_label"],
        na_position="last",
    )
)

# Copy calibrated columns back to python_df for downstream sections.
calibrated_columns_to_keep = [
    column for column in cal_df.columns
    if column.startswith("cal_")
]

python_df = python_df.merge(
    cal_df[["pair_id"] + calibrated_columns_to_keep],
    on="pair_id",
    how="left",
    validate="one_to_one",
)

print(
    f"\nAdded {len(calibrated_columns_to_keep)} calibrated v1.1 columns "
    "to python_df."
)

In [ ]:
# ============================================================
# 8.7 Calibrated v1.1 Gold-Set Diagnostic
# ============================================================

cal_audit_eval_df = python_df.merge(
    audited_gold_df[audit_columns_from_gold],
    on="pair_id",
    how="inner",
    validate="one_to_one",
)

cal_audit_eval_df["rtd_required"] = pd.to_numeric(
    cal_audit_eval_df["rtd_required"],
    errors="coerce",
).astype("Int64")

cal_audit_eval_df["cal_label_matches_audit"] = (
    cal_audit_eval_df["cal_weak_rtd_label"].eq(
        cal_audit_eval_df["rtd_required"]
    )
).astype("Int64")

cal_audit_eval_df["cal_false_positive"] = (
    cal_audit_eval_df["cal_weak_rtd_label"].eq(1)
    & cal_audit_eval_df["rtd_required"].eq(0)
).astype("Int64")

cal_audit_eval_df["cal_false_negative"] = (
    cal_audit_eval_df["cal_weak_rtd_label"].eq(0)
    & cal_audit_eval_df["rtd_required"].eq(1)
).astype("Int64")

cal_audit_eval_df["cal_review_routed"] = (
    cal_audit_eval_df["cal_weak_rtd_label"].isna()
).astype("Int64")

cal_valid_audit_eval_mask = (
    cal_audit_eval_df["rtd_required"].isin([0, 1])
    & cal_audit_eval_df["cal_weak_rtd_label"].isin([0, 1])
)

cal_classified_audit_eval_df = cal_audit_eval_df.loc[
    cal_valid_audit_eval_mask
].copy()

cal_actual = cal_classified_audit_eval_df["rtd_required"].astype(int)
cal_predicted = cal_classified_audit_eval_df[
    "cal_weak_rtd_label"
].astype(int)

cal_true_positive = int(
    ((cal_actual == 1) & (cal_predicted == 1)).sum()
)
cal_false_positive = int(
    ((cal_actual == 0) & (cal_predicted == 1)).sum()
)
cal_true_negative = int(
    ((cal_actual == 0) & (cal_predicted == 0)).sum()
)
cal_false_negative = int(
    ((cal_actual == 1) & (cal_predicted == 0)).sum()
)

cal_weak_confusion_table = pd.DataFrame(
    [
        [cal_true_negative, cal_false_positive],
        [cal_false_negative, cal_true_positive],
    ],
    index=[
        "Audited: No RTD",
        "Audited: RTD Required",
    ],
    columns=[
        "Cal weak label: No RTD",
        "Cal weak label: RTD",
    ],
)

cal_weak_metric_table = pd.DataFrame({
    "metric": [
        "Audited rows evaluated",
        "Rows classified by calibrated weak label",
        "Rows routed to Review",
        "Accuracy on classified audited rows",
        "RTD precision",
        "RTD recall",
        "No-RTD specificity",
        "False positives",
        "False negatives",
    ],
    "value": [
        len(cal_audit_eval_df),
        len(cal_classified_audit_eval_df),
        int(cal_audit_eval_df["cal_review_routed"].sum()),
        safe_divide(
            cal_true_positive + cal_true_negative,
            len(cal_classified_audit_eval_df),
        ),
        safe_divide(
            cal_true_positive,
            cal_true_positive + cal_false_positive,
        ),
        safe_divide(
            cal_true_positive,
            cal_true_positive + cal_false_negative,
        ),
        safe_divide(
            cal_true_negative,
            cal_true_negative + cal_false_positive,
        ),
        cal_false_positive,
        cal_false_negative,
    ],
})

print(
    "Calibrated v1.1 weak-label audit diagnostic "
    "(development/calibration set, not independent holdout)"
)

print("\nCalibrated v1.1 confusion table:")
display(cal_weak_confusion_table)

cal_weak_metric_rate_view = cal_weak_metric_table.copy()
cal_weak_metric_rate_view["display_value"] = cal_weak_metric_rate_view.apply(
    lambda row: (
        f"{row['value']:.1%}"
        if row["metric"] in rate_metric_names and pd.notna(row["value"])
        else row["value"]
    ),
    axis=1,
)

print("\nCalibrated v1.1 weak-label metrics:")
display(cal_weak_metric_rate_view[["metric", "display_value"]])


# ------------------------------------------------------------
# Row-level comparison: original vs calibrated weak label.
# ------------------------------------------------------------

CAL_AUDIT_ROW_DISPLAY_COLUMNS = [
    "pair_id",
    CHEMICAL_A_COLUMN,
    "chemical_a_class",
    CHEMICAL_B_COLUMN,
    "chemical_b_class",
    "reaction_type",
    "reaction_speed",
    "setting",
    "final_severity_num",
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "rtd_required",
    "control_failure_mode",
    "py_weak_rtd_label",
    "py_weak_rtd_decision",
    "cal_weak_rtd_label",
    "cal_weak_rtd_decision",
    "cal_weak_vote_prob",
    "cal_weak_vote_count",
    "cal_weak_rtd_positive_votes",
    "cal_weak_rtd_negative_votes",
    "cal_lf2_bulkliquid_neutralization_rtd",
    "cal_lf4_detectabilityfailure_nortd",
    "cal_lf9_thermal_default_rtd",
    "cal_lf10_oxidizingacid_chelant_nortd",
    "cal_lf11_acidicsurfactant_glycolether_nortd",
    "cal_lf12_acidic_surfactant_volatilebase_rtd",
    "cal_label_matches_audit",
    "cal_false_positive",
    "cal_false_negative",
]

CAL_AUDIT_ROW_DISPLAY_COLUMNS = [
    column for column in CAL_AUDIT_ROW_DISPLAY_COLUMNS
    if column in cal_audit_eval_df.columns
]

cal_audit_row_diagnostics_df = (
    cal_audit_eval_df[CAL_AUDIT_ROW_DISPLAY_COLUMNS]
    .sort_values(
        [
            "cal_false_negative",
            "cal_false_positive",
            "control_failure_mode",
            "pair_id",
        ],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

print("\nCalibrated v1.1 audited row-level diagnostics:")
display(cal_audit_row_diagnostics_df)

## 8.8 Interpretation of Calibrated v1.1 Results

The calibrated v1.1 weak label layer resolves the observed development set failure modes. On the 21 completed audited rows, the calibrated labels match the audited RTD-required target with no false positives and no false negatives.

This result should be interpreted as a calibration result, not as independent validation. The v1.1 adjustments were informed by the completed audit rows, so the audit set is still a development/calibration set. The appropriate next step is to freeze the v1.1 logic and evaluate it prospectively on additional audit rows selected from high-uncertainty, high-impact, or mechanism-coverage-gap cases.

The calibrated layer changes only a small number of rows relative to the original Python weak-label aggregation. This is useful because it shows that v1.1 is not a wholesale rewrite of the weak-supervision system. It is a targeted mechanism calibration addressing specific failure modes:

- over-recommendation for oxidizing acid + chelant cases,
- over-recommendation for acidic anionic surfactant + glycol ether solvent cases,
- under-recommendation for acidic anionic surfactant + volatile/ammoniacal base neutralization.

The calibrated labels are suitable for first pass weak supervised model training, but model results should be presented as learning the calibrated rule structure, not as discovering independent chemical truth.

In [ ]:
# ============================================================
# 8.8 Calibrated v1.1 Evidence Tiers
# ============================================================

# The calibrated v1.1 layer produces a binary label for every public row.
# Before modeling, add evidence tiers so downstream analysis can distinguish
# strong rule support from single rule or recently calibrated support.

CALIBRATION_SPECIFIC_LF_COLUMNS = [
    "cal_lf10_oxidizingacid_chelant_nortd",
    "cal_lf11_acidicsurfactant_glycolether_nortd",
    "cal_lf12_acidic_surfactant_volatilebase_rtd",
]

calibration_specific_vote_df = python_df[
    CALIBRATION_SPECIFIC_LF_COLUMNS
].apply(
    pd.to_numeric,
    errors="coerce",
)

python_df["cal_has_v11_specific_vote"] = (
    calibration_specific_vote_df.notna().any(axis=1)
).astype("Int64")

python_df["cal_evidence_tier"] = "Review / no calibrated label"

python_df.loc[
    python_df["cal_weak_rtd_label"].isin([0, 1])
    & python_df["cal_weak_vote_count"].ge(2)
    & python_df["cal_lf_conflict_flag"].eq(0),
    "cal_evidence_tier",
] = "High confidence - multiple aligned LF votes"

python_df.loc[
    python_df["cal_weak_rtd_label"].isin([0, 1])
    & python_df["cal_weak_vote_count"].eq(1)
    & python_df["cal_lf_conflict_flag"].eq(0),
    "cal_evidence_tier",
] = "Moderate confidence - single LF vote"

python_df.loc[
    python_df["cal_weak_rtd_label"].isin([0, 1])
    & python_df["cal_has_v11_specific_vote"].eq(1),
    "cal_evidence_tier",
] = "Development-calibrated mechanism rule"

python_df.loc[
    python_df["cal_lf_conflict_flag"].eq(1),
    "cal_evidence_tier",
] = "Review - active LF conflict"

cal_evidence_summary = (
    python_df[
        [
            "cal_weak_rtd_decision",
            "cal_weak_rtd_label",
            "cal_evidence_tier",
        ]
    ]
    .value_counts(dropna=False)
    .rename("row_count")
    .reset_index()
    .sort_values(
        [
            "cal_evidence_tier",
            "cal_weak_rtd_decision",
            "cal_weak_rtd_label",
        ],
        na_position="last",
    )
)

print("Calibrated v1.1 evidence-tier summary:")
display(cal_evidence_summary)

print("\nEvidence tier by calibrated RTD decision:")
display(
    pd.crosstab(
        python_df["cal_evidence_tier"],
        python_df["cal_weak_rtd_decision"],
        margins=True,
        dropna=False,
    )
)

# 9. Model-Ready Dataset Preparation

The next phase trains a baseline tabular model using the calibrated v1.1 weak labels as the training target.

The purpose of this model is not to prove independent chemical prediction from limited data. Instead, the model tests whether tabular features can learn the calibrated mechanism-aware weak-supervision structure and generalize it across leakage-safe chemical pair groups.

Training rules for this first baseline:

- completed audited rows are excluded from model training,
- repeated chemical pair groups linked to audited rows are excluded,
- the target is the calibrated v1.1 weak RTD label,
- Review rows would be excluded from binary training,
- `canonical_pair_key` is used for grouped cross-validation,
- target-derived or audit-derived columns are excluded from features.

The audited rows are reserved for development diagnostics only. Future validation should use additional prospective audit rows selected after v1.1 is frozen.

In [ ]:
# ============================================================
# 9.1 Calibrated Model-Development Pools
# ============================================================

cal_weak_label_columns_for_model = [
    "pair_id",
    "cal_weak_vote_prob",
    "cal_weak_vote_count",
    "cal_weak_rtd_positive_votes",
    "cal_weak_rtd_negative_votes",
    "cal_lf_conflict_flag",
    "cal_weak_rtd_label",
    "cal_weak_rtd_decision",
    "cal_evidence_tier",
    "cal_has_v11_specific_vote",
]

cal_model_pool_df = group_safe_training_pool_df.merge(
    python_df[cal_weak_label_columns_for_model],
    on="pair_id",
    how="left",
    validate="one_to_one",
)

cal_weak_labeled_model_pool_df = cal_model_pool_df.loc[
    cal_model_pool_df["cal_weak_rtd_label"].isin([0, 1])
].copy()

cal_review_model_pool_df = cal_model_pool_df.loc[
    cal_model_pool_df["cal_weak_rtd_label"].isna()
].copy()

print("Calibrated model-development pools prepared:")
print(f"  Group-safe training pool:                    {len(cal_model_pool_df)} rows")
print(f"  Calibrated weak-labeled binary training rows: {len(cal_weak_labeled_model_pool_df)} rows")
print(f"  Review / excluded rows:                       {len(cal_review_model_pool_df)} rows")

print("\nCalibrated binary weak-label distribution in model pool:")
display(
    cal_weak_labeled_model_pool_df["cal_weak_rtd_label"]
    .value_counts(dropna=False)
    .rename_axis("cal_weak_rtd_label")
    .reset_index(name="row_count")
)

print("\nCalibrated evidence-tier distribution in model pool:")
display(
    cal_weak_labeled_model_pool_df["cal_evidence_tier"]
    .value_counts(dropna=False)
    .rename_axis("cal_evidence_tier")
    .reset_index(name="row_count")
)

assert not cal_weak_labeled_model_pool_df["pair_id"].isin(audited_pair_ids).any()

assert not cal_weak_labeled_model_pool_df["canonical_pair_key"].isin(
    audited_canonical_pair_keys
).any()

print(
    "\nLeakage check passed: no audited pair IDs or audited pair groups "
    "remain in the calibrated weak-labeled model pool."
)

## 9.2 Class Balance and Training Weights

The calibrated model-development pool is imbalanced: most rows are labeled `RTD not required`, while a smaller subset is labeled `RTD required`.

This imbalance is expected because RTDs are only appropriate for a narrower set of heat-generating, detectable, bulk-liquid scenarios. Plain accuracy would therefore be misleading. The baseline models will use class balancing and will be evaluated with metrics that are more informative for imbalanced binary classification, including balanced accuracy, precision, recall, F1 score, and average precision.

The evidence tier is not used as a model feature because it is derived from the weak labeling process. However, it can be used as a training sample weight so that rows supported by multiple aligned LFs receive slightly more influence than rows supported by only a single LF.

## 9.2 Class Balance, Evidence Weighting, and Leakage-Safe Baseline Modeling

The calibrated model development pool is imbalanced: most rows are labeled `RTD not required`, while a smaller subset is labeled `RTD required`.

This imbalance is expected because RTDs are only appropriate for a narrower set of heat-generating, detectable, bulk-liquid scenarios. Plain accuracy would therefore be misleading. The baseline models will use class balancing and will be evaluated with metrics that are more informative for imbalanced binary classification, including balanced accuracy, precision, recall, F1 score, average precision, and ROC-AUC where defined.

The evidence tier is not used as a model feature because it is derived from the weak-labeling process. However, it can be used as a training sample weight so that rows supported by multiple aligned LFs receive slightly more influence than rows supported by only a single LF.

This section trains first-pass baseline models using grouped cross-validation by `canonical_pair_key` to avoid repeated-pair leakage.

In [ ]:
# ============================================================
# 9.2 Feature Set, Target, Groups, and Evidence Weights
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except ImportError:
    HAS_STRATIFIED_GROUP_KFOLD = False


RANDOM_STATE = 42

# Do not use LF columns, weak-label columns, audit columns, or evidence tiers
# as model input features. The model should learn from source screening fields,
# not from the rules that generated the target.

CATEGORICAL_FEATURE_CANDIDATES = [
    "chemical_a_class",
    "chemical_b_class",
    "reaction_type",
    "reaction_speed",
    "setting",
    "setting_exposure",
    "cameo_descriptor",
]

NUMERIC_FEATURE_CANDIDATES = [
    "final_severity_num",
    "base_score",
    "multiplier",
    "total_score",
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "fire_flammable",
    "corrosive",
    "runaway_potential",
    "self_limiting",
    "needs_tier2_site_check",
]

categorical_features = [
    column for column in CATEGORICAL_FEATURE_CANDIDATES
    if column in cal_weak_labeled_model_pool_df.columns
]

numeric_features = [
    column for column in NUMERIC_FEATURE_CANDIDATES
    if column in cal_weak_labeled_model_pool_df.columns
]

model_feature_columns = categorical_features + numeric_features

if not model_feature_columns:
    raise ValueError("No model feature columns were found.")

X = cal_weak_labeled_model_pool_df[model_feature_columns].copy()

y = (
    cal_weak_labeled_model_pool_df["cal_weak_rtd_label"]
    .astype(int)
    .copy()
)

groups = cal_weak_labeled_model_pool_df["canonical_pair_key"].copy()

# Evidence tier is not a feature. It is only used to optionally weight training rows.
evidence_weight_map = {
    "High confidence - multiple aligned LF votes": 1.00,
    "Moderate confidence - single LF vote": 0.75,
    "Development-calibrated mechanism rule": 0.50,
    "Review - active LF conflict": 0.25,
    "Review / no calibrated label": 0.25,
}

evidence_sample_weight = (
    cal_weak_labeled_model_pool_df["cal_evidence_tier"]
    .map(evidence_weight_map)
    .fillna(0.75)
    .astype(float)
)

modeling_setup_summary = pd.DataFrame({
    "item": [
        "Training rows",
        "Unique canonical pair groups",
        "Positive weak labels",
        "Negative weak labels",
        "Positive label fraction",
        "Categorical features",
        "Numeric features",
    ],
    "value": [
        len(X),
        groups.nunique(),
        int(y.sum()),
        int((y == 0).sum()),
        f"{y.mean():.1%}",
        len(categorical_features),
        len(numeric_features),
    ],
})

print("Modeling setup summary:")
display(modeling_setup_summary)

print("\nCategorical features:")
display(pd.DataFrame({"categorical_feature": categorical_features}))

print("\nNumeric features:")
display(pd.DataFrame({"numeric_feature": numeric_features}))

print("\nEvidence-weight distribution:")
display(
    pd.DataFrame({
        "cal_evidence_tier": cal_weak_labeled_model_pool_df["cal_evidence_tier"],
        "evidence_sample_weight": evidence_sample_weight,
    })
    .value_counts(dropna=False)
    .rename("row_count")
    .reset_index()
    .sort_values(["evidence_sample_weight", "cal_evidence_tier"], ascending=[False, True])
)

In [ ]:
# ============================================================
# 9.3 Grouped Cross-Validation Baseline Models
# ============================================================

def make_one_hot_encoder():
    """Create OneHotEncoder with compatibility across sklearn versions."""
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
        )


numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

baseline_models = {
    "logreg_balanced": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=5000,
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "logreg_balanced_evidence_weighted": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=5000,
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "rf_balanced": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=400,
                    class_weight="balanced_subsample",
                    min_samples_leaf=3,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "rf_balanced_evidence_weighted": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=400,
                    class_weight="balanced_subsample",
                    min_samples_leaf=3,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}


def metric_dict_from_predictions(y_true, y_pred, y_score):
    """Return binary-classification metrics with safe handling for single-class folds."""
    metric_dict = {
        "n_rows": len(y_true),
        "positive_rows": int(np.sum(y_true == 1)),
        "negative_rows": int(np.sum(y_true == 0)),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_rtd": precision_score(y_true, y_pred, zero_division=0),
        "recall_rtd": recall_score(y_true, y_pred, zero_division=0),
        "f1_rtd": f1_score(y_true, y_pred, zero_division=0),
    }

    if len(np.unique(y_true)) == 2:
        metric_dict["average_precision"] = average_precision_score(y_true, y_score)
        metric_dict["roc_auc"] = roc_auc_score(y_true, y_score)
    else:
        metric_dict["average_precision"] = np.nan
        metric_dict["roc_auc"] = np.nan

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    metric_dict.update({
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    })

    return metric_dict


n_splits = min(5, groups.nunique(), int(y.value_counts().min()))

if n_splits < 2:
    raise ValueError("Not enough groups or class examples for grouped cross-validation.")

if HAS_STRATIFIED_GROUP_KFOLD:
    cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    cv_name = "StratifiedGroupKFold"
    split_iterator = cv.split(X, y, groups)
else:
    cv = GroupKFold(n_splits=n_splits)
    cv_name = "GroupKFold"
    split_iterator = cv.split(X, y, groups)

print(f"Using {cv_name} with {n_splits} splits.")

cv_metric_records = []
oof_prediction_frames = []

# Materialize splits once so each model sees the same folds.
cv_splits = list(split_iterator)

for model_name, model_pipeline in baseline_models.items():

    print(f"\nTraining CV model: {model_name}")

    for fold_index, (train_idx, valid_idx) in enumerate(cv_splits, start=1):

        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_valid = y.iloc[valid_idx].copy()

        valid_pair_ids = cal_weak_labeled_model_pool_df.iloc[valid_idx]["pair_id"].copy()
        valid_groups = groups.iloc[valid_idx].copy()

        fit_kwargs = {}

        if "evidence_weighted" in model_name:
            fit_kwargs["model__sample_weight"] = evidence_sample_weight.iloc[train_idx].to_numpy()

        model_pipeline.fit(
            X_train,
            y_train,
            **fit_kwargs,
        )

        y_valid_score = model_pipeline.predict_proba(X_valid)[:, 1]
        y_valid_pred = (y_valid_score >= 0.50).astype(int)

        fold_metrics = metric_dict_from_predictions(
            y_valid.to_numpy(),
            y_valid_pred,
            y_valid_score,
        )

        fold_metrics.update({
            "model_name": model_name,
            "fold": fold_index,
        })

        cv_metric_records.append(fold_metrics)

        fold_oof_df = pd.DataFrame({
            "model_name": model_name,
            "fold": fold_index,
            "pair_id": valid_pair_ids.to_numpy(),
            "canonical_pair_key": valid_groups.to_numpy(),
            "y_true": y_valid.to_numpy(),
            "y_pred": y_valid_pred,
            "y_score": y_valid_score,
            "cal_evidence_tier": cal_weak_labeled_model_pool_df.iloc[valid_idx][
                "cal_evidence_tier"
            ].to_numpy(),
            "cal_weak_vote_count": cal_weak_labeled_model_pool_df.iloc[valid_idx][
                "cal_weak_vote_count"
            ].to_numpy(),
        })

        oof_prediction_frames.append(fold_oof_df)

cv_metrics_df = pd.DataFrame(cv_metric_records)
oof_predictions_df = pd.concat(oof_prediction_frames, ignore_index=True)

metric_columns_to_summarize = [
    "accuracy",
    "balanced_accuracy",
    "precision_rtd",
    "recall_rtd",
    "f1_rtd",
    "average_precision",
    "roc_auc",
    "false_positive",
    "false_negative",
    "true_positive",
    "true_negative",
]

cv_summary_df = (
    cv_metrics_df
    .groupby("model_name")[metric_columns_to_summarize]
    .agg(["mean", "std"])
)

print("\nGrouped CV fold-level metrics:")
display(
    cv_metrics_df.sort_values(["model_name", "fold"]).style.format({
        "accuracy": "{:.1%}",
        "balanced_accuracy": "{:.1%}",
        "precision_rtd": "{:.1%}",
        "recall_rtd": "{:.1%}",
        "f1_rtd": "{:.1%}",
        "average_precision": "{:.1%}",
        "roc_auc": "{:.1%}",
    })
)

print("\nGrouped CV summary by model:")
display(
    cv_summary_df.style.format("{:.3f}")
)

In [ ]:
# ============================================================
# 9.4 Out-of-Fold Diagnostic Tables
# ============================================================

# Compare OOF predictions by model. These are predictions made on rows
# that were not used for fitting in each CV fold.

oof_model_summary_records = []

for model_name, model_oof_df in oof_predictions_df.groupby("model_name"):

    model_metrics = metric_dict_from_predictions(
        model_oof_df["y_true"].to_numpy(),
        model_oof_df["y_pred"].to_numpy(),
        model_oof_df["y_score"].to_numpy(),
    )

    model_metrics["model_name"] = model_name
    oof_model_summary_records.append(model_metrics)

oof_model_summary_df = pd.DataFrame(oof_model_summary_records)

print("Overall out-of-fold metrics by model:")
display(
    oof_model_summary_df[
        [
            "model_name",
            "n_rows",
            "positive_rows",
            "negative_rows",
            "balanced_accuracy",
            "precision_rtd",
            "recall_rtd",
            "f1_rtd",
            "average_precision",
            "roc_auc",
            "false_positive",
            "false_negative",
        ]
    ].style.format({
        "balanced_accuracy": "{:.1%}",
        "precision_rtd": "{:.1%}",
        "recall_rtd": "{:.1%}",
        "f1_rtd": "{:.1%}",
        "average_precision": "{:.1%}",
        "roc_auc": "{:.1%}",
    })
)

# Show where model predictions disagree with the calibrated weak label.
oof_error_detail_df = (
    oof_predictions_df
    .loc[oof_predictions_df["y_true"].ne(oof_predictions_df["y_pred"])]
    .merge(
        cal_weak_labeled_model_pool_df[
            [
                "pair_id",
                "chemical_a_label",
                "chemical_a_class",
                "chemical_b_label",
                "chemical_b_class",
                "reaction_type",
                "reaction_speed",
                "setting",
                "final_severity_num",
                "toxic_gas",
                "generates_gas",
                "generates_heat",
                "explosive_potential",
                "flammable",
                "corrosive",
                "cal_weak_rtd_decision",
                "cal_evidence_tier",
            ]
        ],
        on=["pair_id", "cal_evidence_tier"],
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        ["model_name", "y_true", "y_score"],
        ascending=[True, False, True],
    )
)

print("\nOut-of-fold prediction errors:")
print(f"  Error rows across all model/fold predictions: {len(oof_error_detail_df)}")
display(oof_error_detail_df.head(50))

# Rows closest to 0.50 are useful later for prospective audit selection.
oof_uncertainty_df = oof_predictions_df.copy()
oof_uncertainty_df["distance_from_0p50"] = (
    oof_uncertainty_df["y_score"] - 0.50
).abs()

oof_uncertainty_detail_df = (
    oof_uncertainty_df
    .merge(
        cal_weak_labeled_model_pool_df[
            [
                "pair_id",
                "chemical_a_label",
                "chemical_a_class",
                "chemical_b_label",
                "chemical_b_class",
                "reaction_type",
                "reaction_speed",
                "setting",
                "final_severity_num",
                "toxic_gas",
                "generates_gas",
                "generates_heat",
                "explosive_potential",
                "flammable",
                "corrosive",
                "cal_weak_rtd_decision",
                "cal_evidence_tier",
            ]
        ],
        on=["pair_id", "cal_evidence_tier"],
        how="left",
        validate="many_to_one",
    )
    .sort_values(["model_name", "distance_from_0p50", "pair_id"])
)

print("\nMost uncertain out-of-fold predictions by model:")
display(oof_uncertainty_detail_df.head(50))

## 9.5 Baseline Model Interpretation

The first modeling goal is not to claim that the model has independently discovered chemical truth. The target is still a calibrated weak label. Therefore, the baseline model should be interpreted as testing whether tabular features can learn and generalize the calibrated weak supervision structure across leakage safe chemical pair groups.

The most useful outputs from this section are:

- whether the model can recover the calibrated RTD-required class despite class imbalance,
- whether evidence weighting materially changes results,
- which rows the model finds uncertain,
- which rows the model disagrees with,
- and which rows should be considered for the next prospective audit set.

If the evidence-weighted and unweighted models perform similarly, that supports the stability of the weak-label structure. If they diverge significantly, that indicates that single LF moderate confidence rows may be introducing noise and should be prioritized for audit review.

In [ ]:
# ============================================================
# 9.5 Fit Final Baseline Logistic Model and Inspect Coefficients
# ============================================================

# Use the evidence-weighted logistic regression as the first interpretable baseline.
# This is not the final validated model; it is a diagnostic baseline for the MVP.

final_logreg_model = baseline_models["logreg_balanced_evidence_weighted"]

final_logreg_model.fit(
    X,
    y,
    model__sample_weight=evidence_sample_weight.to_numpy(),
)

final_training_scores = final_logreg_model.predict_proba(X)[:, 1]
final_training_predictions = (final_training_scores >= 0.50).astype(int)

final_training_metrics = metric_dict_from_predictions(
    y.to_numpy(),
    final_training_predictions,
    final_training_scores,
)

print("Final evidence-weighted logistic model fit on full training pool:")
display(
    pd.DataFrame([final_training_metrics]).style.format({
        "accuracy": "{:.1%}",
        "balanced_accuracy": "{:.1%}",
        "precision_rtd": "{:.1%}",
        "recall_rtd": "{:.1%}",
        "f1_rtd": "{:.1%}",
        "average_precision": "{:.1%}",
        "roc_auc": "{:.1%}",
    })
)

# Extract transformed feature names and coefficients.
preprocessor = final_logreg_model.named_steps["preprocess"]
classifier = final_logreg_model.named_steps["model"]

try:
    transformed_feature_names = preprocessor.get_feature_names_out()
except Exception:
    transformed_feature_names = [
        f"feature_{index}"
        for index in range(len(classifier.coef_[0]))
    ]

coefficient_df = pd.DataFrame({
    "transformed_feature": transformed_feature_names,
    "coefficient": classifier.coef_[0],
})

coefficient_df["abs_coefficient"] = coefficient_df["coefficient"].abs()

positive_rtd_coefficients_df = (
    coefficient_df
    .sort_values("coefficient", ascending=False)
    .head(25)
    .reset_index(drop=True)
)

negative_rtd_coefficients_df = (
    coefficient_df
    .sort_values("coefficient", ascending=True)
    .head(25)
    .reset_index(drop=True)
)

print("\nTop positive coefficients toward RTD required:")
display(positive_rtd_coefficients_df)

print("\nTop negative coefficients toward RTD not required:")
display(negative_rtd_coefficients_df)

# Store training set predictions for later audit candidate selection.
final_training_prediction_df = cal_weak_labeled_model_pool_df[
    [
        "pair_id",
        "canonical_pair_key",
        "chemical_a_label",
        "chemical_a_class",
        "chemical_b_label",
        "chemical_b_class",
        "reaction_type",
        "reaction_speed",
        "setting",
        "final_severity_num",
        "cal_weak_rtd_label",
        "cal_weak_rtd_decision",
        "cal_evidence_tier",
        "cal_weak_vote_count",
    ]
].copy()

final_training_prediction_df["final_logreg_score"] = final_training_scores
final_training_prediction_df["final_logreg_pred"] = final_training_predictions
final_training_prediction_df["final_logreg_disagrees_with_weak_label"] = (
    final_training_prediction_df["final_logreg_pred"].ne(
        final_training_prediction_df["cal_weak_rtd_label"].astype(int)
    )
).astype(int)

final_training_prediction_df["final_logreg_distance_from_0p50"] = (
    final_training_prediction_df["final_logreg_score"] - 0.50
).abs()

print("\nMost uncertain final model training pool predictions:")
display(
    final_training_prediction_df
    .sort_values(["final_logreg_distance_from_0p50", "pair_id"])
    .head(25)
)

# 10. Baseline Model Selection, Review Bands, and Audit Candidate Selection

The grouped cross-validation results show that the random forest with class balancing and evidence-tier sample weighting provides the strongest first-pass predictive baseline. The logistic regression model remains useful as an interpretable benchmark, but the random forest is used as the primary model for uncertainty analysis and prospective audit-candidate selection.

The model is still trained on calibrated weak labels, not on independent laboratory or incident outcomes. Therefore, the model should be interpreted as learning the calibrated weak-supervision structure, not as independently proving chemical truth.

This section uses out-of-fold predictions to:

- compare binary threshold behavior,
- test review-band routing around uncertain model scores,
- identify rows where the model disagrees with the calibrated weak label,
- identify rows near the decision boundary,
- and recommend 10–15 high-value candidates for the next prospective audit cycle.

In [ ]:
# ============================================================
# 10.1 Select Primary Baseline Model and Evaluate Thresholds
# ============================================================

PRIMARY_MODEL_NAME = "rf_balanced_evidence_weighted"

primary_oof_df = (
    oof_predictions_df
    .loc[oof_predictions_df["model_name"].eq(PRIMARY_MODEL_NAME)]
    .copy()
    .reset_index(drop=True)
)

if primary_oof_df.empty:
    raise ValueError(f"No OOF predictions found for {PRIMARY_MODEL_NAME}")

print(f"Primary diagnostic model selected: {PRIMARY_MODEL_NAME}")
print(f"OOF prediction rows: {len(primary_oof_df)}")

threshold_records = []

for threshold in np.arange(0.10, 0.91, 0.05):

    threshold_pred = (
        primary_oof_df["y_score"].ge(threshold)
    ).astype(int)

    threshold_metrics = metric_dict_from_predictions(
        primary_oof_df["y_true"].to_numpy(),
        threshold_pred.to_numpy(),
        primary_oof_df["y_score"].to_numpy(),
    )

    threshold_metrics["threshold"] = round(float(threshold), 2)
    threshold_records.append(threshold_metrics)

threshold_diagnostics_df = pd.DataFrame(threshold_records)

print("Primary model threshold diagnostics:")
display(
    threshold_diagnostics_df[
        [
            "threshold",
            "balanced_accuracy",
            "precision_rtd",
            "recall_rtd",
            "f1_rtd",
            "false_positive",
            "false_negative",
            "true_positive",
            "true_negative",
        ]
    ].style.format({
        "balanced_accuracy": "{:.1%}",
        "precision_rtd": "{:.1%}",
        "recall_rtd": "{:.1%}",
        "f1_rtd": "{:.1%}",
    })
)

# Identify the threshold that maximizes F1 while keeping RTD recall reasonably high.
MIN_ACCEPTABLE_RTD_RECALL = 0.85

eligible_thresholds_df = threshold_diagnostics_df.loc[
    threshold_diagnostics_df["recall_rtd"].ge(MIN_ACCEPTABLE_RTD_RECALL)
].copy()

if not eligible_thresholds_df.empty:
    selected_threshold_row = (
        eligible_thresholds_df
        .sort_values(
            ["f1_rtd", "balanced_accuracy", "precision_rtd"],
            ascending=[False, False, False],
        )
        .iloc[0]
    )
else:
    selected_threshold_row = (
        threshold_diagnostics_df
        .sort_values(
            ["f1_rtd", "balanced_accuracy", "recall_rtd"],
            ascending=[False, False, False],
        )
        .iloc[0]
    )

SELECTED_BINARY_THRESHOLD = float(selected_threshold_row["threshold"])

print(
    "\nSelected diagnostic binary threshold: "
    f"{SELECTED_BINARY_THRESHOLD:.2f}"
)

display(
    pd.DataFrame([selected_threshold_row])[
        [
            "threshold",
            "balanced_accuracy",
            "precision_rtd",
            "recall_rtd",
            "f1_rtd",
            "false_positive",
            "false_negative",
            "true_positive",
            "true_negative",
        ]
    ].style.format({
        "balanced_accuracy": "{:.1%}",
        "precision_rtd": "{:.1%}",
        "recall_rtd": "{:.1%}",
        "f1_rtd": "{:.1%}",
    })
)

In [ ]:
# ============================================================
# 10.2 Review Band Routing Diagnostics
# ============================================================

# Review band logic:
# Scores close to 0.50 are not forced into a binary recommendation.
# They are routed to Review. This is useful for process safety decision support
# because uncertain cases should be reviewed rather than over interpreted.

review_band_records = []

for half_width in [0.05, 0.10, 0.15, 0.20, 0.25]:

    lower_bound = 0.50 - half_width
    upper_bound = 0.50 + half_width

    review_mask = primary_oof_df["y_score"].between(
        lower_bound,
        upper_bound,
        inclusive="both",
    )

    classified_mask = ~review_mask

    classified_df = primary_oof_df.loc[classified_mask].copy()

    if len(classified_df) > 0:
        classified_pred = (
            classified_df["y_score"].ge(0.50)
        ).astype(int)

        classified_metrics = metric_dict_from_predictions(
            classified_df["y_true"].to_numpy(),
            classified_pred.to_numpy(),
            classified_df["y_score"].to_numpy(),
        )
    else:
        classified_metrics = {
            "balanced_accuracy": np.nan,
            "precision_rtd": np.nan,
            "recall_rtd": np.nan,
            "f1_rtd": np.nan,
            "false_positive": np.nan,
            "false_negative": np.nan,
            "true_positive": np.nan,
            "true_negative": np.nan,
        }

    review_band_records.append({
        "review_band": f"{lower_bound:.2f} to {upper_bound:.2f}",
        "half_width": half_width,
        "classified_rows": int(classified_mask.sum()),
        "review_rows": int(review_mask.sum()),
        "review_fraction": review_mask.mean(),
        "classified_positive_rows": int(
            classified_df["y_true"].eq(1).sum()
        ) if len(classified_df) > 0 else 0,
        "review_positive_rows": int(
            primary_oof_df.loc[review_mask, "y_true"].eq(1).sum()
        ),
        "balanced_accuracy_on_classified": classified_metrics["balanced_accuracy"],
        "precision_rtd_on_classified": classified_metrics["precision_rtd"],
        "recall_rtd_on_classified": classified_metrics["recall_rtd"],
        "f1_rtd_on_classified": classified_metrics["f1_rtd"],
        "false_positive_on_classified": classified_metrics["false_positive"],
        "false_negative_on_classified": classified_metrics["false_negative"],
    })

review_band_diagnostics_df = pd.DataFrame(review_band_records)

print("Review-band routing diagnostics:")
display(
    review_band_diagnostics_df.style.format({
        "review_fraction": "{:.1%}",
        "balanced_accuracy_on_classified": "{:.1%}",
        "precision_rtd_on_classified": "{:.1%}",
        "recall_rtd_on_classified": "{:.1%}",
        "f1_rtd_on_classified": "{:.1%}",
    })
)

## 10.3 Prospective Audit Candidate Selection

The next audit rows should not be chosen randomly. They should be selected to test the most informative parts of the weak supervision system.

This section ranks candidate rows using out-of-fold predictions from the primary baseline model. Higher priority candidates include:

- rows where the model disagrees with the calibrated weak label,
- rows with model scores close to 0.50,
- moderate confidence single LF rows,
- RTD-required minority class rows,
- and mechanism families that are important to validate prospectively.

The selected candidates are not used to tune v1.1 until after they are audited. They should be treated as a prospective test set for the frozen v1.1 weak-label layer and first-pass model.

In [ ]:
# ============================================================
# 10.3 Prospective Audit Candidate Ranking
# ============================================================

primary_oof_detail_columns = [
    "pair_id",
    "chemical_a_label",
    "chemical_a_class",
    "chemical_b_label",
    "chemical_b_class",
    "reaction_type",
    "reaction_speed",
    "setting",
    "final_severity_num",
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "corrosive",
    "cal_weak_rtd_label",
    "cal_weak_rtd_decision",
    "cal_weak_vote_prob",
    "cal_weak_vote_count",
    "cal_evidence_tier",
]

primary_oof_detail_columns = [
    column for column in primary_oof_detail_columns
    if column in cal_weak_labeled_model_pool_df.columns
]

# Avoid duplicate columns that already exist in primary_oof_df.
merge_detail_columns = [
    column for column in primary_oof_detail_columns
    if column not in primary_oof_df.columns
]

primary_oof_detail_df = primary_oof_df.merge(
    cal_weak_labeled_model_pool_df[
        ["pair_id"] + merge_detail_columns
    ],
    on="pair_id",
    how="left",
    validate="one_to_one",
)

primary_oof_detail_df["model_disagrees_with_weak_label"] = (
    primary_oof_detail_df["y_pred"].ne(
        primary_oof_detail_df["y_true"]
    )
).astype(int)

primary_oof_detail_df["distance_from_0p50"] = (
    primary_oof_detail_df["y_score"] - 0.50
).abs()

primary_oof_detail_df["near_decision_boundary"] = (
    primary_oof_detail_df["distance_from_0p50"].le(0.15)
).astype(int)

primary_oof_detail_df["is_moderate_confidence"] = (
    primary_oof_detail_df["cal_evidence_tier"]
    .astype(str)
    .str.contains("Moderate confidence", case=False, na=False)
).astype(int)

primary_oof_detail_df["is_rtd_required_minority_class"] = (
    primary_oof_detail_df["y_true"].eq(1)
).astype(int)


def mechanism_tags_for_audit(row):
    """Create simple mechanism-coverage tags for audit prioritization."""
    classes = {
        str(row.get("chemical_a_class", "")).lower(),
        str(row.get("chemical_b_class", "")).lower(),
    }

    reaction_type = str(row.get("reaction_type", "")).lower()

    tags = []

    if "oxidizer" in reaction_type:
        tags.append("oxidizer chemistry")

    if "oxidizing_acid" in classes and "chelating_agent" in classes:
        tags.append("oxidizing acid + chelant")

    if "acidic_anionic_surfactant" in classes and "glycol_ether_solvent" in classes:
        tags.append("acidic surfactant + glycol ether")

    if "acidic_anionic_surfactant" in classes and "ammoniacal_base" in classes:
        tags.append("acidic surfactant + volatile/ammoniacal base")

    if "chlorinated_oxidizer" in classes and (
        "alkanolamine" in classes
        or "ammoniacal_base" in classes
    ):
        tags.append("chlorinated oxidizer + amine/ammonia")

    if "alkaline_silicate" in classes or "silicate" in " ".join(classes):
        tags.append("silicate / gel potential")

    if "metal_salt" in classes or "chelating_agent" in classes:
        tags.append("metal / chelant / complexation")

    if int(row.get("toxic_gas", 0)) == 1:
        tags.append("toxic gas flag")

    if int(row.get("generates_gas", 0)) == 1:
        tags.append("gas generation flag")

    if int(row.get("explosive_potential", 0)) == 1:
        tags.append("pressure / explosive potential")

    if not tags:
        tags.append("general mechanism coverage")

    return "; ".join(tags)


primary_oof_detail_df["audit_mechanism_tags"] = primary_oof_detail_df.apply(
    mechanism_tags_for_audit,
    axis=1,
)

primary_oof_detail_df["mechanism_tag_count"] = (
    primary_oof_detail_df["audit_mechanism_tags"]
    .str.count(";")
    .fillna(0)
    .astype(int)
    + 1
)

# Priority score:
# - disagreement and uncertainty get highest weight,
# - moderate-confidence rows are useful because they rely on a single LF,
# - minority-class RTD-required rows are important due to class imbalance,
# - mechanism-tag richness helps diversify the prospective audit set.

primary_oof_detail_df["audit_priority_score"] = (
    4.0 * primary_oof_detail_df["model_disagrees_with_weak_label"]
    + 3.0 * primary_oof_detail_df["near_decision_boundary"]
    + 2.0 * primary_oof_detail_df["is_moderate_confidence"]
    + 1.5 * primary_oof_detail_df["is_rtd_required_minority_class"]
    + 0.25 * primary_oof_detail_df["mechanism_tag_count"]
    + (0.15 - primary_oof_detail_df["distance_from_0p50"]).clip(lower=0) * 5.0
)

# Confirm that the candidate pool still excludes audited rows and audited groups.
assert not primary_oof_detail_df["pair_id"].isin(audited_pair_ids).any()

assert not primary_oof_detail_df["canonical_pair_key"].isin(
    audited_canonical_pair_keys
).any()

candidate_display_columns = [
    "pair_id",
    "chemical_a_label",
    "chemical_a_class",
    "chemical_b_label",
    "chemical_b_class",
    "reaction_type",
    "setting",
    "final_severity_num",
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "cal_weak_rtd_decision",
    "cal_evidence_tier",
    "y_score",
    "y_pred",
    "y_true",
    "model_disagrees_with_weak_label",
    "near_decision_boundary",
    "distance_from_0p50",
    "audit_mechanism_tags",
    "audit_priority_score",
]

candidate_display_columns = [
    column for column in candidate_display_columns
    if column in primary_oof_detail_df.columns
]

prospective_audit_candidates_df = (
    primary_oof_detail_df
    .sort_values(
        [
            "audit_priority_score",
            "model_disagrees_with_weak_label",
            "near_decision_boundary",
            "is_rtd_required_minority_class",
            "distance_from_0p50",
        ],
        ascending=[False, False, False, False, True],
    )
    .drop_duplicates("canonical_pair_key")
    .head(30)
    .reset_index(drop=True)
)

print("Top prospective audit candidates from primary OOF model:")
display(
    prospective_audit_candidates_df[candidate_display_columns]
    .style.format({
        "y_score": "{:.3f}",
        "distance_from_0p50": "{:.3f}",
        "audit_priority_score": "{:.2f}",
    })
)

print("\nCandidate count by calibrated weak decision:")
display(
    prospective_audit_candidates_df["cal_weak_rtd_decision"]
    .value_counts(dropna=False)
    .rename_axis("cal_weak_rtd_decision")
    .reset_index(name="candidate_count")
)

print("\nCandidate count by evidence tier:")
display(
    prospective_audit_candidates_df["cal_evidence_tier"]
    .value_counts(dropna=False)
    .rename_axis("cal_evidence_tier")
    .reset_index(name="candidate_count")
)

In [ ]:
# ============================================================
# 10.4 Export Prospective Audit Candidate List
# ============================================================

audit_candidate_export_columns = [
    "pair_id",
    "chemical_a_label",
    "chemical_a_class",
    "chemical_b_label",
    "chemical_b_class",
    "reaction_type",
    "reaction_speed",
    "setting",
    "final_severity_num",
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "corrosive",
    "cal_weak_rtd_decision",
    "cal_weak_rtd_label",
    "cal_evidence_tier",
    "cal_weak_vote_prob",
    "cal_weak_vote_count",
    "y_score",
    "y_pred",
    "y_true",
    "model_disagrees_with_weak_label",
    "near_decision_boundary",
    "distance_from_0p50",
    "audit_mechanism_tags",
    "audit_priority_score",
]

audit_candidate_export_columns = [
    column for column in audit_candidate_export_columns
    if column in prospective_audit_candidates_df.columns
]

audit_candidate_export_df = prospective_audit_candidates_df[
    audit_candidate_export_columns
].copy()

audit_candidate_export_path = (
    "chemrisk_ai_prospective_audit_candidates_v1_1.csv"
)

audit_candidate_export_df.to_csv(
    audit_candidate_export_path,
    index=False,
)

print(
    "Prospective audit candidate list exported to: "
    f"{audit_candidate_export_path}"
)

display(audit_candidate_export_df.head(15))

In [ ]:
from google.colab import files

files.download('chemrisk_ai_prospective_audit_candidates_v1_1.csv')

## 10.5 Prospective Audit Cycle for Frozen v1.1

The next step is to audit a new prospective set of candidate rows selected after the calibrated v1.1 weak-label layer and first-pass baseline model were frozen.

This audit set is intentionally not a random sample of the public dataset. It is an active-learning / stress-test sample designed to challenge the weakest and most informative parts of the current system. The selected rows emphasize:

- moderate-confidence single-LF decisions,
- model disagreement with the calibrated weak label,
- model scores near the decision boundary,
- minority class RTD-required candidates,
- and mechanism families where additional validation is most useful.

Because of this selection strategy, the next audit set should not be used to estimate population-level model accuracy. Instead, it should be used to test whether frozen v1.1 generalizes to difficult cases and to identify any remaining failure modes.

The current v1.1 logic should remain frozen during this audit cycle. The audit results should be recorded first using engineering judgment, literature/SDS review, and mechanism reasoning. The model score, model prediction, and calibrated weak label should not be used as the answer. They should only be compared after the audit call is made.

The recommended prospective audit batch is:

- `PAIR_144`
- `PAIR_011`
- `PAIR_016`
- `PAIR_137`
- `PAIR_087`
- `PAIR_141`
- `PAIR_142`
- `PAIR_116`
- `PAIR_079`
- `PAIR_161`
- `PAIR_015`
- `PAIR_028`
- `PAIR_166`
- `PAIR_129`
- `PAIR_136`

For each row, the audit should focus on the same engineering target used in the first completed audit set:

- whether an exotherm is present,
- whether the exotherm is hazardous,
- whether runaway or fast pressure generation is credible,
- whether the dominant hazard is thermal or non-thermal,
- whether an RTD would detect the event in time to be creditable,
- the final `rtd_required` judgment,
- the dominant `control_failure_mode`,
- audit confidence,
- and short audit notes documenting the reasoning.

After the 15 rows are audited, the notebook should compare:

1. frozen v1.1 weak label vs. prospective audited `rtd_required`,
2. primary model prediction vs. prospective audited `rtd_required`,
3. model uncertainty vs. audit outcome,
4. failure modes by mechanism family,
5. and whether any observed failures justify a future v1.2 rule update.

No v1.2 changes should be made until the prospective audit results are reviewed.

# 11. Prospective Audit Results for Frozen v1.1

The 15-row prospective audit set was selected after the calibrated v1.1 weak-label layer and first-pass baseline model were frozen. This means the audit can be used to test whether v1.1 generalizes to difficult, model-disagreement, moderate-confidence cases.

This is not a random population sample. It is a targeted active-learning / stress-test sample. Therefore, the results should not be interpreted as population-level accuracy. Instead, they answer a more useful engineering question:

> Where does frozen v1.1 still fail when challenged with ambiguous, high-value, mechanism-sensitive cases?

The completed audit workbook is now loaded and compared against:

1. frozen v1.1 weak-label output,
2. primary model out-of-fold prediction,
3. model uncertainty,
4. audited control failure mode,
5. audited RTD-required judgment.

The audit sheet is used as the source of truth because the audit sheet contains the formula-driven fields for `mechanism_overstated`, `gold_lt_c5`, and `rtd_required`.

In [ ]:
# ============================================================
# 11.1 Load Completed Prospective Audit Workbook
# ============================================================

from pathlib import Path

NEXT15_AUDIT_WORKBOOK = (
    "gold_set_audit_template_next_15_frozen_v1dot1_"
    "AUDIT_IMPORTED_FORMULAS_RESTORED(1).xlsx"
)

if not Path(NEXT15_AUDIT_WORKBOOK).exists():
    try:
        from google.colab import files
        print(
            "Upload the completed 15-row audit workbook now. "
            "Use the final workbook you audited line by line."
        )
        uploaded = files.upload()
        if uploaded:
            NEXT15_AUDIT_WORKBOOK = list(uploaded.keys())[0]
    except Exception:
        pass

if not Path(NEXT15_AUDIT_WORKBOOK).exists():
    raise FileNotFoundError(
        f"Could not find audit workbook: {NEXT15_AUDIT_WORKBOOK}"
    )

print(f"Using completed audit workbook: {NEXT15_AUDIT_WORKBOOK}")

next15_audit_raw_df = pd.read_excel(
    NEXT15_AUDIT_WORKBOOK,
    sheet_name="Gold_Set_Audit_50",
)

next15_context_df = pd.read_excel(
    NEXT15_AUDIT_WORKBOOK,
    sheet_name="v1_1_Candidate_Context",
)

# Keep only real prospective audit rows.
next15_audit_df = next15_audit_raw_df.loc[
    next15_audit_raw_df["pair_id"]
    .astype(str)
    .str.startswith("PAIR_", na=False)
].copy()

# Recalculate the formula-driven fields explicitly in Python so the notebook
# does not depend on Excel formula cache behavior.
numeric_audit_columns = [
    "gold_severity_num",
    "final_severity_num",
    "toxic_gas_credible",
    "exotherm_present",
    "exotherm_hazardous",
    "runaway_credible",
    "pressure_hazard_credible",
    "dilute_condition",
    "flammable_credible",
    "corrosive_credible",
    "explosive_credible",
    "rtd_detectable",
]

for column in numeric_audit_columns:
    if column in next15_audit_df.columns:
        next15_audit_df[column] = pd.to_numeric(
            next15_audit_df[column],
            errors="coerce",
        )

next15_audit_df["control_failure_mode"] = (
    next15_audit_df["control_failure_mode"]
    .astype(str)
    .str.strip()
    .str.upper()
)

next15_audit_df["gold_lt_c5_recalc"] = (
    next15_audit_df["gold_severity_num"].lt(5)
).astype("Int64")

next15_audit_df["mechanism_overstated_recalc"] = (
    next15_audit_df["gold_severity_num"].lt(
        next15_audit_df["final_severity_num"]
    )
).astype("Int64")

next15_audit_df["rtd_required_recalc"] = (
    next15_audit_df["control_failure_mode"].eq("FM5")
).astype("Int64")

# Use recalculated formula-driven values as the authoritative notebook target.
next15_audit_df["rtd_required_final"] = (
    next15_audit_df["rtd_required_recalc"]
)

next15_audit_df["gold_lt_c5_final"] = (
    next15_audit_df["gold_lt_c5_recalc"]
)

next15_audit_df["mechanism_overstated_final"] = (
    next15_audit_df["mechanism_overstated_recalc"]
)

required_next15_columns = [
    "pair_id",
    "gold_severity_num",
    "control_failure_mode",
    "rtd_required_final",
    "reviewed_mechanism_summary",
    "final_auditor_notes",
]

missing_next15_columns = [
    column for column in required_next15_columns
    if column not in next15_audit_df.columns
]

if missing_next15_columns:
    raise KeyError(
        "Missing required prospective audit columns: "
        f"{missing_next15_columns}"
    )

print("Completed prospective audit rows loaded:")
display(
    next15_audit_df[
        [
            "pair_id",
            "chemical_a_class",
            "chemical_b_class",
            "reaction_type",
            "gold_severity_num",
            "gold_lt_c5_final",
            "mechanism_overstated_final",
            "rtd_detectable",
            "rtd_required_final",
            "control_failure_mode",
            "reviewed_mechanism_summary",
            "final_auditor_notes",
        ]
    ]
)

print("\nProspective audit RTD-required distribution:")
display(
    next15_audit_df["rtd_required_final"]
    .value_counts(dropna=False)
    .rename_axis("rtd_required_final")
    .reset_index(name="row_count")
)

print("\nProspective audit control-failure-mode distribution:")
display(
    next15_audit_df["control_failure_mode"]
    .value_counts(dropna=False)
    .rename_axis("control_failure_mode")
    .reset_index(name="row_count")
)

In [ ]:
# ============================================================
# 11.2 Frozen v1.1 and Model Performance on Prospective Audit
# ============================================================

# Merge completed audit calls with the saved v1.1/model context exported
# when the candidates were selected.
prospective_eval_df = next15_audit_df.merge(
    next15_context_df[
        [
            "pair_id",
            "cal_weak_rtd_decision",
            "cal_weak_rtd_label",
            "cal_evidence_tier",
            "cal_weak_vote_prob",
            "cal_weak_vote_count",
            "y_score",
            "y_pred",
            "y_true",
            "model_disagrees_with_weak_label",
            "near_decision_boundary",
            "distance_from_0p50",
            "audit_mechanism_tags",
            "audit_priority_score",
        ]
    ],
    on="pair_id",
    how="left",
    validate="one_to_one",
)

prospective_eval_df["audit_rtd_required"] = (
    prospective_eval_df["rtd_required_final"].astype(int)
)

prospective_eval_df["v11_weak_label"] = (
    prospective_eval_df["cal_weak_rtd_label"].astype(int)
)

prospective_eval_df["primary_model_pred_0p50"] = (
    prospective_eval_df["y_pred"].astype(int)
)

prospective_eval_df["v11_matches_audit"] = (
    prospective_eval_df["v11_weak_label"].eq(
        prospective_eval_df["audit_rtd_required"]
    )
).astype(int)

prospective_eval_df["model_matches_audit"] = (
    prospective_eval_df["primary_model_pred_0p50"].eq(
        prospective_eval_df["audit_rtd_required"]
    )
).astype(int)

prospective_eval_df["v11_false_positive"] = (
    prospective_eval_df["v11_weak_label"].eq(1)
    & prospective_eval_df["audit_rtd_required"].eq(0)
).astype(int)

prospective_eval_df["v11_false_negative"] = (
    prospective_eval_df["v11_weak_label"].eq(0)
    & prospective_eval_df["audit_rtd_required"].eq(1)
).astype(int)

prospective_eval_df["model_false_positive"] = (
    prospective_eval_df["primary_model_pred_0p50"].eq(1)
    & prospective_eval_df["audit_rtd_required"].eq(0)
).astype(int)

prospective_eval_df["model_false_negative"] = (
    prospective_eval_df["primary_model_pred_0p50"].eq(0)
    & prospective_eval_df["audit_rtd_required"].eq(1)
).astype(int)


def summarize_binary_against_audit(df, pred_col, target_col, label):
    y_true_local = df[target_col].astype(int).to_numpy()
    y_pred_local = df[pred_col].astype(int).to_numpy()

    tn, fp, fn, tp = confusion_matrix(
        y_true_local,
        y_pred_local,
        labels=[0, 1],
    ).ravel()

    return {
        "comparison": label,
        "rows": len(df),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
        "accuracy": accuracy_score(y_true_local, y_pred_local),
        "balanced_accuracy": balanced_accuracy_score(y_true_local, y_pred_local),
        "precision_rtd": precision_score(
            y_true_local,
            y_pred_local,
            zero_division=0,
        ),
        "recall_rtd": recall_score(
            y_true_local,
            y_pred_local,
            zero_division=0,
        ),
        "f1_rtd": f1_score(
            y_true_local,
            y_pred_local,
            zero_division=0,
        ),
    }


prospective_summary_df = pd.DataFrame(
    [
        summarize_binary_against_audit(
            prospective_eval_df,
            pred_col="v11_weak_label",
            target_col="audit_rtd_required",
            label="Frozen v1.1 weak label vs prospective audit",
        ),
        summarize_binary_against_audit(
            prospective_eval_df,
            pred_col="primary_model_pred_0p50",
            target_col="audit_rtd_required",
            label="Primary model 0.50 prediction vs prospective audit",
        ),
    ]
)

print(
    "Prospective audit performance summary "
    "(targeted stress-test set, not population accuracy):"
)

display(
    prospective_summary_df.style.format({
        "accuracy": "{:.1%}",
        "balanced_accuracy": "{:.1%}",
        "precision_rtd": "{:.1%}",
        "recall_rtd": "{:.1%}",
        "f1_rtd": "{:.1%}",
    })
)

print("\nProspective audit row-level diagnostic:")
display(
    prospective_eval_df[
        [
            "pair_id",
            "chemical_a_class",
            "chemical_b_class",
            "reaction_type",
            "gold_severity_num",
            "mechanism_overstated_final",
            "rtd_detectable",
            "audit_rtd_required",
            "control_failure_mode",
            "cal_weak_rtd_decision",
            "v11_weak_label",
            "y_score",
            "primary_model_pred_0p50",
            "v11_matches_audit",
            "model_matches_audit",
            "v11_false_positive",
            "v11_false_negative",
            "model_false_positive",
            "model_false_negative",
            "reviewed_mechanism_summary",
            "final_auditor_notes",
        ]
    ]
    .sort_values(
        [
            "v11_false_negative",
            "v11_false_positive",
            "model_false_positive",
            "pair_id",
        ],
        ascending=[False, False, False, True],
    )
    .reset_index(drop=True)
)

In [ ]:
# ============================================================
# 11.3 Review-Band Behavior on Prospective Audit
# ============================================================

prospective_review_band_records = []

for half_width in [0.05, 0.10, 0.15, 0.20, 0.25]:

    lower_bound = 0.50 - half_width
    upper_bound = 0.50 + half_width

    review_mask = prospective_eval_df["y_score"].between(
        lower_bound,
        upper_bound,
        inclusive="both",
    )

    classified_df = prospective_eval_df.loc[~review_mask].copy()

    if len(classified_df) > 0:
        pred_classified = classified_df["y_score"].ge(0.50).astype(int)
        target_classified = classified_df["audit_rtd_required"].astype(int)

        tn, fp, fn, tp = confusion_matrix(
            target_classified,
            pred_classified,
            labels=[0, 1],
        ).ravel()

        classified_bal_acc = balanced_accuracy_score(
            target_classified,
            pred_classified,
        )
        classified_precision = precision_score(
            target_classified,
            pred_classified,
            zero_division=0,
        )
        classified_recall = recall_score(
            target_classified,
            pred_classified,
            zero_division=0,
        )
    else:
        tn = fp = fn = tp = np.nan
        classified_bal_acc = np.nan
        classified_precision = np.nan
        classified_recall = np.nan

    prospective_review_band_records.append({
        "review_band": f"{lower_bound:.2f} to {upper_bound:.2f}",
        "classified_rows": int((~review_mask).sum()),
        "review_rows": int(review_mask.sum()),
        "review_fraction": review_mask.mean(),
        "review_contains_rtd_required_rows": int(
            prospective_eval_df.loc[
                review_mask,
                "audit_rtd_required",
            ].sum()
        ),
        "balanced_accuracy_on_classified": classified_bal_acc,
        "precision_rtd_on_classified": classified_precision,
        "recall_rtd_on_classified": classified_recall,
        "false_positive_on_classified": fp,
        "false_negative_on_classified": fn,
        "true_positive_on_classified": tp,
        "true_negative_on_classified": tn,
    })

prospective_review_band_df = pd.DataFrame(
    prospective_review_band_records
)

print("Prospective audit review-band diagnostics:")
display(
    prospective_review_band_df.style.format({
        "review_fraction": "{:.1%}",
        "balanced_accuracy_on_classified": "{:.1%}",
        "precision_rtd_on_classified": "{:.1%}",
        "recall_rtd_on_classified": "{:.1%}",
    })
)

## 11.4 Prospective Audit Interpretation

The 15-row prospective audit set successfully stress-tested frozen v1.1. Because the selected rows were deliberately biased toward model disagreement, moderate-confidence single-LF decisions, and decision-boundary uncertainty, lower performance on this set does not invalidate the project. Instead, it identifies where the next calibrated rule layer should improve.

The key result is that v1.1 still over-recommends RTD for several rows where the audit judged that the dominant issue was not a creditable RTD-detectable bulk temperature event. The recurring audited failure patterns include:

- gas blanketing or gas/pressure-dominant behavior,
- runaway or pressure generation that outpaces RTD detection,
- emulsion/surfactant/polymer compatibility cases with no meaningful hazardous exotherm,
- precipitation/fouling or coating mechanisms,
- and physical incompatibility rather than chemical runaway.

The v1.1 weak labels and first-pass model should therefore be treated as useful but not final. The next step is to create a documented v1.2 calibration layer using these prospective audit findings, while clearly noting that the 15-row set becomes a calibration/development set once it is used for v1.2 updates.

For the next training pass, both the original 21 audited rows and the 15 prospective audited rows should be excluded from model training to avoid leakage.

In [ ]:
# ============================================================
# 11.5 Combine Audit Sets and Prepare Updated Leakage Exclusions
# ============================================================

# Prepare the next 15 in the same spirit as the original audited gold set.
next15_audit_for_model_df = prospective_eval_df.copy()

next15_audit_for_model_df["rtd_required"] = (
    next15_audit_for_model_df["audit_rtd_required"].astype("Int64")
)

next15_audit_for_model_df["gold_lt_c5"] = (
    next15_audit_for_model_df["gold_lt_c5_final"].astype("Int64")
)

next15_audit_for_model_df["mechanism_overstated"] = (
    next15_audit_for_model_df["mechanism_overstated_final"].astype("Int64")
)

next15_audit_for_model_df["audit_set"] = (
    "prospective_v1_1_stress_test_15"
)

# The original audited_gold_df is the 21-row development/calibration set.
original_audit_for_model_df = audited_gold_df.copy()
original_audit_for_model_df["audit_set"] = "initial_development_audit_21"

combined_audit_columns = sorted(
    set(original_audit_for_model_df.columns)
    .union(set(next15_audit_for_model_df.columns))
)

combined_gold_audit_df = pd.concat(
    [
        original_audit_for_model_df.reindex(columns=combined_audit_columns),
        next15_audit_for_model_df.reindex(columns=combined_audit_columns),
    ],
    ignore_index=True,
)

# Attach canonical pair key for group-level leakage exclusion.
pair_group_lookup_df = python_df[
    [
        "pair_id",
        "canonical_pair_key",
    ]
].drop_duplicates()

combined_gold_audit_df = combined_gold_audit_df.merge(
    pair_group_lookup_df,
    on="pair_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_lookup"),
)

combined_audited_pair_ids = set(
    combined_gold_audit_df["pair_id"].dropna().astype(str)
)

combined_audited_canonical_pair_keys = set(
    combined_gold_audit_df["canonical_pair_key"]
    .dropna()
    .astype(str)
)

print("Combined audited set summary:")
display(
    combined_gold_audit_df["audit_set"]
    .value_counts(dropna=False)
    .rename_axis("audit_set")
    .reset_index(name="row_count")
)

print("\nCombined audited RTD-required distribution:")
display(
    combined_gold_audit_df["rtd_required"]
    .value_counts(dropna=False)
    .rename_axis("rtd_required")
    .reset_index(name="row_count")
)

print("\nCombined audited control-failure-mode distribution:")
display(
    combined_gold_audit_df["control_failure_mode"]
    .value_counts(dropna=False)
    .rename_axis("control_failure_mode")
    .reset_index(name="row_count")
)

# Build a post-audit leakage-safe pool for v1.2 model development.
post_audit_exclusion_mask = (
    python_df["pair_id"].astype(str).isin(combined_audited_pair_ids)
    | python_df["canonical_pair_key"].astype(str).isin(
        combined_audited_canonical_pair_keys
    )
)

post_next15_leakage_safe_pool_df = python_df.loc[
    ~post_audit_exclusion_mask
].copy()

post_next15_excluded_audited_pool_df = python_df.loc[
    post_audit_exclusion_mask
].copy()

print("\nUpdated leakage-safe pool after adding prospective audit rows:")
print(f"  Total public rows:                    {len(python_df)}")
print(f"  Combined audited/development rows:     {len(combined_gold_audit_df)}")
print(f"  Rows excluded by pair/group:           {len(post_next15_excluded_audited_pool_df)}")
print(f"  Remaining leakage-safe pool for v1.2:  {len(post_next15_leakage_safe_pool_df)}")

assert not post_next15_leakage_safe_pool_df["pair_id"].astype(str).isin(
    combined_audited_pair_ids
).any()

assert not post_next15_leakage_safe_pool_df["canonical_pair_key"].astype(str).isin(
    combined_audited_canonical_pair_keys
).any()

print(
    "\nLeakage check passed: original 21 audits, next 15 audits, "
    "and their repeated pair groups are excluded from the next training pool."
)

# 12. v1.2 Calibration Layer After Prospective Audit

The prospective 15-row audit stress tested frozen v1.1 and showed that the system still over-recommended RTD for some moderate confidence, single-LF cases. This does not invalidate the project; it is exactly the purpose of the active learning audit cycle.

Once these 15 rows are used to update the rules, they become part of the development/calibration set. Therefore, the combined 36 audited rows are excluded from the next model-training pool.

The goal of v1.2 is not to memorize individual pair IDs. Instead, v1.2 adds broader mechanism-aware corrections for recurring audited failure patterns:

- gas or pressure dominance where RTD detection is not creditable,
- reactions likely to outrun a preventive RTD response,
- physical incompatibility or phase behavior rather than hazardous bulk exotherm,
- precipitation, fouling, coating, surfactant, polymer, or emulsion effects,
- and directionality-sensitive cases that require tank/inventory context before crediting RTD.

The v1.2 layer preserves the parity-matched spreadsheet columns and v1.1 columns. New calibrated columns use the `v12_` prefix.

In [ ]:
# ============================================================
# 12.1 Identify v1.1 Failure Patterns from Prospective Audit
# ============================================================

if "prospective_eval_df" not in globals():
    raise NameError(
        "prospective_eval_df was not found. Run Sections 11.1-11.3 first."
    )

v11_failure_df = prospective_eval_df.loc[
    prospective_eval_df["v11_matches_audit"].eq(0)
].copy()

print("Frozen v1.1 prospective-audit failures:")
print(f"  Failure rows: {len(v11_failure_df)} of {len(prospective_eval_df)}")

failure_display_columns = [
    "pair_id",
    "chemical_a_label",
    "chemical_a_class",
    "chemical_b_label",
    "chemical_b_class",
    "reaction_type",
    "setting",
    "gold_severity_num",
    "rtd_detectable",
    "audit_rtd_required",
    "control_failure_mode",
    "cal_weak_rtd_decision",
    "v11_weak_label",
    "y_score",
    "primary_model_pred_0p50",
    "reviewed_mechanism_summary",
    "final_auditor_notes",
]

failure_display_columns = [
    column for column in failure_display_columns
    if column in v11_failure_df.columns
]

display(
    v11_failure_df[failure_display_columns]
    .sort_values(["control_failure_mode", "pair_id"])
    .reset_index(drop=True)
)

print("\nFailure count by audited control failure mode:")
display(
    v11_failure_df["control_failure_mode"]
    .value_counts(dropna=False)
    .rename_axis("control_failure_mode")
    .reset_index(name="failure_count")
)

print("\nFailure count by reaction type:")
display(
    v11_failure_df["reaction_type"]
    .value_counts(dropna=False)
    .rename_axis("reaction_type")
    .reset_index(name="failure_count")
)

print("\nFailure count by class pair:")
class_pair_failure_df = v11_failure_df.copy()
class_pair_failure_df["audited_class_pair"] = class_pair_failure_df.apply(
    lambda row: " + ".join(
        sorted(
            [
                str(row.get("chemical_a_class", "")),
                str(row.get("chemical_b_class", "")),
            ]
        )
    ),
    axis=1,
)

display(
    class_pair_failure_df["audited_class_pair"]
    .value_counts(dropna=False)
    .rename_axis("audited_class_pair")
    .reset_index(name="failure_count")
)

In [ ]:
# ============================================================
# 12.2 v1.2 Mechanism Helper Flags
# ============================================================

v12_df = python_df.copy()


def normalize_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower()


def v12_class_pair_is(row, class_1, class_2):
    classes = {
        normalize_text(row.get("chemical_a_class")),
        normalize_text(row.get("chemical_b_class")),
    }
    return classes == {
        normalize_text(class_1),
        normalize_text(class_2),
    }


def v12_either_class_contains(row, fragments):
    class_text = " ".join(
        [
            normalize_text(row.get("chemical_a_class")),
            normalize_text(row.get("chemical_b_class")),
        ]
    )
    return any(fragment in class_text for fragment in fragments)


def v12_reaction_type_contains(row, fragments):
    reaction_type = normalize_text(row.get("reaction_type"))
    return any(fragment in reaction_type for fragment in fragments)


def v12_int_value(row, column, default=0):
    value = row.get(column, default)
    if pd.isna(value):
        return default
    try:
        return int(value)
    except Exception:
        return default


# Directionality-sensitive mechanisms:
# These are pair-level mechanisms where order-of-addition, majority/minority
# inventory, tank level, dilution, concentration, or stored-service context could
# affect whether RTD is creditable.
v12_df["v12_directionality_sensitive_flag"] = v12_df.apply(
    lambda row: int(
        v12_reaction_type_contains(
            row,
            [
                "acid_base",
                "oxidizer",
                "hypochlorite",
                "ammonia",
                "amine",
                "other_unknown",
            ],
        )
        or v12_either_class_contains(
            row,
            [
                "surfactant",
                "polymer",
                "emulsion",
                "silicate",
                "metal",
                "chelant",
                "oxidizer",
                "oxidizing",
                "chlorinated",
            ],
        )
        or v12_int_value(row, "generates_gas") == 1
        or v12_int_value(row, "explosive_potential") == 1
        or normalize_text(row.get("reaction_speed")) in {"fast", "unknown"}
    ),
    axis=1,
).astype("Int64")


# General non-thermal dominance:
# A conservative negative RTD proxy when the hazard flags indicate that gas,
# pressure, fire, oxidizer decomposition, or other non-bulk-thermal behavior may
# dominate.
v12_df["v12_nonthermal_dominance_proxy"] = v12_df.apply(
    lambda row: int(
        v12_int_value(row, "final_severity_num") >= 5
        and v12_int_value(row, "generates_heat") == 1
        and (
            v12_int_value(row, "toxic_gas") == 1
            or v12_int_value(row, "generates_gas") == 1
            or v12_int_value(row, "explosive_potential") == 1
            or v12_int_value(row, "flammable") == 1
            or v12_int_value(row, "fire_flammable") == 1
        )
        and not (
            # Preserve clean acid/base RTD-positive pathway unless another
            # detectability/failure proxy votes against it.
            v12_reaction_type_contains(row, ["acid_base"])
            and v12_class_pair_is(row, "alkali_base", "mineral_acid")
            and v12_int_value(row, "toxic_gas") == 0
            and v12_int_value(row, "generates_gas") == 0
        )
    ),
    axis=1,
).astype("Int64")


# Phase behavior / fouling / coating / physical incompatibility:
v12_df["v12_phase_behavior_failure_proxy"] = v12_df.apply(
    lambda row: int(
        v12_int_value(row, "final_severity_num") >= 5
        and v12_int_value(row, "generates_heat") == 1
        and (
            v12_either_class_contains(
                row,
                [
                    "polymer",
                    "surfactant",
                    "emulsion",
                    "silicate",
                    "phosphate",
                    "metal_salt",
                    "chelating",
                    "chelant",
                    "glycol_ether",
                ],
            )
            or v12_reaction_type_contains(
                row,
                [
                    "precip",
                    "polymer",
                    "gel",
                    "other_unknown",
                ],
            )
        )
    ),
    axis=1,
).astype("Int64")


# Fast pressure / kinetics limitation:
v12_df["v12_fast_pressure_or_kinetics_failure_proxy"] = v12_df.apply(
    lambda row: int(
        v12_int_value(row, "final_severity_num") >= 5
        and (
            v12_int_value(row, "generates_gas") == 1
            or v12_int_value(row, "explosive_potential") == 1
            or v12_reaction_type_contains(
                row,
                [
                    "hypochlorite",
                    "oxidizer_plus_ammonia",
                    "oxidizer_plus_organic",
                ],
            )
        )
        and normalize_text(row.get("reaction_speed")) in {"fast", "unknown"}
    ),
    axis=1,
).astype("Int64")


print("v1.2 mechanism helper flag summary:")
v12_helper_columns = [
    "v12_directionality_sensitive_flag",
    "v12_nonthermal_dominance_proxy",
    "v12_phase_behavior_failure_proxy",
    "v12_fast_pressure_or_kinetics_failure_proxy",
]

display(
    v12_df[v12_helper_columns]
    .sum()
    .rename("flagged_rows")
    .reset_index()
    .rename(columns={"index": "v12_helper_flag"})
)

In [ ]:
# ============================================================
# 12.3 Build v1.2 Calibrated LF Layer
# ============================================================

# Start from v1.1 where possible.
v12_copy_map = {
    "v12_lf1_noheat_or_lowseverity_nortd": "cal_lf1_noheat_or_lowseverity_nortd",
    "v12_lf2_bulkliquid_neutralization_rtd": "cal_lf2_bulkliquid_neutralization_rtd",
    "v12_lf3_toxicgas_dominant_nortd": "cal_lf3_toxicgas_dominant_nortd",
    "v12_lf4_detectabilityfailure_nortd": "cal_lf4_detectabilityfailure_nortd",
    "v12_lf5_fastpressure_decomp_nortd": "cal_lf5_fastpressure_decomp_nortd",
    "v12_lf6_oxidizerorganic_nortd_candidate": "cal_lf6_oxidizerorganic_nortd_candidate",
    "v12_lf8_diluent_nortd_candidate": "cal_lf8_diluent_nortd_candidate",
    "v12_lf10_oxidizingacid_chelant_nortd": "cal_lf10_oxidizingacid_chelant_nortd",
    "v12_lf11_acidicsurfactant_glycolether_nortd": "cal_lf11_acidicsurfactant_glycolether_nortd",
    "v12_lf12_acidic_surfactant_volatilebase_rtd": "cal_lf12_acidic_surfactant_volatilebase_rtd",
}

for v12_column, v11_column in v12_copy_map.items():
    if v11_column in v12_df.columns:
        v12_df[v12_column] = v12_df[v11_column]
    else:
        v12_df[v12_column] = pd.Series(pd.NA, index=v12_df.index, dtype="Int64")


# New v1.2 LF13:
# General no-RTD vote for non-thermal dominance where gas/pressure/fire behavior
# is likely to dominate over bulk temperature detection.
v12_df["v12_lf13_nonthermal_dominance_nortd"] = pd.Series(
    pd.NA,
    index=v12_df.index,
    dtype="Int64",
)

v12_df.loc[
    v12_df["v12_nonthermal_dominance_proxy"].eq(1),
    "v12_lf13_nonthermal_dominance_nortd",
] = 0


# New v1.2 LF14:
# No-RTD vote for phase-behavior / fouling / physical incompatibility patterns.
v12_df["v12_lf14_phasebehavior_fouling_nortd"] = pd.Series(
    pd.NA,
    index=v12_df.index,
    dtype="Int64",
)

v12_df.loc[
    v12_df["v12_phase_behavior_failure_proxy"].eq(1),
    "v12_lf14_phasebehavior_fouling_nortd",
] = 0


# New v1.2 LF15:
# No-RTD vote for fast pressure / kinetics-limited cases.
v12_df["v12_lf15_fastpressure_kinetics_nortd"] = pd.Series(
    pd.NA,
    index=v12_df.index,
    dtype="Int64",
)

v12_df.loc[
    v12_df["v12_fast_pressure_or_kinetics_failure_proxy"].eq(1),
    "v12_lf15_fastpressure_kinetics_nortd",
] = 0


# v1.2 directionality review flag:
# This is not a positive or negative vote by itself. It is a review-routing flag
# to be used later in directionality / tank-level aggregation.
v12_df["v12_directionality_review_flag"] = (
    v12_df["v12_directionality_sensitive_flag"]
).astype("Int64")


# Recompute v1.2 LF9 thermal default after new negative LFs.
V12_NEGATIVE_LF_COLUMNS_FOR_LF9 = [
    "v12_lf3_toxicgas_dominant_nortd",
    "v12_lf4_detectabilityfailure_nortd",
    "v12_lf5_fastpressure_decomp_nortd",
    "v12_lf6_oxidizerorganic_nortd_candidate",
    "v12_lf8_diluent_nortd_candidate",
    "v12_lf10_oxidizingacid_chelant_nortd",
    "v12_lf11_acidicsurfactant_glycolether_nortd",
    "v12_lf13_nonthermal_dominance_nortd",
    "v12_lf14_phasebehavior_fouling_nortd",
    "v12_lf15_fastpressure_kinetics_nortd",
]

v12_negative_vote_df = v12_df[V12_NEGATIVE_LF_COLUMNS_FOR_LF9].apply(
    pd.to_numeric,
    errors="coerce",
)

v12_has_negative_vote = v12_negative_vote_df.eq(0).any(axis=1)

v12_lf9_positive_mask = (
    v12_df["final_severity_num"].ge(5)
    & v12_df["generates_heat"].eq(1)
    & ~v12_has_negative_vote
)

v12_df["v12_lf9_thermal_default_rtd"] = pd.Series(
    pd.NA,
    index=v12_df.index,
    dtype="Int64",
)

v12_df.loc[
    v12_lf9_positive_mask,
    "v12_lf9_thermal_default_rtd",
] = 1


V12_ACTIVE_RTD_LF_COLUMNS = [
    "v12_lf1_noheat_or_lowseverity_nortd",
    "v12_lf2_bulkliquid_neutralization_rtd",
    "v12_lf3_toxicgas_dominant_nortd",
    "v12_lf4_detectabilityfailure_nortd",
    "v12_lf5_fastpressure_decomp_nortd",
    "v12_lf6_oxidizerorganic_nortd_candidate",
    "v12_lf8_diluent_nortd_candidate",
    "v12_lf10_oxidizingacid_chelant_nortd",
    "v12_lf11_acidicsurfactant_glycolether_nortd",
    "v12_lf12_acidic_surfactant_volatilebase_rtd",
    "v12_lf13_nonthermal_dominance_nortd",
    "v12_lf14_phasebehavior_fouling_nortd",
    "v12_lf15_fastpressure_kinetics_nortd",
    "v12_lf9_thermal_default_rtd",
]

v12_vote_df = v12_df[V12_ACTIVE_RTD_LF_COLUMNS].apply(
    pd.to_numeric,
    errors="coerce",
)

v12_df["v12_weak_vote_count"] = v12_vote_df.notna().sum(axis=1).astype("Int64")
v12_df["v12_weak_rtd_positive_votes"] = v12_vote_df.eq(1).sum(axis=1).astype("Int64")
v12_df["v12_weak_rtd_negative_votes"] = v12_vote_df.eq(0).sum(axis=1).astype("Int64")

v12_df["v12_weak_vote_prob"] = pd.Series(
    np.nan,
    index=v12_df.index,
    dtype="float64",
)

v12_voted_mask = v12_df["v12_weak_vote_count"].gt(0)

v12_df.loc[v12_voted_mask, "v12_weak_vote_prob"] = (
    v12_df.loc[v12_voted_mask, "v12_weak_rtd_positive_votes"].astype(float)
    / v12_df.loc[v12_voted_mask, "v12_weak_vote_count"].astype(float)
)

v12_df["v12_lf_conflict_flag"] = (
    v12_df["v12_weak_rtd_positive_votes"].gt(0)
    & v12_df["v12_weak_rtd_negative_votes"].gt(0)
).astype("Int64")

v12_df["v12_weak_rtd_label"] = pd.Series(
    pd.NA,
    index=v12_df.index,
    dtype="Int64",
)

v12_confident_positive_mask = (
    v12_df["v12_weak_vote_count"].gt(0)
    & v12_df["v12_lf_conflict_flag"].eq(0)
    & v12_df["v12_weak_vote_prob"].ge(RTD_LABEL_POSITIVE_THRESHOLD)
)

v12_confident_negative_mask = (
    v12_df["v12_weak_vote_count"].gt(0)
    & v12_df["v12_lf_conflict_flag"].eq(0)
    & v12_df["v12_weak_vote_prob"].le(RTD_LABEL_NEGATIVE_THRESHOLD)
)

v12_df.loc[v12_confident_positive_mask, "v12_weak_rtd_label"] = 1
v12_df.loc[v12_confident_negative_mask, "v12_weak_rtd_label"] = 0

v12_df["v12_weak_rtd_decision"] = (
    "Review - mixed or insufficient LF evidence"
)

v12_df.loc[
    v12_df["v12_weak_vote_count"].eq(0),
    "v12_weak_rtd_decision",
] = "Review - no active LF votes"

v12_df.loc[
    v12_df["v12_lf_conflict_flag"].eq(1),
    "v12_weak_rtd_decision",
] = "Review - active LF conflict"

v12_df.loc[
    v12_confident_positive_mask,
    "v12_weak_rtd_decision",
] = "RTD required"

v12_df.loc[
    v12_confident_negative_mask,
    "v12_weak_rtd_decision",
] = "RTD not required"


print("v1.1 vs v1.2 weak decision cross-tab:")
display(
    pd.crosstab(
        v12_df["cal_weak_rtd_decision"],
        v12_df["v12_weak_rtd_decision"],
        margins=True,
        dropna=False,
    )
)

print("\nv1.2 weak decision distribution:")
display(
    v12_df[
        [
            "v12_weak_rtd_decision",
            "v12_weak_rtd_label",
        ]
    ]
    .value_counts(dropna=False)
    .rename("row_count")
    .reset_index()
    .sort_values(
        ["v12_weak_rtd_decision", "v12_weak_rtd_label"],
        na_position="last",
    )
)

In [ ]:
# ============================================================
# 12.4 Evaluate v1.2 on Combined Audited Development Set
# ============================================================

v12_eval_df = combined_gold_audit_df.merge(
    v12_df[
        [
            "pair_id",
            "v12_weak_rtd_label",
            "v12_weak_rtd_decision",
            "v12_weak_vote_prob",
            "v12_weak_vote_count",
            "v12_weak_rtd_positive_votes",
            "v12_weak_rtd_negative_votes",
            "v12_lf_conflict_flag",
            "v12_directionality_review_flag",
        ]
    ],
    on="pair_id",
    how="left",
    validate="many_to_one",
)

v12_eval_df["audit_rtd_required"] = pd.to_numeric(
    v12_eval_df["rtd_required"],
    errors="coerce",
).astype("Int64")

v12_eval_df = v12_eval_df.loc[
    v12_eval_df["audit_rtd_required"].isin([0, 1])
].copy()

v12_eval_df["v12_matches_audit"] = (
    v12_eval_df["v12_weak_rtd_label"].eq(
        v12_eval_df["audit_rtd_required"]
    )
).astype("Int64")

v12_eval_df["v12_false_positive"] = (
    v12_eval_df["v12_weak_rtd_label"].eq(1)
    & v12_eval_df["audit_rtd_required"].eq(0)
).astype("Int64")

v12_eval_df["v12_false_negative"] = (
    v12_eval_df["v12_weak_rtd_label"].eq(0)
    & v12_eval_df["audit_rtd_required"].eq(1)
).astype("Int64")

v12_classified_eval_df = v12_eval_df.loc[
    v12_eval_df["v12_weak_rtd_label"].isin([0, 1])
].copy()

v12_audit_summary = summarize_binary_against_audit(
    v12_classified_eval_df,
    pred_col="v12_weak_rtd_label",
    target_col="audit_rtd_required",
    label="v1.2 weak label vs combined audited development set",
)

print("v1.2 audit diagnostic on combined 36-row development/calibration set:")
display(
    pd.DataFrame([v12_audit_summary]).style.format({
        "accuracy": "{:.1%}",
        "balanced_accuracy": "{:.1%}",
        "precision_rtd": "{:.1%}",
        "recall_rtd": "{:.1%}",
        "f1_rtd": "{:.1%}",
    })
)

print("\nv1.2 row-level audit diagnostic:")
v12_eval_display_columns = [
    "audit_set",
    "pair_id",
    "chemical_a_class",
    "chemical_b_class",
    "reaction_type",
    "gold_severity_num",
    "audit_rtd_required",
    "control_failure_mode",
    "v12_weak_rtd_label",
    "v12_weak_rtd_decision",
    "v12_weak_vote_prob",
    "v12_weak_vote_count",
    "v12_directionality_review_flag",
    "v12_matches_audit",
    "v12_false_positive",
    "v12_false_negative",
]

v12_eval_display_columns = [
    column for column in v12_eval_display_columns
    if column in v12_eval_df.columns
]

display(
    v12_eval_df[v12_eval_display_columns]
    .sort_values(
        [
            "v12_false_negative",
            "v12_false_positive",
            "audit_set",
            "pair_id",
        ],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

## 12.5 v1.2 Calibration Correction: Avoid Over-Suppressing RTD-Positive Acid/Base Cases

The first v1.2 draft was intentionally conservative but proved too aggressive. It converted nearly all rows to `RTD not required` or `Review`, including known audited cases where RTD is creditable.

This is an important calibration finding. The prospective audit showed that v1.1 over-recommended RTD for several moderate-confidence cases, but the correction should not suppress legitimate bulk-liquid acid/base exotherms.

The corrected v1.2b layer narrows the new negative rules so they target specific audited failure mechanisms:

- oxidizer / organic or oxidizer / amine gas-pressure dominance,
- hypochlorite / acid toxic gas evolution,
- silicate, wax/emulsion, polymer, metal-complex, chelant, and glycol-ether phase/fouling concerns,
- fast gas/pressure mechanisms that are not ordinary bulk-liquid acid/base neutralization.

The corrected layer also protects clear liquid acid/base RTD-positive patterns from broad negative overrides. This preserves the central engineering rule:

> RTD is useful only when the event is a hazardous, detectable, bulk-temperature event — but those cases should not be removed merely because conservative screening flags gas, pressure, or high severity.

In [ ]:
# ============================================================
# 12.5 Corrected v1.2b Calibration Layer
# ============================================================

# The initial v1.2 draft over-suppressed RTD-positive cases.
# This corrected v1.2b layer keeps the same intent but narrows
# the new negative rules and protects credible bulk-liquid acid/base RTD cases.

v12b_df = python_df.copy()


def norm_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower()


def int_value(row, column, default=0):
    value = row.get(column, default)
    if pd.isna(value):
        return default
    try:
        return int(value)
    except Exception:
        return default


def class_set(row):
    return {
        norm_text(row.get("chemical_a_class")),
        norm_text(row.get("chemical_b_class")),
    }


def has_classes(row, class_a, class_b):
    return class_set(row) == {
        norm_text(class_a),
        norm_text(class_b),
    }


def either_class_contains(row, fragments):
    text = " ".join(class_set(row))
    return any(fragment in text for fragment in fragments)


def reaction_contains(row, fragments):
    reaction_type = norm_text(row.get("reaction_type"))
    return any(fragment in reaction_type for fragment in fragments)


def is_high_severity_heat(row):
    return (
        int_value(row, "final_severity_num") >= 5
        and int_value(row, "generates_heat") == 1
    )


# ------------------------------------------------------------------
# Positive protection:
# These are mechanism-level patterns supported by the audits as
# RTD-creditable bulk-liquid acid/base thermal events.
# This avoids allowing broad gas/pressure flags to erase legitimate
# RTD-positive acid/base cases.
# ------------------------------------------------------------------

def is_protected_bulk_acidbase_rtd_pattern(row):
    if not is_high_severity_heat(row):
        return False

    if norm_text(row.get("reaction_type")) != "acid_base_neutralization":
        return False

    protected_pairs = [
        ("alkali_base", "mineral_acid"),
        ("alkali_base", "oxidizing_acid"),
        ("acidic_anionic_surfactant", "ammoniacal_base"),
        ("acidic_anionic_surfactant", "alkali_base"),
    ]

    return any(
        has_classes(row, class_a, class_b)
        for class_a, class_b in protected_pairs
    )


v12b_df["v12b_protected_bulk_acidbase_rtd_pattern"] = (
    v12b_df.apply(is_protected_bulk_acidbase_rtd_pattern, axis=1)
    .astype("Int64")
)


# ------------------------------------------------------------------
# Narrowed v1.2b negative helpers.
# These are deliberately narrower than the first v1.2 draft.
# ------------------------------------------------------------------

def v12b_nonthermal_dominance(row):
    if not is_high_severity_heat(row):
        return False

    if is_protected_bulk_acidbase_rtd_pattern(row):
        return False

    # Strong wrong-hazard patterns from the audit cycle:
    # toxic gas, pressure burst, oxygen/chloramine/chlorine evolution,
    # vapor cloud, or oxidizer decomposition dominance.
    if reaction_contains(
        row,
        [
            "hypochlorite_plus_acid",
            "oxidizer_plus_ammonia_amine",
            "oxidizer_plus_organic_reducer",
        ],
    ):
        return True

    if has_classes(row, "chlorinated_oxidizer", "inorganic_oxidizer"):
        return True

    if has_classes(row, "chlorinated_oxidizer", "metal_ammonium_complex"):
        return True

    if has_classes(row, "inorganic_oxidizer", "metal_ammonium_complex"):
        return True

    if has_classes(row, "inorganic_oxidizer", "alkanolamine"):
        return True

    if has_classes(row, "inorganic_oxidizer", "quaternary_ammonium"):
        return True

    if has_classes(row, "chlorinated_oxidizer", "glycol_ether_solvent"):
        return True

    # General gas/pressure dominance only when paired with oxidizer chemistry.
    if (
        (
            int_value(row, "toxic_gas") == 1
            or int_value(row, "generates_gas") == 1
            or int_value(row, "explosive_potential") == 1
        )
        and either_class_contains(
            row,
            [
                "oxidizer",
                "oxidizing",
                "chlorinated",
            ],
        )
        and not norm_text(row.get("reaction_type")) == "acid_base_neutralization"
    ):
        return True

    return False


def v12b_phase_behavior_failure(row):
    if not is_high_severity_heat(row):
        return False

    if is_protected_bulk_acidbase_rtd_pattern(row):
        return False

    # Specific audited physical / detectability failure families.
    phase_pairs = [
        ("acidic_anionic_surfactant", "glycol_ether_solvent"),
        ("acidic_anionic_surfactant", "alkaline_silicate"),
        ("acidic_anionic_surfactant", "nonionic_wax_emulsion"),
        ("acidic_anionic_surfactant", "metal_ammonium_complex"),
        ("acrylic_polymer_dispersant", "anionic_surfactant"),
        ("anionic_surfactant", "metal_salt"),
        ("chelating_agent", "metal_salt"),
    ]

    if any(has_classes(row, a, b) for a, b in phase_pairs):
        return True

    if either_class_contains(
        row,
        [
            "wax_emulsion",
            "polymer",
            "silicate",
            "metal_ammonium_complex",
        ],
    ):
        return True

    return False


def v12b_fast_pressure_or_kinetics_failure(row):
    if not is_high_severity_heat(row):
        return False

    if is_protected_bulk_acidbase_rtd_pattern(row):
        return False

    if reaction_contains(
        row,
        [
            "hypochlorite_plus_acid",
            "oxidizer_plus_ammonia_amine",
            "oxidizer_plus_organic_reducer",
        ],
    ):
        return True

    if (
        int_value(row, "generates_gas") == 1
        and int_value(row, "explosive_potential") == 1
        and either_class_contains(
            row,
            [
                "oxidizer",
                "oxidizing",
                "chlorinated",
            ],
        )
    ):
        return True

    return False


v12b_df["v12b_nonthermal_dominance_proxy"] = (
    v12b_df.apply(v12b_nonthermal_dominance, axis=1)
    .astype("Int64")
)

v12b_df["v12b_phase_behavior_failure_proxy"] = (
    v12b_df.apply(v12b_phase_behavior_failure, axis=1)
    .astype("Int64")
)

v12b_df["v12b_fast_pressure_or_kinetics_failure_proxy"] = (
    v12b_df.apply(v12b_fast_pressure_or_kinetics_failure, axis=1)
    .astype("Int64")
)

v12b_df["v12b_directionality_sensitive_flag"] = v12b_df.apply(
    lambda row: int(
        reaction_contains(
            row,
            [
                "acid_base",
                "oxidizer",
                "hypochlorite",
                "ammonia",
                "amine",
                "other_unknown",
            ],
        )
        or either_class_contains(
            row,
            [
                "surfactant",
                "polymer",
                "emulsion",
                "silicate",
                "metal",
                "chelant",
                "oxidizer",
                "oxidizing",
                "chlorinated",
            ],
        )
        or int_value(row, "generates_gas") == 1
        or int_value(row, "explosive_potential") == 1
        or norm_text(row.get("reaction_speed")) in {"fast", "unknown"}
    ),
    axis=1,
).astype("Int64")


print("Corrected v1.2b helper flag summary:")
v12b_helper_columns = [
    "v12b_protected_bulk_acidbase_rtd_pattern",
    "v12b_nonthermal_dominance_proxy",
    "v12b_phase_behavior_failure_proxy",
    "v12b_fast_pressure_or_kinetics_failure_proxy",
    "v12b_directionality_sensitive_flag",
]

display(
    v12b_df[v12b_helper_columns]
    .sum()
    .rename("flagged_rows")
    .reset_index()
    .rename(columns={"index": "v12b_helper_flag"})
)

In [ ]:
# ============================================================
# 12.6 Build Corrected v1.2b Weak Label
# ============================================================

# Start from v1.1 calibrated LFs.
v12b_copy_map = {
    "v12b_lf1_noheat_or_lowseverity_nortd": "cal_lf1_noheat_or_lowseverity_nortd",
    "v12b_lf2_bulkliquid_neutralization_rtd": "cal_lf2_bulkliquid_neutralization_rtd",
    "v12b_lf3_toxicgas_dominant_nortd": "cal_lf3_toxicgas_dominant_nortd",
    "v12b_lf4_detectabilityfailure_nortd": "cal_lf4_detectabilityfailure_nortd",
    "v12b_lf5_fastpressure_decomp_nortd": "cal_lf5_fastpressure_decomp_nortd",
    "v12b_lf6_oxidizerorganic_nortd_candidate": "cal_lf6_oxidizerorganic_nortd_candidate",
    "v12b_lf8_diluent_nortd_candidate": "cal_lf8_diluent_nortd_candidate",
    "v12b_lf10_oxidizingacid_chelant_nortd": "cal_lf10_oxidizingacid_chelant_nortd",
    "v12b_lf11_acidicsurfactant_glycolether_nortd": "cal_lf11_acidicsurfactant_glycolether_nortd",
    "v12b_lf12_acidic_surfactant_volatilebase_rtd": "cal_lf12_acidic_surfactant_volatilebase_rtd",
}

for v12b_column, v11_column in v12b_copy_map.items():
    if v11_column in v12b_df.columns:
        v12b_df[v12b_column] = v12b_df[v11_column]
    else:
        v12b_df[v12b_column] = pd.Series(
            pd.NA,
            index=v12b_df.index,
            dtype="Int64",
        )


# Remove negative LF votes when the corrected v1.2b protected acid/base
# pattern applies. This prevents broad detectability or nonthermal rules
# from conflicting with known RTD-creditable acid/base cases.
protected_mask = (
    v12b_df["v12b_protected_bulk_acidbase_rtd_pattern"].eq(1)
)

negative_lfs_to_clear_for_protected_cases = [
    "v12b_lf3_toxicgas_dominant_nortd",
    "v12b_lf4_detectabilityfailure_nortd",
    "v12b_lf5_fastpressure_decomp_nortd",
    "v12b_lf6_oxidizerorganic_nortd_candidate",
    "v12b_lf8_diluent_nortd_candidate",
    "v12b_lf10_oxidizingacid_chelant_nortd",
    "v12b_lf11_acidicsurfactant_glycolether_nortd",
]

for column in negative_lfs_to_clear_for_protected_cases:
    v12b_df.loc[protected_mask, column] = pd.NA


# Add a positive protected acid/base LF.
v12b_df["v12b_lf16_protected_bulk_acidbase_rtd"] = pd.Series(
    pd.NA,
    index=v12b_df.index,
    dtype="Int64",
)

v12b_df.loc[
    protected_mask,
    "v12b_lf16_protected_bulk_acidbase_rtd",
] = 1


# Add narrowed v1.2b negative LFs.
v12b_df["v12b_lf13_nonthermal_dominance_nortd"] = pd.Series(
    pd.NA,
    index=v12b_df.index,
    dtype="Int64",
)

v12b_df.loc[
    v12b_df["v12b_nonthermal_dominance_proxy"].eq(1),
    "v12b_lf13_nonthermal_dominance_nortd",
] = 0

v12b_df["v12b_lf14_phasebehavior_fouling_nortd"] = pd.Series(
    pd.NA,
    index=v12b_df.index,
    dtype="Int64",
)

v12b_df.loc[
    v12b_df["v12b_phase_behavior_failure_proxy"].eq(1),
    "v12b_lf14_phasebehavior_fouling_nortd",
] = 0

v12b_df["v12b_lf15_fastpressure_kinetics_nortd"] = pd.Series(
    pd.NA,
    index=v12b_df.index,
    dtype="Int64",
)

v12b_df.loc[
    v12b_df["v12b_fast_pressure_or_kinetics_failure_proxy"].eq(1),
    "v12b_lf15_fastpressure_kinetics_nortd",
] = 0


# Recompute v1.2b thermal default.
V12B_NEGATIVE_LF_COLUMNS_FOR_LF9 = [
    "v12b_lf3_toxicgas_dominant_nortd",
    "v12b_lf4_detectabilityfailure_nortd",
    "v12b_lf5_fastpressure_decomp_nortd",
    "v12b_lf6_oxidizerorganic_nortd_candidate",
    "v12b_lf8_diluent_nortd_candidate",
    "v12b_lf10_oxidizingacid_chelant_nortd",
    "v12b_lf11_acidicsurfactant_glycolether_nortd",
    "v12b_lf13_nonthermal_dominance_nortd",
    "v12b_lf14_phasebehavior_fouling_nortd",
    "v12b_lf15_fastpressure_kinetics_nortd",
]

v12b_negative_vote_df = v12b_df[
    V12B_NEGATIVE_LF_COLUMNS_FOR_LF9
].apply(
    pd.to_numeric,
    errors="coerce",
)

v12b_has_negative_vote = v12b_negative_vote_df.eq(0).any(axis=1)

v12b_lf9_positive_mask = (
    v12b_df["final_severity_num"].ge(5)
    & v12b_df["generates_heat"].eq(1)
    & ~v12b_has_negative_vote
)

v12b_df["v12b_lf9_thermal_default_rtd"] = pd.Series(
    pd.NA,
    index=v12b_df.index,
    dtype="Int64",
)

v12b_df.loc[
    v12b_lf9_positive_mask,
    "v12b_lf9_thermal_default_rtd",
] = 1


V12B_ACTIVE_RTD_LF_COLUMNS = [
    "v12b_lf1_noheat_or_lowseverity_nortd",
    "v12b_lf2_bulkliquid_neutralization_rtd",
    "v12b_lf3_toxicgas_dominant_nortd",
    "v12b_lf4_detectabilityfailure_nortd",
    "v12b_lf5_fastpressure_decomp_nortd",
    "v12b_lf6_oxidizerorganic_nortd_candidate",
    "v12b_lf8_diluent_nortd_candidate",
    "v12b_lf10_oxidizingacid_chelant_nortd",
    "v12b_lf11_acidicsurfactant_glycolether_nortd",
    "v12b_lf12_acidic_surfactant_volatilebase_rtd",
    "v12b_lf13_nonthermal_dominance_nortd",
    "v12b_lf14_phasebehavior_fouling_nortd",
    "v12b_lf15_fastpressure_kinetics_nortd",
    "v12b_lf16_protected_bulk_acidbase_rtd",
    "v12b_lf9_thermal_default_rtd",
]

v12b_vote_df = v12b_df[V12B_ACTIVE_RTD_LF_COLUMNS].apply(
    pd.to_numeric,
    errors="coerce",
)

v12b_df["v12b_weak_vote_count"] = (
    v12b_vote_df.notna().sum(axis=1).astype("Int64")
)

v12b_df["v12b_weak_rtd_positive_votes"] = (
    v12b_vote_df.eq(1).sum(axis=1).astype("Int64")
)

v12b_df["v12b_weak_rtd_negative_votes"] = (
    v12b_vote_df.eq(0).sum(axis=1).astype("Int64")
)

v12b_df["v12b_weak_vote_prob"] = pd.Series(
    np.nan,
    index=v12b_df.index,
    dtype="float64",
)

v12b_voted_mask = v12b_df["v12b_weak_vote_count"].gt(0)

v12b_df.loc[v12b_voted_mask, "v12b_weak_vote_prob"] = (
    v12b_df.loc[
        v12b_voted_mask,
        "v12b_weak_rtd_positive_votes",
    ].astype(float)
    / v12b_df.loc[
        v12b_voted_mask,
        "v12b_weak_vote_count",
    ].astype(float)
)

v12b_df["v12b_lf_conflict_flag"] = (
    v12b_df["v12b_weak_rtd_positive_votes"].gt(0)
    & v12b_df["v12b_weak_rtd_negative_votes"].gt(0)
).astype("Int64")

v12b_df["v12b_weak_rtd_label"] = pd.Series(
    pd.NA,
    index=v12b_df.index,
    dtype="Int64",
)

v12b_confident_positive_mask = (
    v12b_df["v12b_weak_vote_count"].gt(0)
    & v12b_df["v12b_lf_conflict_flag"].eq(0)
    & v12b_df["v12b_weak_vote_prob"].ge(RTD_LABEL_POSITIVE_THRESHOLD)
)

v12b_confident_negative_mask = (
    v12b_df["v12b_weak_vote_count"].gt(0)
    & v12b_df["v12b_lf_conflict_flag"].eq(0)
    & v12b_df["v12b_weak_vote_prob"].le(RTD_LABEL_NEGATIVE_THRESHOLD)
)

v12b_df.loc[
    v12b_confident_positive_mask,
    "v12b_weak_rtd_label",
] = 1

v12b_df.loc[
    v12b_confident_negative_mask,
    "v12b_weak_rtd_label",
] = 0

v12b_df["v12b_weak_rtd_decision"] = (
    "Review - mixed or insufficient LF evidence"
)

v12b_df.loc[
    v12b_df["v12b_weak_vote_count"].eq(0),
    "v12b_weak_rtd_decision",
] = "Review - no active LF votes"

v12b_df.loc[
    v12b_df["v12b_lf_conflict_flag"].eq(1),
    "v12b_weak_rtd_decision",
] = "Review - active LF conflict"

v12b_df.loc[
    v12b_confident_positive_mask,
    "v12b_weak_rtd_decision",
] = "RTD required"

v12b_df.loc[
    v12b_confident_negative_mask,
    "v12b_weak_rtd_decision",
] = "RTD not required"


print("v1.1 vs corrected v1.2b weak decision cross-tab:")
display(
    pd.crosstab(
        v12b_df["cal_weak_rtd_decision"],
        v12b_df["v12b_weak_rtd_decision"],
        margins=True,
        dropna=False,
    )
)

print("\nCorrected v1.2b weak decision distribution:")
display(
    v12b_df[
        [
            "v12b_weak_rtd_decision",
            "v12b_weak_rtd_label",
        ]
    ]
    .value_counts(dropna=False)
    .rename("row_count")
    .reset_index()
    .sort_values(
        ["v12b_weak_rtd_decision", "v12b_weak_rtd_label"],
        na_position="last",
    )
)

In [ ]:
# ============================================================
# 12.7 Evaluate Corrected v1.2b on Combined Audited Set
# ============================================================

v12b_eval_df = combined_gold_audit_df.merge(
    v12b_df[
        [
            "pair_id",
            "v12b_weak_rtd_label",
            "v12b_weak_rtd_decision",
            "v12b_weak_vote_prob",
            "v12b_weak_vote_count",
            "v12b_weak_rtd_positive_votes",
            "v12b_weak_rtd_negative_votes",
            "v12b_lf_conflict_flag",
            "v12b_directionality_sensitive_flag",
            "v12b_protected_bulk_acidbase_rtd_pattern",
            "v12b_nonthermal_dominance_proxy",
            "v12b_phase_behavior_failure_proxy",
            "v12b_fast_pressure_or_kinetics_failure_proxy",
        ]
    ],
    on="pair_id",
    how="left",
    validate="many_to_one",
)

v12b_eval_df["audit_rtd_required"] = pd.to_numeric(
    v12b_eval_df["rtd_required"],
    errors="coerce",
).astype("Int64")

v12b_eval_df = v12b_eval_df.loc[
    v12b_eval_df["audit_rtd_required"].isin([0, 1])
].copy()

v12b_eval_df["v12b_matches_audit"] = (
    v12b_eval_df["v12b_weak_rtd_label"].eq(
        v12b_eval_df["audit_rtd_required"]
    )
).astype("Int64")

v12b_eval_df["v12b_false_positive"] = (
    v12b_eval_df["v12b_weak_rtd_label"].eq(1)
    & v12b_eval_df["audit_rtd_required"].eq(0)
).astype("Int64")

v12b_eval_df["v12b_false_negative"] = (
    v12b_eval_df["v12b_weak_rtd_label"].eq(0)
    & v12b_eval_df["audit_rtd_required"].eq(1)
).astype("Int64")

v12b_classified_eval_df = v12b_eval_df.loc[
    v12b_eval_df["v12b_weak_rtd_label"].isin([0, 1])
].copy()

v12b_audit_summary = summarize_binary_against_audit(
    v12b_classified_eval_df,
    pred_col="v12b_weak_rtd_label",
    target_col="audit_rtd_required",
    label="corrected v1.2b weak label vs combined audited development set",
)

print("Corrected v1.2b audit diagnostic on combined 36-row development/calibration set:")
display(
    pd.DataFrame([v12b_audit_summary]).style.format({
        "accuracy": "{:.1%}",
        "balanced_accuracy": "{:.1%}",
        "precision_rtd": "{:.1%}",
        "recall_rtd": "{:.1%}",
        "f1_rtd": "{:.1%}",
    })
)

print("\nCorrected v1.2b row-level audit diagnostic:")
v12b_eval_display_columns = [
    "audit_set",
    "pair_id",
    "chemical_a_class",
    "chemical_b_class",
    "reaction_type",
    "gold_severity_num",
    "audit_rtd_required",
    "control_failure_mode",
    "v12b_weak_rtd_label",
    "v12b_weak_rtd_decision",
    "v12b_weak_vote_prob",
    "v12b_weak_vote_count",
    "v12b_protected_bulk_acidbase_rtd_pattern",
    "v12b_nonthermal_dominance_proxy",
    "v12b_phase_behavior_failure_proxy",
    "v12b_fast_pressure_or_kinetics_failure_proxy",
    "v12b_directionality_sensitive_flag",
    "v12b_matches_audit",
    "v12b_false_positive",
    "v12b_false_negative",
]

v12b_eval_display_columns = [
    column for column in v12b_eval_display_columns
    if column in v12b_eval_df.columns
]

display(
    v12b_eval_df[v12b_eval_display_columns]
    .sort_values(
        [
            "v12b_false_negative",
            "v12b_false_positive",
            "audit_set",
            "pair_id",
        ],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

## 12.8 Corrected v1.2b Calibration Decision

The corrected v1.2b layer resolves the over suppression problem observed in the first v1.2 draft. Unlike the first draft, v1.2b preserves known RTD-creditable acid/base cases while reducing the over recommendation seen in v1.1.

On the combined 36-row audited development/calibration set, corrected v1.2b produced no false negatives and only one false positive. For a process safety decision support tool, this is preferable to suppressing all RTD-required cases. The remaining false positive is retained for diagnostic review rather than being used to immediately create another rule version.

At this point, v1.2b is treated as the current calibrated weak label layer for the next model training pass. The combined 36 audited rows and their repeated pair groups remain excluded from model training to prevent leakage.

In [ ]:
# ============================================================
# 12.8 Inspect Remaining v1.2b Audit Error and Freeze v1.2b
# ============================================================

v12b_remaining_errors_df = v12b_eval_df.loc[
    v12b_eval_df["v12b_matches_audit"].eq(0)
].copy()

print("Remaining corrected v1.2b audit errors:")
print(f"  Error rows: {len(v12b_remaining_errors_df)}")

remaining_error_display_columns = [
    "audit_set",
    "pair_id",
    "chemical_a_label",
    "chemical_a_class",
    "chemical_b_label",
    "chemical_b_class",
    "reaction_type",
    "gold_severity_num",
    "audit_rtd_required",
    "control_failure_mode",
    "v12b_weak_rtd_label",
    "v12b_weak_rtd_decision",
    "v12b_weak_vote_prob",
    "v12b_weak_vote_count",
    "v12b_protected_bulk_acidbase_rtd_pattern",
    "v12b_nonthermal_dominance_proxy",
    "v12b_phase_behavior_failure_proxy",
    "v12b_fast_pressure_or_kinetics_failure_proxy",
    "v12b_directionality_sensitive_flag",
    "reviewed_mechanism_summary",
    "final_auditor_notes",
]

remaining_error_display_columns = [
    column for column in remaining_error_display_columns
    if column in v12b_remaining_errors_df.columns
]

display(
    v12b_remaining_errors_df[remaining_error_display_columns]
    .reset_index(drop=True)
)

# Copy frozen v1.2b columns into python_df for downstream modeling.
# If this cell is rerun, avoid duplicating columns by dropping old v12b columns first.
existing_v12b_columns = [
    column for column in python_df.columns
    if column.startswith("v12b_")
]

if existing_v12b_columns:
    python_df = python_df.drop(columns=existing_v12b_columns)

v12b_columns_to_keep = [
    column for column in v12b_df.columns
    if column.startswith("v12b_")
]

python_df = python_df.merge(
    v12b_df[["pair_id"] + v12b_columns_to_keep],
    on="pair_id",
    how="left",
    validate="one_to_one",
)

print(
    f"\nFrozen corrected v1.2b layer added to python_df "
    f"({len(v12b_columns_to_keep)} columns)."
)

print("\nFrozen v1.2b distribution on full public dataset:")
display(
    python_df[
        [
            "v12b_weak_rtd_decision",
            "v12b_weak_rtd_label",
        ]
    ]
    .value_counts(dropna=False)
    .rename("row_count")
    .reset_index()
    .sort_values(
        ["v12b_weak_rtd_decision", "v12b_weak_rtd_label"],
        na_position="last",
    )
)

# 13. v1.2b Model Training After Prospective Audit

The next model training pass uses the corrected v1.2b weak labels as the target.

The training pool is now more conservative than the v1.1 training pool because the original 21 audited rows, the 15 prospective audit rows, and their repeated chemical pair groups are excluded. This leaves a leakage-safe v1.2b training pool.

The purpose of this training pass is to test whether tabular features can learn the corrected v1.2b weak-label structure after the prospective audit calibration. The model is still not trained on laboratory outcomes or incident data, so results should be interpreted as weak-supervised decision support modeling rather than independent chemical prediction.

Because v1.2b substantially reduces the number of RTD-required labels, class imbalance is expected to be stronger in this training pass. The model evaluation therefore emphasizes balanced accuracy, RTD precision, RTD recall, F1 score, average precision, ROC-AUC, false positives, and false negatives rather than plain accuracy.

In [ ]:
# ============================================================
# 13.1 v1.2b Model Pool and Evidence Tiers
# ============================================================

# Use the post-audit leakage-safe pool from Section 11.5, but attach
# frozen corrected v1.2b labels from python_df.

v12b_label_columns_for_model = [
    "pair_id",
    "v12b_weak_vote_prob",
    "v12b_weak_vote_count",
    "v12b_weak_rtd_positive_votes",
    "v12b_weak_rtd_negative_votes",
    "v12b_lf_conflict_flag",
    "v12b_weak_rtd_label",
    "v12b_weak_rtd_decision",
    "v12b_directionality_sensitive_flag",
    "v12b_protected_bulk_acidbase_rtd_pattern",
    "v12b_nonthermal_dominance_proxy",
    "v12b_phase_behavior_failure_proxy",
    "v12b_fast_pressure_or_kinetics_failure_proxy",
]

v12b_model_pool_df = post_next15_leakage_safe_pool_df.drop(
    columns=[
        column for column in v12b_label_columns_for_model
        if column in post_next15_leakage_safe_pool_df.columns
        and column != "pair_id"
    ],
    errors="ignore",
).merge(
    python_df[v12b_label_columns_for_model],
    on="pair_id",
    how="left",
    validate="one_to_one",
)

v12b_weak_labeled_model_pool_df = v12b_model_pool_df.loc[
    v12b_model_pool_df["v12b_weak_rtd_label"].isin([0, 1])
].copy()

v12b_review_model_pool_df = v12b_model_pool_df.loc[
    v12b_model_pool_df["v12b_weak_rtd_label"].isna()
].copy()

# Evidence tiers for v1.2b.
v12b_weak_labeled_model_pool_df["v12b_evidence_tier"] = (
    "Moderate confidence - single LF vote"
)

v12b_weak_labeled_model_pool_df.loc[
    v12b_weak_labeled_model_pool_df["v12b_weak_vote_count"].ge(2)
    & v12b_weak_labeled_model_pool_df["v12b_lf_conflict_flag"].eq(0),
    "v12b_evidence_tier",
] = "High confidence - multiple aligned LF votes"

v12b_weak_labeled_model_pool_df.loc[
    v12b_weak_labeled_model_pool_df[
        [
            "v12b_nonthermal_dominance_proxy",
            "v12b_phase_behavior_failure_proxy",
            "v12b_fast_pressure_or_kinetics_failure_proxy",
        ]
    ].eq(1).any(axis=1),
    "v12b_evidence_tier",
] = "v1.2b calibrated mechanism rule"

v12b_weak_labeled_model_pool_df.loc[
    v12b_weak_labeled_model_pool_df["v12b_lf_conflict_flag"].eq(1),
    "v12b_evidence_tier",
] = "Review - active LF conflict"

print("v1.2b model-development pools prepared:")
print(f"  Post-audit leakage-safe pool:           {len(v12b_model_pool_df)} rows")
print(f"  v1.2b weak-labeled binary training rows: {len(v12b_weak_labeled_model_pool_df)} rows")
print(f"  Review / excluded rows:                  {len(v12b_review_model_pool_df)} rows")

print("\nv1.2b binary weak-label distribution in model pool:")
display(
    v12b_weak_labeled_model_pool_df["v12b_weak_rtd_label"]
    .value_counts(dropna=False)
    .rename_axis("v12b_weak_rtd_label")
    .reset_index(name="row_count")
)

print("\nv1.2b evidence-tier distribution in model pool:")
display(
    v12b_weak_labeled_model_pool_df["v12b_evidence_tier"]
    .value_counts(dropna=False)
    .rename_axis("v12b_evidence_tier")
    .reset_index(name="row_count")
)

assert not v12b_weak_labeled_model_pool_df["pair_id"].astype(str).isin(
    combined_audited_pair_ids
).any()

assert not v12b_weak_labeled_model_pool_df["canonical_pair_key"].astype(str).isin(
    combined_audited_canonical_pair_keys
).any()

print(
    "\nLeakage check passed: combined audited rows and their repeated "
    "pair groups are excluded from the v1.2b model pool."
)

In [ ]:
# ============================================================
# 13.2 v1.2b Grouped Cross-Validation Baseline Models
# ============================================================

# Feature set remains source data only. Do not use LF outputs, weak-label
# columns, audit outputs, or evidence tier as input features.

v12b_categorical_features = [
    column for column in CATEGORICAL_FEATURE_CANDIDATES
    if column in v12b_weak_labeled_model_pool_df.columns
]

v12b_numeric_features = [
    column for column in NUMERIC_FEATURE_CANDIDATES
    if column in v12b_weak_labeled_model_pool_df.columns
]

v12b_model_feature_columns = (
    v12b_categorical_features + v12b_numeric_features
)

if not v12b_model_feature_columns:
    raise ValueError("No v1.2b model feature columns were found.")

X_v12b = v12b_weak_labeled_model_pool_df[
    v12b_model_feature_columns
].copy()

y_v12b = (
    v12b_weak_labeled_model_pool_df["v12b_weak_rtd_label"]
    .astype(int)
    .copy()
)

groups_v12b = (
    v12b_weak_labeled_model_pool_df["canonical_pair_key"]
    .copy()
)

v12b_evidence_weight_map = {
    "High confidence - multiple aligned LF votes": 1.00,
    "Moderate confidence - single LF vote": 0.75,
    "v1.2b calibrated mechanism rule": 0.75,
    "Review - active LF conflict": 0.25,
}

v12b_evidence_sample_weight = (
    v12b_weak_labeled_model_pool_df["v12b_evidence_tier"]
    .map(v12b_evidence_weight_map)
    .fillna(0.75)
    .astype(float)
)

print("v1.2b modeling setup:")
display(
    pd.DataFrame({
        "item": [
            "Training rows",
            "Unique canonical pair groups",
            "Positive v1.2b weak labels",
            "Negative v1.2b weak labels",
            "Positive label fraction",
            "Categorical features",
            "Numeric features",
        ],
        "value": [
            len(X_v12b),
            groups_v12b.nunique(),
            int(y_v12b.sum()),
            int((y_v12b == 0).sum()),
            f"{y_v12b.mean():.1%}",
            len(v12b_categorical_features),
            len(v12b_numeric_features),
        ],
    })
)

v12b_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ]
)

v12b_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", make_one_hot_encoder()),
    ]
)

v12b_preprocess = ColumnTransformer(
    transformers=[
        ("numeric", v12b_numeric_transformer, v12b_numeric_features),
        ("categorical", v12b_categorical_transformer, v12b_categorical_features),
    ],
    remainder="drop",
)

v12b_baseline_models = {
    "v12b_logreg_balanced": Pipeline(
        steps=[
            ("preprocess", v12b_preprocess),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=5000,
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "v12b_logreg_balanced_evidence_weighted": Pipeline(
        steps=[
            ("preprocess", v12b_preprocess),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=5000,
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "v12b_rf_balanced": Pipeline(
        steps=[
            ("preprocess", v12b_preprocess),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=400,
                    class_weight="balanced_subsample",
                    min_samples_leaf=3,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "v12b_rf_balanced_evidence_weighted": Pipeline(
        steps=[
            ("preprocess", v12b_preprocess),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=400,
                    class_weight="balanced_subsample",
                    min_samples_leaf=3,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

v12b_min_class_count = int(y_v12b.value_counts().min())
v12b_n_splits = min(5, groups_v12b.nunique(), v12b_min_class_count)

if v12b_n_splits < 2:
    raise ValueError(
        "Not enough v1.2b positive/negative examples for grouped CV."
    )

if HAS_STRATIFIED_GROUP_KFOLD:
    v12b_cv = StratifiedGroupKFold(
        n_splits=v12b_n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    v12b_cv_name = "StratifiedGroupKFold"
    v12b_split_iterator = v12b_cv.split(
        X_v12b,
        y_v12b,
        groups_v12b,
    )
else:
    v12b_cv = GroupKFold(n_splits=v12b_n_splits)
    v12b_cv_name = "GroupKFold"
    v12b_split_iterator = v12b_cv.split(
        X_v12b,
        y_v12b,
        groups_v12b,
    )

print(f"Using {v12b_cv_name} with {v12b_n_splits} splits for v1.2b.")

v12b_cv_splits = list(v12b_split_iterator)

v12b_cv_metric_records = []
v12b_oof_prediction_frames = []

for model_name, model_pipeline in v12b_baseline_models.items():

    print(f"\nTraining v1.2b CV model: {model_name}")

    for fold_index, (train_idx, valid_idx) in enumerate(
        v12b_cv_splits,
        start=1,
    ):

        X_train = X_v12b.iloc[train_idx].copy()
        X_valid = X_v12b.iloc[valid_idx].copy()

        y_train = y_v12b.iloc[train_idx].copy()
        y_valid = y_v12b.iloc[valid_idx].copy()

        fit_kwargs = {}

        if "evidence_weighted" in model_name:
            fit_kwargs["model__sample_weight"] = (
                v12b_evidence_sample_weight
                .iloc[train_idx]
                .to_numpy()
            )

        model_pipeline.fit(
            X_train,
            y_train,
            **fit_kwargs,
        )

        y_valid_score = model_pipeline.predict_proba(X_valid)[:, 1]
        y_valid_pred = (y_valid_score >= 0.50).astype(int)

        fold_metrics = metric_dict_from_predictions(
            y_valid.to_numpy(),
            y_valid_pred,
            y_valid_score,
        )

        fold_metrics.update({
            "model_name": model_name,
            "fold": fold_index,
        })

        v12b_cv_metric_records.append(fold_metrics)

        v12b_oof_prediction_frames.append(
            pd.DataFrame({
                "model_name": model_name,
                "fold": fold_index,
                "pair_id": v12b_weak_labeled_model_pool_df.iloc[
                    valid_idx
                ]["pair_id"].to_numpy(),
                "canonical_pair_key": groups_v12b.iloc[valid_idx].to_numpy(),
                "y_true": y_valid.to_numpy(),
                "y_pred": y_valid_pred,
                "y_score": y_valid_score,
                "v12b_evidence_tier": v12b_weak_labeled_model_pool_df.iloc[
                    valid_idx
                ]["v12b_evidence_tier"].to_numpy(),
                "v12b_weak_vote_count": v12b_weak_labeled_model_pool_df.iloc[
                    valid_idx
                ]["v12b_weak_vote_count"].to_numpy(),
            })
        )

v12b_cv_metrics_df = pd.DataFrame(v12b_cv_metric_records)
v12b_oof_predictions_df = pd.concat(
    v12b_oof_prediction_frames,
    ignore_index=True,
)

v12b_oof_model_summary_records = []

for model_name, model_oof_df in v12b_oof_predictions_df.groupby("model_name"):

    model_metrics = metric_dict_from_predictions(
        model_oof_df["y_true"].to_numpy(),
        model_oof_df["y_pred"].to_numpy(),
        model_oof_df["y_score"].to_numpy(),
    )

    model_metrics["model_name"] = model_name
    v12b_oof_model_summary_records.append(model_metrics)

v12b_oof_model_summary_df = pd.DataFrame(
    v12b_oof_model_summary_records
)

print("\nv1.2b grouped CV fold-level metrics:")
display(
    v12b_cv_metrics_df.sort_values(["model_name", "fold"]).style.format({
        "accuracy": "{:.1%}",
        "balanced_accuracy": "{:.1%}",
        "precision_rtd": "{:.1%}",
        "recall_rtd": "{:.1%}",
        "f1_rtd": "{:.1%}",
        "average_precision": "{:.1%}",
        "roc_auc": "{:.1%}",
    })
)

print("\nv1.2b overall out-of-fold metrics by model:")
display(
    v12b_oof_model_summary_df[
        [
            "model_name",
            "n_rows",
            "positive_rows",
            "negative_rows",
            "balanced_accuracy",
            "precision_rtd",
            "recall_rtd",
            "f1_rtd",
            "average_precision",
            "roc_auc",
            "false_positive",
            "false_negative",
        ]
    ].style.format({
        "balanced_accuracy": "{:.1%}",
        "precision_rtd": "{:.1%}",
        "recall_rtd": "{:.1%}",
        "f1_rtd": "{:.1%}",
        "average_precision": "{:.1%}",
        "roc_auc": "{:.1%}",
    })
)

## 13.3 v1.2b Primary Model Selection and Threshold Diagnostics

The corrected v1.2b model pool is more imbalanced than the earlier v1.1 pool. Only a small fraction of rows are labeled `RTD required`, which is expected because v1.2b intentionally narrowed RTD-positive recommendations to more clearly creditable bulk-temperature scenarios.

The evidence-weighted logistic regression model is selected as the primary v1.2b diagnostic model because it achieved the best recall on the RTD-required class with no out-of-fold false negatives. This is preferred for a process safety decision-support model, where missing a potentially creditable safeguard is more concerning than sending extra cases to review.

The next step is threshold and review band analysis. The default 0.50 threshold is only a starting point. Because the model is trained on weak labels, threshold tuning is used for diagnostic routing, not for automatic safety decisions.

In [ ]:
# ============================================================
# 13.3 v1.2b Primary Model Threshold Diagnostics
# ============================================================

V12B_PRIMARY_MODEL_NAME = "v12b_logreg_balanced_evidence_weighted"

v12b_primary_oof_df = (
    v12b_oof_predictions_df
    .loc[v12b_oof_predictions_df["model_name"].eq(V12B_PRIMARY_MODEL_NAME)]
    .copy()
    .reset_index(drop=True)
)

if v12b_primary_oof_df.empty:
    raise ValueError(f"No OOF predictions found for {V12B_PRIMARY_MODEL_NAME}")

print(f"Primary v1.2b diagnostic model selected: {V12B_PRIMARY_MODEL_NAME}")
print(f"OOF prediction rows: {len(v12b_primary_oof_df)}")

v12b_threshold_records = []

for threshold in np.arange(0.05, 0.96, 0.05):

    threshold_pred = (
        v12b_primary_oof_df["y_score"].ge(threshold)
    ).astype(int)

    threshold_metrics = metric_dict_from_predictions(
        v12b_primary_oof_df["y_true"].to_numpy(),
        threshold_pred.to_numpy(),
        v12b_primary_oof_df["y_score"].to_numpy(),
    )

    threshold_metrics["threshold"] = round(float(threshold), 2)
    v12b_threshold_records.append(threshold_metrics)

v12b_threshold_diagnostics_df = pd.DataFrame(v12b_threshold_records)

print("v1.2b primary model threshold diagnostics:")
display(
    v12b_threshold_diagnostics_df[
        [
            "threshold",
            "balanced_accuracy",
            "precision_rtd",
            "recall_rtd",
            "f1_rtd",
            "false_positive",
            "false_negative",
            "true_positive",
            "true_negative",
        ]
    ].style.format({
        "balanced_accuracy": "{:.1%}",
        "precision_rtd": "{:.1%}",
        "recall_rtd": "{:.1%}",
        "f1_rtd": "{:.1%}",
    })
)

# Select the highest-F1 threshold that preserves 100% RTD recall if possible.
v12b_zero_fn_thresholds_df = v12b_threshold_diagnostics_df.loc[
    v12b_threshold_diagnostics_df["false_negative"].eq(0)
].copy()

if not v12b_zero_fn_thresholds_df.empty:
    v12b_selected_threshold_row = (
        v12b_zero_fn_thresholds_df
        .sort_values(
            ["f1_rtd", "precision_rtd", "balanced_accuracy", "threshold"],
            ascending=[False, False, False, False],
        )
        .iloc[0]
    )
else:
    v12b_selected_threshold_row = (
        v12b_threshold_diagnostics_df
        .sort_values(
            ["f1_rtd", "recall_rtd", "balanced_accuracy", "precision_rtd"],
            ascending=[False, False, False, False],
        )
        .iloc[0]
    )

V12B_SELECTED_BINARY_THRESHOLD = float(
    v12b_selected_threshold_row["threshold"]
)

print(
    "\nSelected v1.2b diagnostic binary threshold: "
    f"{V12B_SELECTED_BINARY_THRESHOLD:.2f}"
)

display(
    pd.DataFrame([v12b_selected_threshold_row])[
        [
            "threshold",
            "balanced_accuracy",
            "precision_rtd",
            "recall_rtd",
            "f1_rtd",
            "false_positive",
            "false_negative",
            "true_positive",
            "true_negative",
        ]
    ].style.format({
        "balanced_accuracy": "{:.1%}",
        "precision_rtd": "{:.1%}",
        "recall_rtd": "{:.1%}",
        "f1_rtd": "{:.1%}",
    })
)

In [ ]:
# ============================================================
# 13.4 v1.2b Review-Band Routing Diagnostics
# ============================================================

v12b_review_band_records = []

for half_width in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:

    lower_bound = 0.50 - half_width
    upper_bound = 0.50 + half_width

    review_mask = v12b_primary_oof_df["y_score"].between(
        lower_bound,
        upper_bound,
        inclusive="both",
    )

    classified_df = v12b_primary_oof_df.loc[~review_mask].copy()

    if len(classified_df) > 0:
        classified_pred = (
            classified_df["y_score"].ge(0.50)
        ).astype(int)

        classified_metrics = metric_dict_from_predictions(
            classified_df["y_true"].to_numpy(),
            classified_pred.to_numpy(),
            classified_df["y_score"].to_numpy(),
        )
    else:
        classified_metrics = {
            "balanced_accuracy": np.nan,
            "precision_rtd": np.nan,
            "recall_rtd": np.nan,
            "f1_rtd": np.nan,
            "false_positive": np.nan,
            "false_negative": np.nan,
            "true_positive": np.nan,
            "true_negative": np.nan,
        }

    v12b_review_band_records.append({
        "review_band": f"{lower_bound:.2f} to {upper_bound:.2f}",
        "half_width": half_width,
        "classified_rows": int((~review_mask).sum()),
        "review_rows": int(review_mask.sum()),
        "review_fraction": review_mask.mean(),
        "review_positive_rows": int(
            v12b_primary_oof_df.loc[review_mask, "y_true"].eq(1).sum()
        ),
        "review_negative_rows": int(
            v12b_primary_oof_df.loc[review_mask, "y_true"].eq(0).sum()
        ),
        "balanced_accuracy_on_classified": classified_metrics["balanced_accuracy"],
        "precision_rtd_on_classified": classified_metrics["precision_rtd"],
        "recall_rtd_on_classified": classified_metrics["recall_rtd"],
        "f1_rtd_on_classified": classified_metrics["f1_rtd"],
        "false_positive_on_classified": classified_metrics["false_positive"],
        "false_negative_on_classified": classified_metrics["false_negative"],
        "true_positive_on_classified": classified_metrics["true_positive"],
        "true_negative_on_classified": classified_metrics["true_negative"],
    })

v12b_review_band_diagnostics_df = pd.DataFrame(v12b_review_band_records)

print("v1.2b review-band routing diagnostics:")
display(
    v12b_review_band_diagnostics_df.style.format({
        "review_fraction": "{:.1%}",
        "balanced_accuracy_on_classified": "{:.1%}",
        "precision_rtd_on_classified": "{:.1%}",
        "recall_rtd_on_classified": "{:.1%}",
        "f1_rtd_on_classified": "{:.1%}",
    })
)

In [ ]:
# ============================================================
# 13.5 v1.2b OOF Error and Uncertainty Diagnostics
# ============================================================

v12b_primary_oof_detail_columns = [
    "pair_id",
    "chemical_a_label",
    "chemical_a_class",
    "chemical_b_label",
    "chemical_b_class",
    "reaction_type",
    "reaction_speed",
    "setting",
    "final_severity_num",
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "corrosive",
    "v12b_weak_rtd_label",
    "v12b_weak_rtd_decision",
    "v12b_weak_vote_prob",
    "v12b_weak_vote_count",
    "v12b_evidence_tier",
    "v12b_directionality_sensitive_flag",
    "v12b_protected_bulk_acidbase_rtd_pattern",
    "v12b_nonthermal_dominance_proxy",
    "v12b_phase_behavior_failure_proxy",
    "v12b_fast_pressure_or_kinetics_failure_proxy",
]

v12b_primary_oof_detail_columns = [
    column for column in v12b_primary_oof_detail_columns
    if column in v12b_weak_labeled_model_pool_df.columns
]

v12b_primary_oof_detail_df = v12b_primary_oof_df.merge(
    v12b_weak_labeled_model_pool_df[
        ["pair_id"] + [
            column for column in v12b_primary_oof_detail_columns
            if column not in v12b_primary_oof_df.columns
        ]
    ],
    on="pair_id",
    how="left",
    validate="one_to_one",
)

v12b_primary_oof_detail_df["model_disagrees_with_v12b_label"] = (
    v12b_primary_oof_detail_df["y_pred"].ne(
        v12b_primary_oof_detail_df["y_true"]
    )
).astype(int)

v12b_primary_oof_detail_df["distance_from_0p50"] = (
    v12b_primary_oof_detail_df["y_score"] - 0.50
).abs()

v12b_primary_oof_detail_df["selected_threshold_pred"] = (
    v12b_primary_oof_detail_df["y_score"]
    .ge(V12B_SELECTED_BINARY_THRESHOLD)
).astype(int)

v12b_primary_oof_detail_df["selected_threshold_error"] = (
    v12b_primary_oof_detail_df["selected_threshold_pred"]
    .ne(v12b_primary_oof_detail_df["y_true"])
).astype(int)

print("v1.2b OOF errors at default 0.50 threshold:")
display(
    v12b_primary_oof_detail_df
    .loc[v12b_primary_oof_detail_df["model_disagrees_with_v12b_label"].eq(1)]
    .sort_values(["y_true", "y_score"], ascending=[False, False])
    .reset_index(drop=True)
    .head(50)
)

print(
    "\nv1.2b OOF errors at selected threshold "
    f"{V12B_SELECTED_BINARY_THRESHOLD:.2f}:"
)
display(
    v12b_primary_oof_detail_df
    .loc[v12b_primary_oof_detail_df["selected_threshold_error"].eq(1)]
    .sort_values(["y_true", "y_score"], ascending=[False, False])
    .reset_index(drop=True)
    .head(50)
)

print("\nMost uncertain v1.2b OOF predictions:")
display(
    v12b_primary_oof_detail_df
    .sort_values(["distance_from_0p50", "pair_id"])
    .reset_index(drop=True)
    .head(50)
)

## 13.6 v1.2b Modeling Interpretation

The v1.2b model is trained on a smaller and more conservative positive class than the v1.1 model. This makes RTD-required prediction more difficult, but it better reflects the updated engineering intent: RTD should be recommended only for scenarios where a hazardous, detectable bulk-temperature event is credible.

For the capstone, the strongest interpretation is:

- v1.2b reduced over-recommendation after the prospective audit cycle,
- the model can still recover most or all v1.2b RTD-required weak labels under grouped cross-validation,
- review-band routing remains important because the model is trained on weak labels and not on lab or incident outcomes,
- and directionality-sensitive cases should be carried forward into the tank-level scenario-expansion layer rather than hidden inside the pair-level classifier.

The next section builds the directionality-aware tank-scenario layer by expanding each chemical pair into `A_into_B` and `B_into_A` views.

## 13.7 v1.2b Model Assisted Review Overlay

The v1.2b primary model recovered all RTD-required weak-label rows in out-of-fold testing, but the remaining model errors were false positives. This means the model sometimes predicts RTD required for rows where the calibrated v1.2b weak-label layer says RTD is not required.

For the Phase 1 MVP, the model should not override the mechanism-aware weak-label layer. Instead, model disagreement is used as a review-routing signal.

Final pair-level logic for this stage:

- If v1.2b weak label and model agree, keep the v1.2b recommendation.
- If v1.2b says `RTD not required` but the model predicts `RTD required`, route to Review.
- If v1.2b says `RTD required` but the model predicts `RTD not required`, route to Review.
- If v1.2b has an LF conflict or no confident label, route to Review.

This makes the model useful without allowing it to silently overturn process safety logic. The model becomes a second-opinion layer that identifies cases where the learned feature pattern conflicts with the rule-based mechanism interpretation.

In [ ]:
# ============================================================
# 13.7 Fit Final v1.2b Diagnostic Model and Add Review Overlay
# ============================================================

# Fit the selected primary v1.2b model on the leakage-safe v1.2b training pool.
# This model is used as a review-routing overlay, not as an automatic override.

v12b_final_model = v12b_baseline_models[V12B_PRIMARY_MODEL_NAME]

v12b_final_model.fit(
    X_v12b,
    y_v12b,
    model__sample_weight=v12b_evidence_sample_weight.to_numpy(),
)

# Score all public rows for downstream diagnostics and tank-level aggregation.
# This includes audited rows for reporting, but the model itself was fit only
# on the leakage-safe training pool prepared above.

X_all_v12b = python_df[v12b_model_feature_columns].copy()

python_df["v12b_model_score"] = v12b_final_model.predict_proba(
    X_all_v12b
)[:, 1]

python_df["v12b_model_pred_0p50"] = (
    python_df["v12b_model_score"].ge(0.50)
).astype("Int64")

python_df["v12b_model_distance_from_0p50"] = (
    python_df["v12b_model_score"] - 0.50
).abs()

python_df["v12b_model_near_boundary_flag"] = (
    python_df["v12b_model_distance_from_0p50"].le(0.15)
).astype("Int64")

python_df["v12b_model_disagrees_with_weak_label"] = (
    python_df["v12b_weak_rtd_label"].isin([0, 1])
    & python_df["v12b_model_pred_0p50"].isin([0, 1])
    & python_df["v12b_model_pred_0p50"].ne(
        python_df["v12b_weak_rtd_label"]
    )
).astype("Int64")

# Final pair-level recommendation with model-assisted review overlay.
python_df["v12b_pair_recommendation"] = "Review - no confident v1.2b weak label"

python_df.loc[
    python_df["v12b_weak_rtd_label"].eq(0),
    "v12b_pair_recommendation",
] = "RTD not required"

python_df.loc[
    python_df["v12b_weak_rtd_label"].eq(1),
    "v12b_pair_recommendation",
] = "RTD required"

python_df.loc[
    python_df["v12b_lf_conflict_flag"].eq(1),
    "v12b_pair_recommendation",
] = "Review - active LF conflict"

python_df.loc[
    python_df["v12b_model_disagrees_with_weak_label"].eq(1),
    "v12b_pair_recommendation",
] = "Review - model challenges v1.2b weak label"

# Directionality-sensitive rows are not automatically changed here.
# The flag is carried forward to the directionality/tank-level layer.
python_df["v12b_pair_review_reason"] = ""

python_df.loc[
    python_df["v12b_lf_conflict_flag"].eq(1),
    "v12b_pair_review_reason",
] = "Active LF conflict"

python_df.loc[
    python_df["v12b_model_disagrees_with_weak_label"].eq(1),
    "v12b_pair_review_reason",
] = "Model prediction disagrees with calibrated v1.2b weak label"

python_df.loc[
    python_df["v12b_model_near_boundary_flag"].eq(1)
    & python_df["v12b_pair_review_reason"].eq(""),
    "v12b_pair_review_reason",
] = "Model score near decision boundary"

python_df.loc[
    python_df["v12b_directionality_sensitive_flag"].eq(1)
    & python_df["v12b_pair_review_reason"].eq(""),
    "v12b_pair_review_reason",
] = "Directionality / inventory context may affect tank-level decision"

print("v1.2b pair-level recommendation distribution with model-assisted review overlay:")
display(
    python_df["v12b_pair_recommendation"]
    .value_counts(dropna=False)
    .rename_axis("v12b_pair_recommendation")
    .reset_index(name="row_count")
)

print("\nv1.2b model-assisted review reasons:")
display(
    python_df["v12b_pair_review_reason"]
    .replace("", "No additional review reason")
    .value_counts(dropna=False)
    .rename_axis("v12b_pair_review_reason")
    .reset_index(name="row_count")
)

print("\nRows where model challenges v1.2b weak label:")
model_challenge_display_columns = [
    "pair_id",
    "chemical_a_label",
    "chemical_a_class",
    "chemical_b_label",
    "chemical_b_class",
    "reaction_type",
    "final_severity_num",
    "v12b_weak_rtd_decision",
    "v12b_weak_rtd_label",
    "v12b_model_score",
    "v12b_model_pred_0p50",
    "v12b_pair_recommendation",
    "v12b_pair_review_reason",
    "v12b_directionality_sensitive_flag",
    "v12b_nonthermal_dominance_proxy",
    "v12b_phase_behavior_failure_proxy",
    "v12b_fast_pressure_or_kinetics_failure_proxy",
]

model_challenge_display_columns = [
    column for column in model_challenge_display_columns
    if column in python_df.columns
]

display(
    python_df.loc[
        python_df["v12b_model_disagrees_with_weak_label"].eq(1),
        model_challenge_display_columns,
    ]
    .sort_values("v12b_model_score", ascending=False)
    .reset_index(drop=True)
)

# 14. Directionality Aware Scenario Expansion

The pair-level model evaluates a chemical combination, but a tank-level safeguard decision depends on direction.

For a pair `A + B`, two abnormal transfer scenarios are possible:

1. `A_into_B`: chemical A is introduced into a tank normally storing chemical B.
2. `B_into_A`: chemical B is introduced into a tank normally storing chemical A.

This matters because the stored chemical is typically the majority inventory, while the incoming chemical may be a smaller contaminant or transfer error. The majority/minority relationship can affect dilution, pH endpoint, gas evolution, precipitation, heat capacity, heat release rate, local concentration, and whether an RTD would detect a meaningful bulk temperature rise in time.

The Phase 1 public dataset does not contain tank levels, concentrations, order-of-addition, transfer rates, or measured kinetics. Therefore, directionality is handled as a scenario expansion and review-routing layer rather than hidden inside the pair level classifier.

The goal of this section is to create two directional scenario rows for each chemical pair and carry forward:

- stored chemical,
- incoming chemical,
- stored class,
- incoming class,
- pair-level v1.2b recommendation,
- model challenge flags,
- directionality-sensitive flags,
- and tank-level review reasons.

In [ ]:
# ============================================================
# 14.1 Create Directionality Aware Scenario Rows
# ============================================================

directionality_base_columns = [
    "pair_id",
    "canonical_pair_key",
    "chemical_a_label",
    "chemical_a_class",
    "chemical_b_label",
    "chemical_b_class",
    "reaction_type",
    "reaction_speed",
    "setting",
    "final_severity_num",
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "corrosive",
    "v12b_weak_rtd_label",
    "v12b_weak_rtd_decision",
    "v12b_pair_recommendation",
    "v12b_pair_review_reason",
    "v12b_model_score",
    "v12b_model_pred_0p50",
    "v12b_model_disagrees_with_weak_label",
    "v12b_model_near_boundary_flag",
    "v12b_directionality_sensitive_flag",
    "v12b_protected_bulk_acidbase_rtd_pattern",
    "v12b_nonthermal_dominance_proxy",
    "v12b_phase_behavior_failure_proxy",
    "v12b_fast_pressure_or_kinetics_failure_proxy",
]

directionality_base_columns = [
    column for column in directionality_base_columns
    if column in python_df.columns
]

base_direction_df = python_df[directionality_base_columns].copy()

a_into_b_df = base_direction_df.copy()
a_into_b_df["directional_scenario"] = "A_into_B"
a_into_b_df["incoming_chemical_label"] = a_into_b_df["chemical_a_label"]
a_into_b_df["incoming_chemical_class"] = a_into_b_df["chemical_a_class"]
a_into_b_df["stored_chemical_label"] = a_into_b_df["chemical_b_label"]
a_into_b_df["stored_chemical_class"] = a_into_b_df["chemical_b_class"]

b_into_a_df = base_direction_df.copy()
b_into_a_df["directional_scenario"] = "B_into_A"
b_into_a_df["incoming_chemical_label"] = b_into_a_df["chemical_b_label"]
b_into_a_df["incoming_chemical_class"] = b_into_a_df["chemical_b_class"]
b_into_a_df["stored_chemical_label"] = b_into_a_df["chemical_a_label"]
b_into_a_df["stored_chemical_class"] = b_into_a_df["chemical_a_class"]

directional_scenario_df = pd.concat(
    [a_into_b_df, b_into_a_df],
    ignore_index=True,
)

# Add directional inventory context status.
directional_scenario_df["inventory_context_available"] = 0

directional_scenario_df["directionality_review_reason"] = ""

directional_scenario_df.loc[
    directional_scenario_df["v12b_directionality_sensitive_flag"].eq(1),
    "directionality_review_reason",
] = (
    "Directionality-sensitive pair: order of addition, stored inventory, "
    "tank level, concentration, or transfer rate may affect RTD creditability"
)

# Stored service RTD practicality placeholder.
# This will be refined later using a site-specific tank/service lookup.
FOULING_OR_RTD_IMPRACTICAL_CLASS_FRAGMENTS = [
    "polymer",
    "emulsion",
    "wax",
    "silicate",
    "slurry",
    "surfactant",
    "metal_complex",
    "metal_ammonium_complex",
]

def stored_service_rtd_practicality_proxy(stored_class):
    stored_class_text = normalize_text(stored_class)
    if any(
        fragment in stored_class_text
        for fragment in FOULING_OR_RTD_IMPRACTICAL_CLASS_FRAGMENTS
    ):
        return 0
    return 1

directional_scenario_df["stored_service_rtd_practicality_proxy"] = (
    directional_scenario_df["stored_chemical_class"]
    .apply(stored_service_rtd_practicality_proxy)
    .astype("Int64")
)

directional_scenario_df["stored_service_practicality_reason"] = ""

directional_scenario_df.loc[
    directional_scenario_df["stored_service_rtd_practicality_proxy"].eq(0),
    "stored_service_practicality_reason",
] = (
    "Stored service may foul, coat, phase-separate, or otherwise reduce RTD reliability"
)

# Directional pair recommendation.
directional_scenario_df["directional_pair_recommendation"] = (
    directional_scenario_df["v12b_pair_recommendation"]
)

directional_scenario_df.loc[
    directional_scenario_df["stored_service_rtd_practicality_proxy"].eq(0)
    & directional_scenario_df["v12b_pair_recommendation"].eq("RTD required"),
    "directional_pair_recommendation",
] = "Review - RTD required by pair model but stored service may be impractical"

directional_scenario_df.loc[
    directional_scenario_df["v12b_directionality_sensitive_flag"].eq(1)
    & directional_scenario_df["v12b_pair_recommendation"].eq("RTD required"),
    "directional_pair_recommendation",
] = "Review - RTD candidate requires directionality / inventory validation"

print("Directional scenario rows created:")
print(f"  Pair rows:        {len(python_df)}")
print(f"  Directional rows: {len(directional_scenario_df)}")

print("\nDirectional scenario recommendation distribution:")
display(
    directional_scenario_df["directional_pair_recommendation"]
    .value_counts(dropna=False)
    .rename_axis("directional_pair_recommendation")
    .reset_index(name="row_count")
)

print("\nStored-service RTD practicality proxy distribution:")
display(
    directional_scenario_df["stored_service_rtd_practicality_proxy"]
    .value_counts(dropna=False)
    .rename_axis("stored_service_rtd_practicality_proxy")
    .reset_index(name="row_count")
)

print("\nExample directional scenarios:")
display(
    directional_scenario_df[
        [
            "pair_id",
            "directional_scenario",
            "incoming_chemical_label",
            "incoming_chemical_class",
            "stored_chemical_label",
            "stored_chemical_class",
            "v12b_pair_recommendation",
            "directional_pair_recommendation",
            "stored_service_rtd_practicality_proxy",
            "directionality_review_reason",
            "stored_service_practicality_reason",
        ]
    ].head(20)
)

## 14.2 Directionality Layer Cleanup and Interpretation

The first directionality expansion successfully created two directional scenarios for each chemical pair: `A_into_B` and `B_into_A`.

One cleanup is needed before tank-level aggregation. Directionality sensitivity is not a pair-level model error. It is a tank-context flag. Therefore, it should be carried into the directional scenario layer rather than treated as a pair-level review reason for every row.

The refined logic keeps the pair-level recommendation focused on the v1.2b weak label and model disagreement, then applies directionality and stored-service practicality at the directional scenario level.

This separation matters because the pair model answers:

> Is this chemical pair generally an RTD-creditable thermal safeguard case?

The directional layer answers:

> If chemical A enters a tank storing B, or B enters a tank storing A, does the stored-service context make RTD creditable, impractical, or review-dependent?

This keeps the model architecture aligned with the engineering reality that tank inventory, concentration, level, and order of addition affect the final safeguard decision.

In [ ]:
# ============================================================
# 14.2 Refine Pair Level vs Directional Review Logic
# ============================================================

# Pair-level recommendation should be based on:
# - v1.2b weak label
# - active LF conflicts
# - model disagreement
#
# Directionality is carried as a context flag into the directional layer,
# not as a pair-level review reason by itself.

python_df["v12b_pair_recommendation_clean"] = "Review - no confident v1.2b weak label"

python_df.loc[
    python_df["v12b_weak_rtd_label"].eq(0),
    "v12b_pair_recommendation_clean",
] = "RTD not required"

python_df.loc[
    python_df["v12b_weak_rtd_label"].eq(1),
    "v12b_pair_recommendation_clean",
] = "RTD required"

python_df.loc[
    python_df["v12b_lf_conflict_flag"].eq(1),
    "v12b_pair_recommendation_clean",
] = "Review - active LF conflict"

python_df.loc[
    python_df["v12b_model_disagrees_with_weak_label"].eq(1),
    "v12b_pair_recommendation_clean",
] = "Review - model challenges v1.2b weak label"


python_df["v12b_pair_review_reason_clean"] = ""

python_df.loc[
    python_df["v12b_pair_recommendation_clean"].eq("Review - no confident v1.2b weak label"),
    "v12b_pair_review_reason_clean",
] = "No confident v1.2b weak label"

python_df.loc[
    python_df["v12b_lf_conflict_flag"].eq(1),
    "v12b_pair_review_reason_clean",
] = "Active LF conflict"

python_df.loc[
    python_df["v12b_model_disagrees_with_weak_label"].eq(1),
    "v12b_pair_review_reason_clean",
] = "Model prediction disagrees with calibrated v1.2b weak label"


def build_pair_context_flags(row):
    flags = []

    if int(row.get("v12b_directionality_sensitive_flag", 0)) == 1:
        flags.append("directionality/inventory-sensitive")

    if int(row.get("v12b_model_near_boundary_flag", 0)) == 1:
        flags.append("model score near boundary")

    if int(row.get("v12b_nonthermal_dominance_proxy", 0)) == 1:
        flags.append("non-thermal dominance proxy")

    if int(row.get("v12b_phase_behavior_failure_proxy", 0)) == 1:
        flags.append("phase/fouling proxy")

    if int(row.get("v12b_fast_pressure_or_kinetics_failure_proxy", 0)) == 1:
        flags.append("fast pressure/kinetics proxy")

    if int(row.get("v12b_protected_bulk_acidbase_rtd_pattern", 0)) == 1:
        flags.append("protected bulk acid/base RTD pattern")

    if not flags:
        return "no additional context flags"

    return "; ".join(flags)


python_df["v12b_pair_context_flags"] = python_df.apply(
    build_pair_context_flags,
    axis=1,
)

print("Cleaned v1.2b pair-level recommendation distribution:")
display(
    python_df["v12b_pair_recommendation_clean"]
    .value_counts(dropna=False)
    .rename_axis("v12b_pair_recommendation_clean")
    .reset_index(name="row_count")
)

print("\nCleaned pair-level review reasons:")
display(
    python_df["v12b_pair_review_reason_clean"]
    .replace("", "No pair-level review reason")
    .value_counts(dropna=False)
    .rename_axis("v12b_pair_review_reason_clean")
    .reset_index(name="row_count")
)

print("\nPair-level context flags:")
display(
    python_df["v12b_pair_context_flags"]
    .value_counts(dropna=False)
    .rename_axis("v12b_pair_context_flags")
    .reset_index(name="row_count")
    .head(25)
)

In [ ]:
# ============================================================
# 14.3 Rebuild Directional Scenario Table with Combined Review Reasons
# ============================================================

directionality_base_columns_clean = [
    "pair_id",
    "canonical_pair_key",
    "chemical_a_label",
    "chemical_a_class",
    "chemical_b_label",
    "chemical_b_class",
    "reaction_type",
    "reaction_speed",
    "setting",
    "final_severity_num",
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "corrosive",
    "v12b_weak_rtd_label",
    "v12b_weak_rtd_decision",
    "v12b_pair_recommendation_clean",
    "v12b_pair_review_reason_clean",
    "v12b_pair_context_flags",
    "v12b_model_score",
    "v12b_model_pred_0p50",
    "v12b_model_disagrees_with_weak_label",
    "v12b_model_near_boundary_flag",
    "v12b_directionality_sensitive_flag",
    "v12b_protected_bulk_acidbase_rtd_pattern",
    "v12b_nonthermal_dominance_proxy",
    "v12b_phase_behavior_failure_proxy",
    "v12b_fast_pressure_or_kinetics_failure_proxy",
]

directionality_base_columns_clean = [
    column for column in directionality_base_columns_clean
    if column in python_df.columns
]

base_direction_clean_df = python_df[directionality_base_columns_clean].copy()

a_into_b_clean_df = base_direction_clean_df.copy()
a_into_b_clean_df["directional_scenario"] = "A_into_B"
a_into_b_clean_df["incoming_chemical_label"] = a_into_b_clean_df["chemical_a_label"]
a_into_b_clean_df["incoming_chemical_class"] = a_into_b_clean_df["chemical_a_class"]
a_into_b_clean_df["stored_chemical_label"] = a_into_b_clean_df["chemical_b_label"]
a_into_b_clean_df["stored_chemical_class"] = a_into_b_clean_df["chemical_b_class"]

b_into_a_clean_df = base_direction_clean_df.copy()
b_into_a_clean_df["directional_scenario"] = "B_into_A"
b_into_a_clean_df["incoming_chemical_label"] = b_into_a_clean_df["chemical_b_label"]
b_into_a_clean_df["incoming_chemical_class"] = b_into_a_clean_df["chemical_b_class"]
b_into_a_clean_df["stored_chemical_label"] = b_into_a_clean_df["chemical_a_label"]
b_into_a_clean_df["stored_chemical_class"] = b_into_a_clean_df["chemical_a_class"]

directional_scenario_v12b_df = pd.concat(
    [a_into_b_clean_df, b_into_a_clean_df],
    ignore_index=True,
)

directional_scenario_v12b_df["inventory_context_available"] = 0


FOULING_OR_RTD_IMPRACTICAL_CLASS_FRAGMENTS = [
    "polymer",
    "emulsion",
    "wax",
    "silicate",
    "slurry",
    "surfactant",
    "metal_complex",
    "metal_ammonium_complex",
]


def stored_service_rtd_practicality_proxy(stored_class):
    stored_class_text = normalize_text(stored_class)
    if any(
        fragment in stored_class_text
        for fragment in FOULING_OR_RTD_IMPRACTICAL_CLASS_FRAGMENTS
    ):
        return 0
    return 1


directional_scenario_v12b_df["stored_service_rtd_practicality_proxy"] = (
    directional_scenario_v12b_df["stored_chemical_class"]
    .apply(stored_service_rtd_practicality_proxy)
    .astype("Int64")
)


def build_directional_review_reasons(row):
    reasons = []

    pair_reason = str(row.get("v12b_pair_review_reason_clean", "")).strip()
    if pair_reason:
        reasons.append(pair_reason)

    if int(row.get("v12b_directionality_sensitive_flag", 0)) == 1:
        reasons.append(
            "Directionality-sensitive: order of addition, stored inventory, tank level, "
            "concentration, or transfer rate may affect RTD creditability"
        )

    if int(row.get("stored_service_rtd_practicality_proxy", 1)) == 0:
        reasons.append(
            "Stored service may foul, coat, phase-separate, or reduce RTD reliability"
        )

    if int(row.get("v12b_model_near_boundary_flag", 0)) == 1:
        reasons.append("Model score near decision boundary")

    if not reasons:
        return ""

    # Remove duplicate reason strings while preserving order.
    unique_reasons = []
    for reason in reasons:
        if reason not in unique_reasons:
            unique_reasons.append(reason)

    return "; ".join(unique_reasons)


directional_scenario_v12b_df["directional_review_reasons"] = (
    directional_scenario_v12b_df.apply(
        build_directional_review_reasons,
        axis=1,
    )
)


directional_scenario_v12b_df["directional_pair_recommendation_clean"] = (
    directional_scenario_v12b_df["v12b_pair_recommendation_clean"]
)

# Pair-level model/LF reviews remain reviews.
# For RTD-required pair outputs, tank context determines whether it can be credited.

rtd_required_directional_mask = (
    directional_scenario_v12b_df["v12b_pair_recommendation_clean"]
    .eq("RTD required")
)

directionality_sensitive_mask = (
    directional_scenario_v12b_df["v12b_directionality_sensitive_flag"].eq(1)
)

stored_service_impractical_mask = (
    directional_scenario_v12b_df["stored_service_rtd_practicality_proxy"].eq(0)
)

directional_scenario_v12b_df.loc[
    rtd_required_directional_mask
    & directionality_sensitive_mask
    & stored_service_impractical_mask,
    "directional_pair_recommendation_clean",
] = "Review - RTD candidate requires directionality and stored-service validation"

directional_scenario_v12b_df.loc[
    rtd_required_directional_mask
    & directionality_sensitive_mask
    & ~stored_service_impractical_mask,
    "directional_pair_recommendation_clean",
] = "Review - RTD candidate requires directionality / inventory validation"

directional_scenario_v12b_df.loc[
    rtd_required_directional_mask
    & ~directionality_sensitive_mask
    & stored_service_impractical_mask,
    "directional_pair_recommendation_clean",
] = "Review - RTD candidate but stored service may be impractical"

directional_scenario_v12b_df.loc[
    rtd_required_directional_mask
    & ~directionality_sensitive_mask
    & ~stored_service_impractical_mask,
    "directional_pair_recommendation_clean",
] = "RTD candidate - directionally credible in screening"


print("Refined directional scenario rows created:")
print(f"  Pair rows:        {len(python_df)}")
print(f"  Directional rows: {len(directional_scenario_v12b_df)}")

print("\nRefined directional scenario recommendation distribution:")
display(
    directional_scenario_v12b_df["directional_pair_recommendation_clean"]
    .value_counts(dropna=False)
    .rename_axis("directional_pair_recommendation_clean")
    .reset_index(name="row_count")
)

print("\nStored-service RTD practicality proxy distribution:")
display(
    directional_scenario_v12b_df["stored_service_rtd_practicality_proxy"]
    .value_counts(dropna=False)
    .rename_axis("stored_service_rtd_practicality_proxy")
    .reset_index(name="row_count")
)

print("\nDirectional review reason examples:")
display(
    directional_scenario_v12b_df.loc[
        directional_scenario_v12b_df["directional_pair_recommendation_clean"]
        .astype(str)
        .str.startswith("Review"),
        [
            "pair_id",
            "directional_scenario",
            "incoming_chemical_label",
            "incoming_chemical_class",
            "stored_chemical_label",
            "stored_chemical_class",
            "v12b_pair_recommendation_clean",
            "directional_pair_recommendation_clean",
            "stored_service_rtd_practicality_proxy",
            "directional_review_reasons",
        ],
    ]
    .head(30)
)

# 15. Service Level / Tank Level RTD Aggregation

The directional scenario table contains one row for each possible abnormal transfer direction: `A_into_B` and `B_into_A`.

A real plant implementation would aggregate these directional scenarios to the actual tank level using a site-specific tank map. The Phase 1 public dataset does not include actual tank IDs, tank levels, concentrations, transfer rates, or normal stored inventories. Therefore, this MVP uses `stored_chemical_label` as a stand-in for tank service.

This section aggregates directional scenarios by stored chemical service and produces an engineer-facing recommendation:

- `RTD not required based on current screening`
- `Review - model challenge scenarios present`
- `Review - potential RTD candidate requires tank/inventory validation`
- `Review - potential RTD candidate requires tank/inventory and stored-service validation`
- `RTD candidate - proceed to site validation`

This is intentionally conservative. The tank/service recommendation does not automatically authorize installing or omitting an RTD. It identifies where RTD may be useful, where it is likely not useful, and where additional tank-specific engineering review is required.

In [ ]:
# ============================================================
# 15.1 Aggregate Directional Scenarios by Stored Chemical Service
# ============================================================

if "directional_scenario_v12b_df" not in globals():
    raise NameError(
        "directional_scenario_v12b_df not found. "
        "Run Sections 14.2-14.3 before tank/service aggregation."
    )


def unique_join(values, max_items=8):
    """Join unique nonblank values for readable summaries."""
    clean_values = []
    for value in values:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if text and text not in clean_values:
            clean_values.append(text)

    if len(clean_values) > max_items:
        return "; ".join(clean_values[:max_items]) + "; ..."

    return "; ".join(clean_values)


def count_contains(series, text):
    """Count rows where a string column contains a phrase."""
    return int(
        series.astype(str)
        .str.contains(text, case=False, na=False, regex=False)
        .sum()
    )


directional_for_aggregation_df = directional_scenario_v12b_df.copy()

directional_for_aggregation_df["is_directional_rtd_candidate_review"] = (
    directional_for_aggregation_df[
        "directional_pair_recommendation_clean"
    ]
    .astype(str)
    .str.contains(
        "RTD candidate requires",
        case=False,
        na=False,
        regex=False,
    )
).astype("Int64")

directional_for_aggregation_df["is_directionally_credible_rtd_candidate"] = (
    directional_for_aggregation_df[
        "directional_pair_recommendation_clean"
    ]
    .astype(str)
    .eq("RTD candidate - directionally credible in screening")
).astype("Int64")

directional_for_aggregation_df["is_model_challenge_review"] = (
    directional_for_aggregation_df[
        "directional_pair_recommendation_clean"
    ]
    .astype(str)
    .str.contains(
        "model challenges",
        case=False,
        na=False,
        regex=False,
    )
).astype("Int64")

directional_for_aggregation_df["is_stored_service_validation_needed"] = (
    directional_for_aggregation_df[
        "directional_pair_recommendation_clean"
    ]
    .astype(str)
    .str.contains(
        "stored-service validation",
        case=False,
        na=False,
        regex=False,
    )
).astype("Int64")

directional_for_aggregation_df["is_directionality_validation_needed"] = (
    directional_for_aggregation_df[
        "directional_pair_recommendation_clean"
    ]
    .astype(str)
    .str.contains(
        "directionality",
        case=False,
        na=False,
        regex=False,
    )
).astype("Int64")

directional_for_aggregation_df["is_rtd_not_required_directional"] = (
    directional_for_aggregation_df[
        "directional_pair_recommendation_clean"
    ]
    .astype(str)
    .eq("RTD not required")
).astype("Int64")


service_group_columns = [
    "stored_chemical_label",
    "stored_chemical_class",
]

stored_service_summary_df = (
    directional_for_aggregation_df
    .groupby(service_group_columns, dropna=False)
    .agg(
        directional_scenario_count=("pair_id", "count"),
        unique_pair_count=("pair_id", "nunique"),
        rtd_not_required_scenarios=("is_rtd_not_required_directional", "sum"),
        model_challenge_review_scenarios=("is_model_challenge_review", "sum"),
        rtd_candidate_review_scenarios=("is_directional_rtd_candidate_review", "sum"),
        directionally_credible_rtd_candidate_scenarios=(
            "is_directionally_credible_rtd_candidate",
            "sum",
        ),
        directionality_validation_needed_scenarios=(
            "is_directionality_validation_needed",
            "sum",
        ),
        stored_service_validation_needed_scenarios=(
            "is_stored_service_validation_needed",
            "sum",
        ),
        stored_service_rtd_practicality_proxy=(
            "stored_service_rtd_practicality_proxy",
            "min",
        ),
        max_v12b_model_score=("v12b_model_score", "max"),
        mean_v12b_model_score=("v12b_model_score", "mean"),
        example_pair_ids=("pair_id", unique_join),
        example_review_reasons=("directional_review_reasons", unique_join),
    )
    .reset_index()
)

stored_service_summary_df["tank_level_recommendation"] = (
    "RTD not required based on current screening"
)

stored_service_summary_df.loc[
    stored_service_summary_df[
        "model_challenge_review_scenarios"
    ].gt(0),
    "tank_level_recommendation",
] = "Review - model challenge scenarios present"

stored_service_summary_df.loc[
    stored_service_summary_df[
        "rtd_candidate_review_scenarios"
    ].gt(0)
    & stored_service_summary_df[
        "stored_service_validation_needed_scenarios"
    ].eq(0),
    "tank_level_recommendation",
] = "Review - potential RTD candidate requires tank/inventory validation"

stored_service_summary_df.loc[
    stored_service_summary_df[
        "stored_service_validation_needed_scenarios"
    ].gt(0),
    "tank_level_recommendation",
] = (
    "Review - potential RTD candidate requires tank/inventory "
    "and stored-service validation"
)

stored_service_summary_df.loc[
    stored_service_summary_df[
        "directionally_credible_rtd_candidate_scenarios"
    ].gt(0)
    & stored_service_summary_df[
        "rtd_candidate_review_scenarios"
    ].eq(0)
    & stored_service_summary_df[
        "model_challenge_review_scenarios"
    ].eq(0)
    & stored_service_summary_df[
        "stored_service_rtd_practicality_proxy"
    ].eq(1),
    "tank_level_recommendation",
] = "RTD candidate - proceed to site validation"


def build_tank_level_reason(row):
    reasons = []

    if row["directionally_credible_rtd_candidate_scenarios"] > 0:
        reasons.append("At least one directionally credible RTD candidate scenario")

    if row["rtd_candidate_review_scenarios"] > 0:
        reasons.append("Potential RTD candidate scenario requires tank/inventory validation")

    if row["stored_service_validation_needed_scenarios"] > 0:
        reasons.append("Stored service may reduce RTD reliability")

    if row["model_challenge_review_scenarios"] > 0:
        reasons.append("Model challenges calibrated weak-label result for one or more scenarios")

    if row["stored_service_rtd_practicality_proxy"] == 0:
        reasons.append("Stored-service RTD practicality proxy is unfavorable")

    if not reasons:
        reasons.append("No RTD-creditable scenario identified in current screening")

    return "; ".join(reasons)


stored_service_summary_df["tank_level_reason"] = (
    stored_service_summary_df.apply(build_tank_level_reason, axis=1)
)

stored_service_summary_df["tank_level_priority_score"] = (
    5.0 * stored_service_summary_df["directionally_credible_rtd_candidate_scenarios"]
    + 4.0 * stored_service_summary_df["rtd_candidate_review_scenarios"]
    + 3.0 * stored_service_summary_df["stored_service_validation_needed_scenarios"]
    + 2.0 * stored_service_summary_df["model_challenge_review_scenarios"]
    + stored_service_summary_df["max_v12b_model_score"].fillna(0)
)

stored_service_summary_df = stored_service_summary_df.sort_values(
    [
        "tank_level_priority_score",
        "rtd_candidate_review_scenarios",
        "model_challenge_review_scenarios",
        "stored_chemical_label",
    ],
    ascending=[False, False, False, True],
).reset_index(drop=True)


print("Stored-service / tank-level recommendation distribution:")
display(
    stored_service_summary_df["tank_level_recommendation"]
    .value_counts(dropna=False)
    .rename_axis("tank_level_recommendation")
    .reset_index(name="service_count")
)

print("\nTop stored-service recommendations by priority:")
display(
    stored_service_summary_df[
        [
            "stored_chemical_label",
            "stored_chemical_class",
            "directional_scenario_count",
            "unique_pair_count",
            "rtd_not_required_scenarios",
            "model_challenge_review_scenarios",
            "rtd_candidate_review_scenarios",
            "directionally_credible_rtd_candidate_scenarios",
            "stored_service_validation_needed_scenarios",
            "stored_service_rtd_practicality_proxy",
            "tank_level_recommendation",
            "tank_level_reason",
            "tank_level_priority_score",
            "example_pair_ids",
        ]
    ].head(30)
)

In [ ]:
# ============================================================
# 15.2 Export Directional and Stored-Service Recommendation Tables
# ============================================================

directional_export_columns = [
    "pair_id",
    "canonical_pair_key",
    "directional_scenario",
    "incoming_chemical_label",
    "incoming_chemical_class",
    "stored_chemical_label",
    "stored_chemical_class",
    "reaction_type",
    "reaction_speed",
    "setting",
    "final_severity_num",
    "toxic_gas",
    "generates_gas",
    "generates_heat",
    "explosive_potential",
    "flammable",
    "corrosive",
    "v12b_weak_rtd_decision",
    "v12b_pair_recommendation_clean",
    "directional_pair_recommendation_clean",
    "directional_review_reasons",
    "stored_service_rtd_practicality_proxy",
    "v12b_model_score",
    "v12b_model_pred_0p50",
    "v12b_model_disagrees_with_weak_label",
    "v12b_directionality_sensitive_flag",
    "v12b_protected_bulk_acidbase_rtd_pattern",
    "v12b_nonthermal_dominance_proxy",
    "v12b_phase_behavior_failure_proxy",
    "v12b_fast_pressure_or_kinetics_failure_proxy",
]

directional_export_columns = [
    column for column in directional_export_columns
    if column in directional_scenario_v12b_df.columns
]

service_export_columns = [
    "stored_chemical_label",
    "stored_chemical_class",
    "directional_scenario_count",
    "unique_pair_count",
    "rtd_not_required_scenarios",
    "model_challenge_review_scenarios",
    "rtd_candidate_review_scenarios",
    "directionally_credible_rtd_candidate_scenarios",
    "directionality_validation_needed_scenarios",
    "stored_service_validation_needed_scenarios",
    "stored_service_rtd_practicality_proxy",
    "max_v12b_model_score",
    "mean_v12b_model_score",
    "tank_level_recommendation",
    "tank_level_reason",
    "tank_level_priority_score",
    "example_pair_ids",
    "example_review_reasons",
]

service_export_columns = [
    column for column in service_export_columns
    if column in stored_service_summary_df.columns
]

directional_export_path = (
    "chemrisk_ai_v12b_directional_scenarios.csv"
)

service_export_path = (
    "chemrisk_ai_v12b_stored_service_recommendations.csv"
)

directional_scenario_v12b_df[directional_export_columns].to_csv(
    directional_export_path,
    index=False,
)

stored_service_summary_df[service_export_columns].to_csv(
    service_export_path,
    index=False,
)

print(f"Directional scenario export written to: {directional_export_path}")
print(f"Stored-service recommendation export written to: {service_export_path}")

print("\nExport preview - stored-service recommendations:")
display(
    stored_service_summary_df[service_export_columns].head(15)
)

## 15.3 Tank Level Aggregation Interpretation

The service level aggregation is a Phase 1 proxy for true tank level aggregation. It uses the stored chemical service as a stand-in for a real tank because the public dataset does not contain site-specific tank IDs, normal inventories, tank levels, concentrations, transfer rates, or instrumentation details.

The output should therefore be interpreted as a prioritized engineering review table:

- services with no RTD-creditable directional scenarios are screened as `RTD not required based on current screening`,
- services with model disagreement scenarios are routed to review,
- services with potential RTD candidate scenarios are routed to tank/inventory validation,
- and services where stored-service reliability may be poor require additional validation before crediting RTD.

In a real deployment, this table would be joined to a site tank map. Each tank would inherit directional scenarios based on its stored chemical service, then the final recommendation would be reviewed against tank geometry, inventory, fill level, transfer pathway, existing nozzles, instrument reliability, and operating procedures.

## 15.4 Stored Service Reliability Review Terminology

The stored service practicality proxy is intentionally conservative. It does not prove that an RTD is unreliable in a given tank. Instead, it flags stored services where RTD reliability should be reviewed before the instrument is credited as a safeguard.

This distinction is important for surfactants, polymers, emulsions, silicates, waxes, slurries, and metal complex services. These materials may create foaming, coating, fouling, phase separation, poor mixing, gas entrainment, or localized reaction behavior, but the public dataset does not contain enough tank-specific information to conclude that RTD measurement is impractical.

Therefore, the final wording uses:

> stored-service RTD reliability review

rather than:

> stored-service RTD impracticality

This preserves conservative review routing without overstating what the proxy can prove.

In [ ]:
# ============================================================
# 15.4 Stored Service Reliability Review Terminology Cleanup
# ============================================================

# Rename the interpretation of stored_service_rtd_practicality_proxy.
# Existing proxy convention:
#   1 = no obvious stored service RTD reliability concern from class proxy
#   0 = stored-service RTD reliability review needed

directional_scenario_v12b_df["stored_service_rtd_reliability_review_flag"] = (
    directional_scenario_v12b_df["stored_service_rtd_practicality_proxy"]
    .eq(0)
    .astype("Int64")
)

stored_service_summary_df["stored_service_rtd_reliability_review_flag"] = (
    stored_service_summary_df["stored_service_rtd_practicality_proxy"]
    .eq(0)
    .astype("Int64")
)

# Create cleaned recommendation/reason fields for final reporting.
stored_service_summary_df["tank_level_recommendation_final"] = (
    stored_service_summary_df["tank_level_recommendation"]
    .astype(str)
    .str.replace(
        "stored-service validation",
        "stored-service reliability validation",
        regex=False,
    )
)

stored_service_summary_df["tank_level_reason_final"] = (
    stored_service_summary_df["tank_level_reason"]
    .astype(str)
    .str.replace(
        "Stored service may reduce RTD reliability",
        "Stored service requires RTD reliability review before crediting RTD",
        regex=False,
    )
    .str.replace(
        "Stored-service RTD practicality proxy is unfavorable",
        "Stored-service RTD reliability review flag is active",
        regex=False,
    )
)

directional_scenario_v12b_df["directional_pair_recommendation_final"] = (
    directional_scenario_v12b_df["directional_pair_recommendation_clean"]
    .astype(str)
    .str.replace(
        "stored-service validation",
        "stored-service reliability validation",
        regex=False,
    )
)

directional_scenario_v12b_df["directional_review_reasons_final"] = (
    directional_scenario_v12b_df["directional_review_reasons"]
    .astype(str)
    .str.replace(
        "Stored service may foul, coat, phase-separate, or reduce RTD reliability",
        "Stored service requires RTD reliability review before crediting RTD",
        regex=False,
    )
)

print("Final stored-service recommendation distribution:")
display(
    stored_service_summary_df["tank_level_recommendation_final"]
    .value_counts(dropna=False)
    .rename_axis("tank_level_recommendation_final")
    .reset_index(name="service_count")
)

print("\nStored-service RTD reliability review flag distribution:")
display(
    stored_service_summary_df["stored_service_rtd_reliability_review_flag"]
    .value_counts(dropna=False)
    .rename_axis("stored_service_rtd_reliability_review_flag")
    .reset_index(name="service_count")
)

print("\nTop final stored-service recommendations:")
display(
    stored_service_summary_df[
        [
            "stored_chemical_label",
            "stored_chemical_class",
            "directional_scenario_count",
            "unique_pair_count",
            "model_challenge_review_scenarios",
            "rtd_candidate_review_scenarios",
            "stored_service_rtd_reliability_review_flag",
            "tank_level_recommendation_final",
            "tank_level_reason_final",
            "tank_level_priority_score",
            "example_pair_ids",
        ]
    ].head(20)
)

# 16. Final Capstone Summary and Model Card

This final section summarizes what the Phase 1 MVP achieved, what the model should and should not be used for, and what remains site specific.

ChemRisk-AI is best interpreted as a mechanism-aware safeguard decision support workflow. It does not claim to predict chemical truth from first principles. Instead, it combines chemical engineering judgment, weak supervision, audit feedback, interpretable tabular modeling, and tank-level review routing to support one practical question:

> When two incompatible chemicals could be accidentally mixed, is an RTD a useful, creditable, and practical safeguard?

The final workflow includes:

1. spreadsheet-to-Python parity checks,
2. weak-label aggregation,
3. audited calibration,
4. grouped leakage-safe model training,
5. model-assisted review routing,
6. directionality-aware scenario expansion,
7. stored-service reliability review,
8. and service-level aggregation as a proxy for tank-level recommendation.

The final recommendation is intentionally human-in-the-loop. The model can identify likely RTD-relevant scenarios, likely no-RTD scenarios, and cases requiring review, but actual safeguard decisions still require site engineering validation.

In [ ]:
# ============================================================
# 16.1 Final Quantitative Summary
# ============================================================

final_summary_records = []

final_summary_records.append({
    "section": "Dataset",
    "metric": "Public chemical-pair rows",
    "value": len(python_df),
})

final_summary_records.append({
    "section": "Dataset",
    "metric": "Directional scenario rows",
    "value": len(directional_scenario_v12b_df),
})

if "combined_gold_audit_df" in globals():
    final_summary_records.append({
        "section": "Audit",
        "metric": "Combined audited rows",
        "value": len(combined_gold_audit_df),
    })

    final_summary_records.append({
        "section": "Audit",
        "metric": "Audited RTD-required rows",
        "value": int(
            pd.to_numeric(
                combined_gold_audit_df["rtd_required"],
                errors="coerce",
            ).fillna(0).sum()
        ),
    })

    final_summary_records.append({
        "section": "Audit",
        "metric": "Audited RTD-not-required rows",
        "value": int(
            pd.to_numeric(
                combined_gold_audit_df["rtd_required"],
                errors="coerce",
            ).eq(0).sum()
        ),
    })

if "post_next15_leakage_safe_pool_df" in globals():
    final_summary_records.append({
        "section": "Training pool",
        "metric": "Post-audit leakage-safe rows",
        "value": len(post_next15_leakage_safe_pool_df),
    })

if "v12b_weak_labeled_model_pool_df" in globals():
    final_summary_records.append({
        "section": "Training pool",
        "metric": "v1.2b weak-labeled training rows",
        "value": len(v12b_weak_labeled_model_pool_df),
    })

    final_summary_records.append({
        "section": "Training pool",
        "metric": "v1.2b RTD-required weak labels in training pool",
        "value": int(v12b_weak_labeled_model_pool_df["v12b_weak_rtd_label"].sum()),
    })

    final_summary_records.append({
        "section": "Training pool",
        "metric": "v1.2b RTD-required fraction in training pool",
        "value": f"{v12b_weak_labeled_model_pool_df['v12b_weak_rtd_label'].mean():.1%}",
    })

if "v12b_audit_summary" in globals():
    for metric in [
        "accuracy",
        "balanced_accuracy",
        "precision_rtd",
        "recall_rtd",
        "f1_rtd",
        "false_positive",
        "false_negative",
    ]:
        value = v12b_audit_summary.get(metric, None)
        if value is not None:
            if metric in {
                "accuracy",
                "balanced_accuracy",
                "precision_rtd",
                "recall_rtd",
                "f1_rtd",
            }:
                value = f"{value:.1%}"
            final_summary_records.append({
                "section": "v1.2b audit calibration",
                "metric": metric,
                "value": value,
            })

if "v12b_oof_model_summary_df" in globals():
    primary_model_summary = v12b_oof_model_summary_df.loc[
        v12b_oof_model_summary_df["model_name"].eq(V12B_PRIMARY_MODEL_NAME)
    ]

    if not primary_model_summary.empty:
        primary_row = primary_model_summary.iloc[0]

        for metric in [
            "balanced_accuracy",
            "precision_rtd",
            "recall_rtd",
            "f1_rtd",
            "average_precision",
            "roc_auc",
            "false_positive",
            "false_negative",
        ]:
            value = primary_row[metric]
            if metric in {
                "balanced_accuracy",
                "precision_rtd",
                "recall_rtd",
                "f1_rtd",
                "average_precision",
                "roc_auc",
            }:
                value = f"{value:.1%}"

            final_summary_records.append({
                "section": "v1.2b grouped CV model",
                "metric": metric,
                "value": value,
            })

final_summary_records.append({
    "section": "Pair-level final overlay",
    "metric": "RTD not required pair recommendations",
    "value": int(
        python_df["v12b_pair_recommendation_clean"]
        .eq("RTD not required")
        .sum()
    ),
})

final_summary_records.append({
    "section": "Pair-level final overlay",
    "metric": "RTD required pair recommendations before directional validation",
    "value": int(
        python_df["v12b_pair_recommendation_clean"]
        .eq("RTD required")
        .sum()
    ),
})

final_summary_records.append({
    "section": "Pair-level final overlay",
    "metric": "Model-challenge review pair recommendations",
    "value": int(
        python_df["v12b_pair_recommendation_clean"]
        .eq("Review - model challenges v1.2b weak label")
        .sum()
    ),
})

final_summary_df = pd.DataFrame(final_summary_records)

print("Final ChemRisk-AI Phase 1 MVP quantitative summary:")
display(final_summary_df)

In [ ]:
# ============================================================
# 16.2 Final MVP Model Card
# ============================================================

model_card_df = pd.DataFrame({
    "model_card_item": [
        "Project name",
        "Primary decision",
        "Primary target",
        "Model type",
        "Training label source",
        "Training-data limitation",
        "Validation approach",
        "Leakage controls",
        "Final calibrated LF version",
        "Primary diagnostic model",
        "Directionality handling",
        "Tank-level handling",
        "Human-in-the-loop requirement",
        "Not intended for",
        "Appropriate use",
        "Next validation step",
    ],
    "summary": [
        "ChemRisk-AI Phase 1 MVP",
        "Whether RTD is useful, creditable, and practical as a safeguard for incompatible mixing scenarios",
        "Pair-level RTD required vs. not required, with Review routing",
        "Weak-supervised tabular model with mechanism-aware rules and grouped cross-validation",
        "Engineering labeling functions calibrated against audited chemical-pair reviews",
        "No direct kinetics, heat of reaction, tank inventory, concentration, transfer-rate, or incident-outcome data",
        "Initial 21-row development audit plus 15-row prospective stress-test audit; both become development/calibration data once used for v1.2b",
        "Audited rows and repeated canonical chemical-pair groups excluded from model training",
        "Corrected v1.2b",
        V12B_PRIMARY_MODEL_NAME,
        "Each pair expanded into A_into_B and B_into_A directional scenarios",
        "Stored chemical service used as a Phase 1 proxy for tank service; actual deployment requires site tank map",
        "Required for all safety-critical decisions",
        "Automatic safeguard approval, chemical truth prediction, first-principles reaction modeling, or replacing HAZOP / PHA review",
        "Screening, prioritization, audit support, RTD applicability review, and engineering decision support",
        "Apply frozen v1.2b to additional prospective audit rows or a site-specific tank map with tank inventory and service data",
    ],
})

print("Final MVP model card:")
display(model_card_df)

## 16.3 Final Interpretation

The Phase 1 MVP successfully demonstrates a credible industrial AI workflow for chemical process safety decision support.

The strongest result is not that the model predicts chemical reality from limited data. The strongest result is that the workflow converts conservative pair-level screening data into a more decision-relevant safeguard recommendation framework. It separates cases where RTD may be useful from cases where RTD is likely the wrong safeguard or requires additional review.

The final v1.2b system is conservative in the right way:

- it avoids treating C5/C6 severity alone as an RTD trigger,
- it preserves known RTD-creditable bulk-liquid acid/base exotherm cases,
- it reduces over-recommendation for gas, pressure, phase-behavior, fouling, and non-thermal dominant mechanisms,
- it uses the tabular model as a review overlay rather than an automatic override,
- it expands pair-level predictions into direction-specific scenarios,
- and it routes tank-level decisions to site validation when inventory, directionality, or stored-service reliability matters.

What remains site-specific:

- actual tank ID and stored service mapping,
- normal and minimum tank inventory,
- concentration and dilution state,
- transfer rate and credible misdirected volume,
- order of addition,
- tank geometry and mixing,
- existing RTD location and thermowell design,
- fouling/coating history,
- venting and pressure relief configuration,
- operator response time,
- and whether RTD is credited as alarm-only, interlock input, or independent protection layer.

The final recommendation should therefore be read as a screening and prioritization output. It identifies where RTD is likely not required, where RTD may be useful, and where engineering review is required before crediting or rejecting RTD as a safeguard.

This framing is appropriate for an industrial AI capstone because it demonstrates responsible ML use in a safety-critical environment: interpretable features, weak supervision, audit feedback, leakage control, calibration versioning, active learning audit selection, model-assisted review routing, and explicit treatment of domain limitations.

In [ ]:
# ============================================================
# 16.4 Export Final Summary Tables
# ============================================================

final_summary_path = "chemrisk_ai_phase1_final_summary.csv"
model_card_path = "chemrisk_ai_phase1_model_card.csv"
stored_service_final_path = "chemrisk_ai_v12b_stored_service_recommendations_final.csv"
directional_final_path = "chemrisk_ai_v12b_directional_scenarios_final.csv"

final_summary_df.to_csv(final_summary_path, index=False)
model_card_df.to_csv(model_card_path, index=False)

stored_service_summary_df[
    [
        column for column in [
            "stored_chemical_label",
            "stored_chemical_class",
            "directional_scenario_count",
            "unique_pair_count",
            "rtd_not_required_scenarios",
            "model_challenge_review_scenarios",
            "rtd_candidate_review_scenarios",
            "stored_service_rtd_reliability_review_flag",
            "max_v12b_model_score",
            "mean_v12b_model_score",
            "tank_level_recommendation_final",
            "tank_level_reason_final",
            "tank_level_priority_score",
            "example_pair_ids",
            "example_review_reasons",
        ]
        if column in stored_service_summary_df.columns
    ]
].to_csv(stored_service_final_path, index=False)

directional_scenario_v12b_df[
    [
        column for column in [
            "pair_id",
            "canonical_pair_key",
            "directional_scenario",
            "incoming_chemical_label",
            "incoming_chemical_class",
            "stored_chemical_label",
            "stored_chemical_class",
            "reaction_type",
            "reaction_speed",
            "setting",
            "final_severity_num",
            "toxic_gas",
            "generates_gas",
            "generates_heat",
            "explosive_potential",
            "flammable",
            "corrosive",
            "v12b_weak_rtd_decision",
            "v12b_pair_recommendation_clean",
            "directional_pair_recommendation_final",
            "directional_review_reasons_final",
            "stored_service_rtd_reliability_review_flag",
            "v12b_model_score",
            "v12b_model_pred_0p50",
            "v12b_model_disagrees_with_weak_label",
            "v12b_directionality_sensitive_flag",
            "v12b_protected_bulk_acidbase_rtd_pattern",
            "v12b_nonthermal_dominance_proxy",
            "v12b_phase_behavior_failure_proxy",
            "v12b_fast_pressure_or_kinetics_failure_proxy",
        ]
        if column in directional_scenario_v12b_df.columns
    ]
].to_csv(directional_final_path, index=False)

print("Final summary package exported:")
print(f"  {final_summary_path}")
print(f"  {model_card_path}")
print(f"  {stored_service_final_path}")
print(f"  {directional_final_path}")

try:
    from google.colab import files

    print(
        "\nTo download these files manually, use the Colab file browser "
        "or uncomment the files.download lines below."
    )

    # files.download(final_summary_path)
    # files.download(model_card_path)
    # files.download(stored_service_final_path)
    # files.download(directional_final_path)

except Exception:
    pass

## 16.4 Validation Strength and Statistical Limitations

The Phase 1 MVP results should be interpreted as calibrated decision-support evidence, not as a statistically powered validation study.

The combined audited set contains 36 rows, including 5 RTD-required cases and 31 RTD-not-required cases. This is sufficient to support MVP calibration, error analysis, and active-learning stress testing, but it is not enough to claim final statistical validation or production readiness.

The grouped cross-validation model results should also be interpreted in the context of class imbalance. In the v1.2b leakage-safe training pool, only 11 of 271 rows are labeled RTD required, or approximately 4.1%. Because the positive class is rare, plain accuracy is not a useful success metric. The model is evaluated using balanced accuracy, RTD precision, RTD recall, F1 score, average precision, ROC-AUC, and false-negative count.

The selected v1.2b diagnostic model achieved 100% RTD recall with zero false negatives in grouped cross-validation, but RTD precision was 55%. This is acceptable for the Phase 1 safety-screening use case because model-positive disagreements are routed to engineering review rather than automatically becoming RTD installation scope. Additionally, false negatives would be the larger concern from a process safety perspective, and the model is optimized with 0 false negatives in the grouped CV model metrics.

The model should therefore be described as a prioritization and review-routing tool. It enriches rare RTD-required candidates and highlights disagreements, but it does not independently authorize safeguard decisions.

A stronger validation package would require additional prospective audit rows, more RTD-required examples, site-specific tank mapping, and ideally reaction test or incident/outcome data. Future validation target: expand to at least 100 audited rows with intentional oversampling of RTD-required candidates.

# 17. Portfolio Visuals for GitHub and Presentation

This section creates presentation-ready figures for the GitHub README, interview slides, and final project summary.

The visuals focus on the core story:

1. the end-to-end ChemRisk-AI workflow,
2. class imbalance,
3. audited calibration performance,
4. grouped cross-validation model performance,
5. final pair level recommendations,
6. directionality aware scenario recommendations,
7. stored service / tank level recommendation distribution,
8. and top stored service priorities.

These figures are saved to the `figures/` folder as PNG files.

In [ ]:
# ============================================================
# 17.0 Visualization Setup and Helper Functions
# ============================================================

from pathlib import Path
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

figures_dir = Path("figures")
figures_dir.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})


def save_current_figure(filename):
    """Save current matplotlib figure to the figures folder."""
    path = figures_dir / filename
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")
    plt.show()


def wrap_label(text, width=28):
    """Wrap long chart labels for readability."""
    return "\n".join(textwrap.wrap(str(text), width=width))


def add_bar_labels(ax, orientation="vertical", fmt="{:.0f}"):
    """Add value labels to bars."""
    if orientation == "vertical":
        for patch in ax.patches:
            height = patch.get_height()
            ax.annotate(
                fmt.format(height),
                (patch.get_x() + patch.get_width() / 2, height),
                ha="center",
                va="bottom",
                xytext=(0, 3),
                textcoords="offset points",
            )
    else:
        for patch in ax.patches:
            width = patch.get_width()
            ax.annotate(
                fmt.format(width),
                (width, patch.get_y() + patch.get_height() / 2),
                ha="left",
                va="center",
                xytext=(4, 0),
                textcoords="offset points",
            )


def require_variable(variable_name):
    """Fail clearly if a required notebook variable is missing."""
    if variable_name not in globals():
        raise NameError(
            f"Required variable `{variable_name}` was not found. "
            f"Run the prior notebook sections before Section 17."
        )

In [ ]:
# ============================================================
# 17.1 Visual 1 - ChemRisk-AI Architecture Diagram
# ============================================================

architecture_steps = [
    ("Public pair\nscreening data", "313 chemical-pair rows"),
    ("Mechanism-aware\nlabeling functions", "RTD usefulness logic"),
    ("Audit calibration", "21 initial + 15 prospective rows"),
    ("Leakage-safe\nmodel training", "Grouped CV by canonical pair"),
    ("Model-assisted\nreview overlay", "Disagreement routed to Review"),
    ("Directionality\nexpansion", "A_into_B and B_into_A"),
    ("Stored-service\nreliability review", "Fouling / coating / mixing flags"),
    ("Service-level\nrecommendations", "Tank map proxy output"),
]

fig, ax = plt.subplots(figsize=(16, 5))
ax.set_xlim(0, len(architecture_steps))
ax.set_ylim(0, 1)
ax.axis("off")

box_width = 0.86
box_height = 0.46
y0 = 0.33

for idx, (title, subtitle) in enumerate(architecture_steps):
    x0 = idx + 0.07

    box = FancyBboxPatch(
        (x0, y0),
        box_width,
        box_height,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        linewidth=1.2,
        facecolor="white",
        edgecolor="black",
    )
    ax.add_patch(box)

    ax.text(
        x0 + box_width / 2,
        y0 + 0.30,
        title,
        ha="center",
        va="center",
        weight="bold",
    )

    ax.text(
        x0 + box_width / 2,
        y0 + 0.13,
        subtitle,
        ha="center",
        va="center",
        fontsize=8,
    )

    if idx < len(architecture_steps) - 1:
        ax.annotate(
            "",
            xy=(idx + 1.04, y0 + box_height / 2),
            xytext=(idx + 0.94, y0 + box_height / 2),
            arrowprops=dict(arrowstyle="->", linewidth=1.2),
        )

ax.set_title(
    "ChemRisk-AI Phase 1 MVP Workflow",
    pad=16,
    weight="bold",
)

save_current_figure("01_chemrisk_ai_architecture.png")

In [ ]:
# ============================================================
# 17.2 Visual 2 - v1.2b Class Imbalance
# ============================================================

require_variable("v12b_weak_labeled_model_pool_df")

class_counts = (
    v12b_weak_labeled_model_pool_df["v12b_weak_rtd_label"]
    .map({0: "RTD not required", 1: "RTD required"})
    .value_counts()
    .reindex(["RTD not required", "RTD required"])
    .fillna(0)
)

positive_fraction = (
    v12b_weak_labeled_model_pool_df["v12b_weak_rtd_label"].mean()
)

fig, ax = plt.subplots(figsize=(7.5, 5))
class_counts.plot(kind="bar", ax=ax)

ax.set_title(
    "v1.2b Training Pool Class Imbalance",
    weight="bold",
)
ax.set_ylabel("Rows")
ax.set_xlabel("")
ax.set_xticklabels(class_counts.index, rotation=0)

add_bar_labels(ax)

# Keep some headroom for labels and annotation.
ax.set_ylim(0, max(class_counts) * 1.18)

# Bar centers are 0 and 1. Place the annotation over the RTD-required bar.
rtd_required_x_position = list(class_counts.index).index("RTD required")
rtd_required_count = class_counts.loc["RTD required"]

ax.annotate(
    f"RTD-required fraction: {positive_fraction:.1%}",
    xy=(rtd_required_x_position, rtd_required_count),
    xytext=(rtd_required_x_position, max(class_counts) * 0.35),
    ha="center",
    va="center",
    arrowprops=dict(arrowstyle="->", linewidth=1.0),
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="black",
    ),
)

save_current_figure("02_v12b_class_imbalance.png")

In [ ]:
# ============================================================
# 17.3 Visual 3 - v1.2b Audit Calibration Confusion Matrix
# ============================================================

# Prefer computed v12b_eval_df if available. Fall back to known final values.
if "v12b_eval_df" in globals() and {
    "audit_rtd_required",
    "v12b_weak_rtd_label",
}.issubset(v12b_eval_df.columns):

    audit_eval_plot_df = v12b_eval_df.loc[
        v12b_eval_df["v12b_weak_rtd_label"].isin([0, 1])
    ].copy()

    audit_eval_plot_df["audit_rtd_required_plot"] = pd.to_numeric(
        audit_eval_plot_df["audit_rtd_required"],
        errors="coerce",
    ).astype("Int64")

    audit_eval_plot_df["v12b_weak_rtd_label_plot"] = pd.to_numeric(
        audit_eval_plot_df["v12b_weak_rtd_label"],
        errors="coerce",
    ).astype("Int64")

    audit_eval_plot_df = audit_eval_plot_df.dropna(
        subset=[
            "audit_rtd_required_plot",
            "v12b_weak_rtd_label_plot",
        ]
    ).copy()

    audit_eval_plot_df["audit_rtd_required_plot"] = (
        audit_eval_plot_df["audit_rtd_required_plot"].astype(int)
    )

    audit_eval_plot_df["v12b_weak_rtd_label_plot"] = (
        audit_eval_plot_df["v12b_weak_rtd_label_plot"].astype(int)
    )

    tn = int(
        (
            audit_eval_plot_df["audit_rtd_required_plot"].eq(0)
            & audit_eval_plot_df["v12b_weak_rtd_label_plot"].eq(0)
        ).sum()
    )

    fp = int(
        (
            audit_eval_plot_df["audit_rtd_required_plot"].eq(0)
            & audit_eval_plot_df["v12b_weak_rtd_label_plot"].eq(1)
        ).sum()
    )

    fn = int(
        (
            audit_eval_plot_df["audit_rtd_required_plot"].eq(1)
            & audit_eval_plot_df["v12b_weak_rtd_label_plot"].eq(0)
        ).sum()
    )

    tp = int(
        (
            audit_eval_plot_df["audit_rtd_required_plot"].eq(1)
            & audit_eval_plot_df["v12b_weak_rtd_label_plot"].eq(1)
        ).sum()
    )

else:
    # Final values from corrected v1.2b combined 36-row audit diagnostic.
    tn, fp, fn, tp = 30, 1, 0, 5

confusion_matrix = np.array([
    [tn, fp],
    [fn, tp],
])

fig, ax = plt.subplots(figsize=(6.5, 5.5))
image = ax.imshow(confusion_matrix)

ax.set_title(
    "Corrected v1.2b vs. Combined Audited Development Set",
    weight="bold",
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Audited")
ax.set_xticks([0, 1])
ax.set_xticklabels(["RTD not required", "RTD required"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["RTD not required", "RTD required"])

for row in range(confusion_matrix.shape[0]):
    for col in range(confusion_matrix.shape[1]):
        ax.text(
            col,
            row,
            str(confusion_matrix[row, col]),
            ha="center",
            va="center",
            fontsize=16,
            weight="bold",
        )

plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

ax.text(
    0.5,
    -0.23,
    f"36 audited rows: {tn + fp} RTD-not-required, {fn + tp} RTD-required",
    transform=ax.transAxes,
    ha="center",
    va="center",
)

save_current_figure("03_v12b_audit_confusion_matrix.png")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

require_variable("v12b_oof_model_summary_df")
require_variable("V12B_PRIMARY_MODEL_NAME")

primary_metrics_row = v12b_oof_model_summary_df.loc[
    v12b_oof_model_summary_df["model_name"].eq(V12B_PRIMARY_MODEL_NAME)
].iloc[0]

cv_metric_series = pd.Series({
    "Balanced\naccuracy": primary_metrics_row["balanced_accuracy"],
    "Precision\nRTD": primary_metrics_row["precision_rtd"],
    "Recall\nRTD": primary_metrics_row["recall_rtd"],
    "F1\nRTD": primary_metrics_row["f1_rtd"],
    "Average\nprecision": primary_metrics_row["average_precision"],
    "ROC\nAUC": primary_metrics_row["roc_auc"],
})

fig, ax = plt.subplots(figsize=(9, 6)) # Increased figure height
(cv_metric_series * 100).plot(kind="bar", ax=ax)

ax.set_title(
    "v1.2b Primary Model Grouped CV Performance",
    weight="bold",
)
ax.set_ylabel("Metric value (%)")
ax.set_xlabel("")
ax.set_ylim(0, 120) # Adjusted y-axis limit to accommodate higher text
ax.set_xticklabels(cv_metric_series.index, rotation=0)

add_bar_labels(ax, fmt="{:.1f}")

ax.text(
    0.5,
    0.95, # Moved text lower to avoid title overlap
    f"Primary model: {V12B_PRIMARY_MODEL_NAME}",
    transform=ax.transAxes,
    ha="center",
    va="center",
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="black"),
)

save_current_figure("04_v12b_grouped_cv_metrics.png")

In [ ]:
# ============================================================
# 17.5 Visual 5 - Final Pair-Level Recommendation Distribution
# ============================================================

require_variable("python_df")

pair_recommendation_counts = (
    python_df["v12b_pair_recommendation_clean"]
    .value_counts()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 5))
pair_recommendation_counts.plot(kind="barh", ax=ax)

ax.set_title(
    "Final Pair-Level Recommendations with Model-Assisted Review Overlay",
    weight="bold",
)
ax.set_xlabel("Chemical-pair rows")
ax.set_ylabel("")
ax.set_yticklabels([
    wrap_label(label, width=34)
    for label in pair_recommendation_counts.index
])

add_bar_labels(ax, orientation="horizontal")

save_current_figure("05_pair_level_recommendation_distribution.png")

In [ ]:
# ============================================================
# 17.6 Visual 6 - Directional Scenario Recommendation Distribution
# ============================================================

require_variable("directional_scenario_v12b_df")

directional_recommendation_counts = (
    directional_scenario_v12b_df["directional_pair_recommendation_final"]
    .value_counts()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 5.5))
directional_recommendation_counts.plot(kind="barh", ax=ax)

ax.set_title(
    "Directionality-Aware Scenario Recommendations",
    weight="bold",
)
ax.set_xlabel("Directional scenario rows")
ax.set_ylabel("")
ax.set_yticklabels([
    wrap_label(label, width=42)
    for label in directional_recommendation_counts.index
])

add_bar_labels(ax, orientation="horizontal")

ax.text(
    0.98,
    0.06,
    "313 pair rows expanded to 626 directional scenarios",
    transform=ax.transAxes,
    ha="right",
    va="center",
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="black"),
)

save_current_figure("06_directional_scenario_recommendation_distribution.png")

In [ ]:
# ============================================================
# 17.7 Visual 7 - Stored-Service / Tank-Level Recommendation Distribution
# ============================================================

require_variable("stored_service_summary_df")

service_recommendation_counts = (
    stored_service_summary_df["tank_level_recommendation_final"]
    .value_counts()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 5.5))
service_recommendation_counts.plot(kind="barh", ax=ax)

ax.set_title(
    "Stored-Service / Tank-Level Recommendation Distribution",
    weight="bold",
)
ax.set_xlabel("Stored chemical services")
ax.set_ylabel("")
ax.set_yticklabels([
    wrap_label(label, width=45)
    for label in service_recommendation_counts.index
])

add_bar_labels(ax, orientation="horizontal")

save_current_figure("07_stored_service_recommendation_distribution.png")

In [ ]:
# ============================================================
# 17.8 Visual 8 - RTD-Candidate Stored-Service Priorities
# ============================================================

require_variable("stored_service_summary_df")

# Focus this chart only on services with potential RTD-candidate scenarios.
# This aligns the visual with the tank/service recommendation distribution.

rtd_candidate_service_priority_df = stored_service_summary_df.loc[
    stored_service_summary_df["rtd_candidate_review_scenarios"].gt(0)
    | stored_service_summary_df[
        "directionally_credible_rtd_candidate_scenarios"
    ].gt(0)
].copy()

if rtd_candidate_service_priority_df.empty:
    raise ValueError(
        "No RTD-candidate stored services were found. "
        "Check the tank/service aggregation logic."
    )

rtd_candidate_service_priority_df = (
    rtd_candidate_service_priority_df
    .sort_values("tank_level_priority_score", ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(
    rtd_candidate_service_priority_df["stored_chemical_label"],
    rtd_candidate_service_priority_df["tank_level_priority_score"],
)

ax.set_title(
    "Stored Services with Potential RTD-Candidate Scenarios",
    weight="bold",
)
ax.set_xlabel("Priority score")
ax.set_ylabel("")
ax.set_yticklabels([
    wrap_label(label, width=30)
    for label in rtd_candidate_service_priority_df["stored_chemical_label"]
])

add_bar_labels(ax, orientation="horizontal", fmt="{:.1f}")

ax.text(
    0.98,
    0.05,
    (
        f"{len(rtd_candidate_service_priority_df)} stored services have "
        "potential RTD-candidate scenarios requiring validation"
    ),
    transform=ax.transAxes,
    ha="right",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="black",
    ),
)

save_current_figure("08_rtd_candidate_stored_service_priorities.png")

In [ ]:
# ============================================================
# 17.9 Figure Index and Optional ZIP Export
# ============================================================

figure_index_df = pd.DataFrame({
    "figure_file": [
        "01_chemrisk_ai_architecture.png",
        "02_v12b_class_imbalance.png",
        "03_v12b_audit_confusion_matrix.png",
        "04_v12b_grouped_cv_metrics.png",
        "05_pair_level_recommendation_distribution.png",
        "06_directional_scenario_recommendation_distribution.png",
        "07_stored_service_recommendation_distribution.png",
        "08_rtd_candidate_stored_service_priorities.png",
    ],
    "recommended_use": [
        "README overview and presentation intro",
        "Model limitation / class imbalance explanation",
        "Audit calibration result",
        "Model performance summary",
        "Final pair-level decision output",
        "Directionality-aware expansion result",
        "Tank/service-level aggregation result",
        "RTD-candidate stored-service prioritization",
    ],
})

figure_index_path = figures_dir / "figure_index.csv"
figure_index_df.to_csv(figure_index_path, index=False)

print("Figure index:")
display(figure_index_df)

# Create a ZIP of all generated figures for easy download from Colab.
import shutil

zip_base_name = "chemrisk_ai_phase1_figures"
zip_path = shutil.make_archive(zip_base_name, "zip", figures_dir)

print(f"\nFigure ZIP created: {zip_path}")

try:
    from google.colab import files

    print(
        "\nTo download the ZIP now, uncomment and run the line below:"
    )
    print(f'files.download("{zip_path}")')

    # files.download(zip_path)

except Exception:
    pass